# 3\.3\.3 Calibrate and Generate Isolation Forest Anomalies

Isolation Forest approaches anomaly detection from a fundamentally different perspective than DBSCAN\. Rather than identifying observations located in sparse regions of feature space, Isolation Forest identifies observations that can be isolated quickly through recursive random partitioning\.

This notebook calibrates Isolation Forest inside the shared anomaly framework established in 3\.3\.1\. The goal is to determine whether Isolation Forest can produce stable, interpretable mobility\-state anomaly surfaces across the retained downstream representations and modalities, and whether those anomaly surfaces contribute useful structure beyond the previously generated DBSCAN outputs\.

In [1]:
# ---------------------------------------------------------------------
# Import libraries for Isolation Forest calibration and anomaly generation
# ---------------------------------------------------------------------

from pathlib import Path
from time import perf_counter
import json
import gc
import shutil

import duckdb
import geopandas as gpd
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from shapely import wkb
from sklearn.ensemble import IsolationForest

from project_branding import (
    BRAND_COLORS,
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
    BRAND_MAP_COLORS,
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = BRAND_COLOR_SEQUENCE

IF_REPRESENTATION_COLOR_MAP = {
    "PCA": BRAND_COLORS["dark_teal"],
    "UMAP": BRAND_COLORS["terracotta"],
}

IF_FRAMEWORK_COLOR_MAP = {
    "Isolation Forest": BRAND_COLORS["dark_teal"],
    "DBSCAN": BRAND_COLORS["terracotta"],
}

IF_OVERLAP_COLOR_MAP = {
    "IF only": BRAND_COLORS["dark_teal"],
    "DBSCAN only": BRAND_COLORS["terracotta"],
    "Shared": BRAND_COLORS["seafoam"],
}

IF_STABILITY_SCALE = [
    [0.00, BRAND_COLORS["terracotta"]],
    [0.50, BRAND_COLORS["pale_peach"]],
    [0.75, BRAND_COLORS["seafoam"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]

IF_MAP_SCALE = [
    [0.00, BRAND_COLORS["ice"]],
    [0.35, BRAND_COLORS["pale_peach"]],
    [0.70, BRAND_COLORS["terracotta"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]


def apply_if_brand_layout(fig, title=None, width=None, height=None):
    fig.update_layout(
        template="plotly_white",
        paper_bgcolor="white",
        plot_bgcolor=BRAND_COLORS["ice"],
        font=dict(color=BRAND_COLORS["dark_teal"]),
        width=width,
        height=height,
    )

    if title is not None:
        fig.update_layout(
            title=dict(
                text=title,
                font=dict(color=BRAND_COLORS["dark_teal"]),
            )
        )

    fig.update_xaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont=dict(color=BRAND_COLORS["dark_teal"]),
        title_font=dict(color=BRAND_COLORS["dark_teal"]),
    )
    fig.update_yaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont=dict(color=BRAND_COLORS["dark_teal"]),
        title_font=dict(color=BRAND_COLORS["dark_teal"]),
    )

    return fig

In [2]:
# ---------------------------------------------------------------------
# Define project directories and notebook output locations
# ---------------------------------------------------------------------

PROJECT_ROOT = Path("/datasets/_deepnote_work")
PIPELINE_DATA_DIR = PROJECT_ROOT / "pipeline_data"

FINAL_311_DIR = PIPELINE_DATA_DIR / "3.1.1.final_tables"
FINAL_321_DIR = PIPELINE_DATA_DIR / "3.2.1.final_tables"
FINAL_322_DIR = PIPELINE_DATA_DIR / "3.2.2.final_tables"
FINAL_324_DIR = PIPELINE_DATA_DIR / "3.2.4.final_tables"
FINAL_331_DIR = PIPELINE_DATA_DIR / "3.3.1.final_tables"
FINAL_332_DIR = PIPELINE_DATA_DIR / "3.3.2.final_tables"

INTERMEDIATE_333_DIR = PIPELINE_DATA_DIR / "3.3.3.intermediate_tables"
FINAL_333_DIR = PIPELINE_DATA_DIR / "3.3.3.final_tables"

INTERMEDIATE_333_DIR.mkdir(parents=True, exist_ok=True)
FINAL_333_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# ---------------------------------------------------------------------
# Define inherited anomaly-framework defaults and core columns
# ---------------------------------------------------------------------

MODEL_FEATURE_SET_NAMES = ["bus", "subway", "taxi", "fhvhv", "multimodal"]
ISOLATION_FOREST_REPRESENTATION_NAMES = ["pca_80pct", "umap_pca_to_umap_10d"]

MODEL_ID_COLUMN = "modeling_row_id"
DATE_COLUMN = "date"
TEMPORAL_BUCKET_COLUMN = "temporal_bucket"
PRE_POST_CP_COLUMN = "pre_post_cp"
ZONE_ID_COLUMN = "taxi_zone_id"
ZONE_COLUMN = "zone"
BOROUGH_COLUMN = "borough"
POLICY_GEOGRAPHY_COLUMN = "policy_geography_class"

DEFAULT_METADATA_COLUMNS = [
    MODEL_ID_COLUMN,
    DATE_COLUMN,
    TEMPORAL_BUCKET_COLUMN,
    PRE_POST_CP_COLUMN,
    ZONE_ID_COLUMN,
    ZONE_COLUMN,
    BOROUGH_COLUMN,
    POLICY_GEOGRAPHY_COLUMN,
]

SHARED_COMPARISON_UNIVERSE_NAME = "same_temporal_bucket_and_policy_geography"
SHARED_COMPARISON_GROUPING_COLUMNS = [
    TEMPORAL_BUCKET_COLUMN,
    POLICY_GEOGRAPHY_COLUMN,
]

TAXI_PCA_HANDLING_DECISION = "split_prepost_period"

REPRESENTATION_ROLE_BY_NAME = {
    "pca_80pct": "primary_linear_representation",
    "umap_pca_to_umap_10d": "nonlinear_complement_representation",
}

In [4]:
# ---------------------------------------------------------------------
# Define candidate contamination grid and notebook rebuild toggles
# ---------------------------------------------------------------------

ISOLATION_FOREST_CONTAMINATION_GRID = [
    0.005,
    0.010,
    0.015,
    0.020,
    0.030,
    0.040,
    0.050,
]

ISOLATION_FOREST_RANDOM_STATE = 42
ISOLATION_FOREST_N_ESTIMATORS = 300
ISOLATION_FOREST_MAX_SAMPLES = "auto"
ISOLATION_FOREST_MAX_FEATURES = 1.0

REPRESENTATIVE_INSPECTION_ROWS_PER_FEATURE_SET = 3

REBUILD_IF_INPUT_SUMMARIES = False
REBUILD_IF_CONTAMINATION_CALIBRATION = False
REBUILD_IF_STABILITY_DIAGNOSTICS = False
REBUILD_IF_DBSCAN_COMPARISONS = False
REBUILD_IF_HUMAN_REVIEW = False
REBUILD_IF_FINAL_OUTPUTS = False
WRITE_333_OUTPUTS = True

In [5]:
# ---------------------------------------------------------------------
# Define inherited asset paths from 3.3.1 and 3.3.2
# ---------------------------------------------------------------------

ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS = {
    "shared_representation_defaults": FINAL_331_DIR / "shared_representation_defaults.csv",
    "shared_calibration_defaults": FINAL_331_DIR / "shared_calibration_defaults.csv",
    "shared_comparison_universe_defaults": FINAL_331_DIR / "shared_comparison_universe_defaults.csv",
    "shared_framework_interpretation_defaults": FINAL_331_DIR / "shared_framework_interpretation_defaults.csv",
    "shared_framework_asset_manifest": FINAL_331_DIR / "shared_framework_asset_manifest.csv",
}

ROW_METADATA_PATHS_BY_SET = {
    feature_set: FINAL_311_DIR / f"{feature_set}_row_metadata.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

ANOMALY_CALIBRATION_ID_PATHS_BY_SET = {
    feature_set: FINAL_331_DIR / f"{feature_set}_anomaly_calibration_ids.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET = {
    feature_set: FINAL_331_DIR / f"{feature_set}_anomaly_calibration_metadata.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

PCA_80_FINAL_PATHS_BY_SET = {
    "bus": FINAL_321_DIR / "bus_pca_modeling_scores.parquet",
    "subway": FINAL_321_DIR / "subway_pca_modeling_scores.parquet",
    "taxi": FINAL_321_DIR / "taxi_pca_modeling_scores.parquet",
    "fhvhv": FINAL_321_DIR / "fhvhv_pca_modeling_scores.parquet",
    "multimodal": FINAL_321_DIR / "multimodal_pca_modeling_scores.parquet",
}

TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD = {
    "pre_cp": FINAL_322_DIR / "taxi_pre_cp_pca_80pct_modeling_scores.parquet",
    "post_cp": FINAL_322_DIR / "taxi_post_cp_pca_80pct_modeling_scores.parquet",
}

UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET = {
    feature_set: FINAL_324_DIR / f"{feature_set}_umap_pca_to_umap_10d.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

DBSCAN_SELECTED_CONFIGURATION_PATH = FINAL_332_DIR / "selected_dbscan_configurations.csv"
DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH = FINAL_332_DIR / "dbscan_anomaly_export_manifest.csv"
DBSCAN_FINAL_ROW_LEVEL_MANIFEST_PATH = DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH
DBSCAN_FINAL_ROW_LEVEL_DIR = FINAL_332_DIR / "dbscan_candidate_anomaly_row_level_parts"

IF_SPATIAL_FOOTPRINT_SUMMARY_PATH = (
    INTERMEDIATE_333_DIR / "if_spatial_footprint_summary.parquet"
)
IF_ZONE_MAP_SUMMARY_PATH = (
    INTERMEDIATE_333_DIR / "if_zone_map_summary.parquet"
)

In [6]:
# ---------------------------------------------------------------------
# Define 3.3.3 intermediate and final output paths
# ---------------------------------------------------------------------

IF_INPUT_ASSET_SUMMARY_PATH = INTERMEDIATE_333_DIR / "if_input_asset_summary.parquet"
IF_CONTAMINATION_SWEEP_SUMMARY_PATH = INTERMEDIATE_333_DIR / "if_contamination_sweep_summary.parquet"
IF_CONTAMINATION_SWEEP_DETAIL_PATH = INTERMEDIATE_333_DIR / "if_contamination_sweep_detail.parquet"
IF_SCORE_PROFILE_PATH = INTERMEDIATE_333_DIR / "if_score_profile.parquet"
IF_STABILITY_SUMMARY_PATH = INTERMEDIATE_333_DIR / "if_stability_summary.parquet"
IF_REPRESENTATION_COMPARISON_PATH = INTERMEDIATE_333_DIR / "if_representation_comparison.parquet"
IF_DBSCAN_COMPARISON_PATH = INTERMEDIATE_333_DIR / "if_dbscan_comparison.parquet"
IF_REPRESENTATIVE_INSPECTION_ROWS_PATH = INTERMEDIATE_333_DIR / "if_representative_inspection_rows.parquet"
IF_REPRESENTATIVE_INSPECTION_DRIVERS_PATH = INTERMEDIATE_333_DIR / "if_representative_inspection_drivers.parquet"

IF_SELECTED_CONFIGURATION_PATH = FINAL_333_DIR / "selected_isolation_forest_configurations.csv"
IF_ANOMALY_EXPORT_MANIFEST_PATH = FINAL_333_DIR / "isolation_forest_anomaly_export_manifest.csv"
IF_FINAL_ROW_LEVEL_DIR = FINAL_333_DIR / "isolation_forest_candidate_anomaly_row_level_parts"

# New: cache the full retained PCA modeling panels so final export is rerunnable
# and does not need to keep rebuilding the same merged panel after crashes.
IF_FINAL_PREPARED_PANELS_DIR = INTERMEDIATE_333_DIR / "if_final_prepared_panels"

# New: write one temporary row-level export per modality before assembling the
# final manifest, so interrupted runs can resume cleanly.
IF_FINAL_ROW_LEVEL_TEMP_DIR = INTERMEDIATE_333_DIR / "if_final_row_level_temp_parts"

IF_FINAL_ROW_LEVEL_DIR.mkdir(parents=True, exist_ok=True)
IF_FINAL_PREPARED_PANELS_DIR.mkdir(parents=True, exist_ok=True)
IF_FINAL_ROW_LEVEL_TEMP_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
# ---------------------------------------------------------------------
# Define general helper functions used throughout the notebook
# ---------------------------------------------------------------------

def sql_path(path_like):
    return str(Path(path_like)).replace("'", "''")


def duckdb_identifier(identifier):
    return '"' + str(identifier).replace('"', '""') + '"'


def should_rebuild(flag, output_path):
    return flag or not Path(output_path).exists()


def load_parquet_columns(parquet_path):
    return pd.read_parquet(parquet_path, engine="pyarrow").columns.tolist()


def load_shared_framework_table(table_key):
    table_path = ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS[table_key]
    if not table_path.exists():
        raise FileNotFoundError(f"Missing shared framework asset: {table_path}")
    if table_path.suffix.lower() == ".csv":
        return pd.read_csv(table_path)
    return pd.read_parquet(table_path)


def get_representation_path(feature_set, representation_name, row_metadata=None):
    if representation_name == "umap_pca_to_umap_10d":
        return UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET[feature_set]

    if representation_name != "pca_80pct":
        raise ValueError(f"Unsupported representation: {representation_name}")

    if feature_set != "taxi" or TAXI_PCA_HANDLING_DECISION != "split_prepost_period":
        return PCA_80_FINAL_PATHS_BY_SET[feature_set]

    if row_metadata is None:
        raise ValueError("Taxi PCA path resolution requires row metadata with pre/post CP labels.")

    if PRE_POST_CP_COLUMN not in row_metadata.columns:
        raise KeyError(f"Expected '{PRE_POST_CP_COLUMN}' in taxi row metadata.")

    unique_periods = pd.Index(row_metadata[PRE_POST_CP_COLUMN].dropna().unique()).tolist()
    if len(unique_periods) != 1:
        raise ValueError(
            "Taxi PCA path resolution expects one policy period at a time; "
            f"found {unique_periods}."
        )

    return TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD[unique_periods[0]]

In [8]:
# ---------------------------------------------------------------------
# Define shared asset-loading helpers for calibration and comparison work
# ---------------------------------------------------------------------

def get_numeric_representation_columns(feature_set, representation_name, representation_path):
    all_columns = load_parquet_columns(representation_path)

    if representation_name == "pca_80pct":
        numeric_columns = [column for column in all_columns if str(column).startswith("PC")]
    elif representation_name == "umap_pca_to_umap_10d":
        numeric_columns = [column for column in all_columns if str(column).startswith("umap_")]
    else:
        raise ValueError(f"Unsupported representation: {representation_name}")

    if not numeric_columns:
        raise ValueError(
            f"No numeric representation columns found for {feature_set} / {representation_name} "
            f"at {representation_path}."
        )

    return numeric_columns


def load_calibration_metadata(feature_set):
    calibration_ids_path = ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set]
    calibration_metadata_path = ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set]
    row_metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set]

    if not calibration_ids_path.exists():
        raise FileNotFoundError(f"Missing calibration IDs for {feature_set}: {calibration_ids_path}")
    if not calibration_metadata_path.exists():
        raise FileNotFoundError(
            f"Missing calibration metadata for {feature_set}: {calibration_metadata_path}"
        )
    if not row_metadata_path.exists():
        raise FileNotFoundError(f"Missing row metadata for {feature_set}: {row_metadata_path}")

    calibration_ids_df = pd.read_parquet(calibration_ids_path)
    calibration_metadata_df = pd.read_parquet(calibration_metadata_path)

    merged_df = calibration_ids_df.merge(
        calibration_metadata_df,
        on=MODEL_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )

    missing_default_columns = [
        column for column in DEFAULT_METADATA_COLUMNS if column not in merged_df.columns
    ]
    if missing_default_columns:
        row_metadata_df = pd.read_parquet(
            row_metadata_path,
            columns=[MODEL_ID_COLUMN, *missing_default_columns],
        )
        merged_df = merged_df.merge(
            row_metadata_df,
            on=MODEL_ID_COLUMN,
            how="left",
            validate="one_to_one",
        )

    return merged_df


def load_representation_rows(feature_set, representation_name, metadata_df):
    representation_path = get_representation_path(
        feature_set,
        representation_name,
        row_metadata=metadata_df,
    )
    numeric_columns = get_numeric_representation_columns(
        feature_set,
        representation_name,
        representation_path,
    )

    representation_df = pd.read_parquet(
        representation_path,
        columns=[MODEL_ID_COLUMN, *numeric_columns],
    )

    merged_df = metadata_df.merge(
        representation_df,
        on=MODEL_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )

    return merged_df, numeric_columns, representation_path


def get_final_pca_representation_df(feature_set, metadata_df):
    representation_df, numeric_columns, representation_path = load_representation_rows(
        feature_set=feature_set,
        representation_name="pca_80pct",
        metadata_df=metadata_df,
    )
    return representation_df, numeric_columns, representation_path


def get_final_umap_representation_df(feature_set, metadata_df):
    representation_df, numeric_columns, representation_path = load_representation_rows(
        feature_set=feature_set,
        representation_name="umap_pca_to_umap_10d",
        metadata_df=metadata_df,
    )
    return representation_df, numeric_columns, representation_path

The main work here is contamination calibration, stability review, representation sensitivity, and a lightweight comparison against DBSCAN\. This notebook should not reopen baseline\-selection or feature\-selection decisions from 3\.3\.1, and it should not choose a project\-wide winning representation\. Instead, it should determine which Isolation Forest branches are coherent enough to carry forward into downstream anomaly\-framework comparison\.

## 3\.3\.3\.1 Load Shared Anomaly Framework Assets

Load the shared calibration panel, downstream PCA and UMAP representations, settled comparison\-universe defaults, metadata assets, and retained DBSCAN outputs that Isolation Forest will inherit for calibration and later comparison\.

Start with a compact inheritance inventory\. Before loading any anomaly model, we want to verify that the framework decisions from 3\.3\.1 and the retained DBSCAN outputs from 3\.3\.2 are actually present on disk and ready to be reused here\.

In [9]:
# ---------------------------------------------------------------------
# Inventory inherited anomaly-framework assets and retained DBSCAN handoff assets
# ---------------------------------------------------------------------

if_framework_asset_inventory_records = []

for asset_name, asset_path in ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS.items():
    if_framework_asset_inventory_records.append(
        {
            "asset_group": "3.3.1 shared framework",
            "asset_name": asset_name,
            "path": str(asset_path),
            "path_exists": asset_path.exists(),
        }
    )

for feature_set in MODEL_FEATURE_SET_NAMES:
    if_framework_asset_inventory_records.extend(
        [
            {
                "asset_group": "3.3.1 calibration metadata",
                "asset_name": f"{feature_set}_anomaly_calibration_ids",
                "path": str(ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set]),
                "path_exists": ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set].exists(),
            },
            {
                "asset_group": "3.3.1 calibration metadata",
                "asset_name": f"{feature_set}_anomaly_calibration_metadata",
                "path": str(ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set]),
                "path_exists": ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set].exists(),
            },
            {
                "asset_group": "3.2.x representation branch",
                "asset_name": f"{feature_set}_pca_80pct",
                "path": str(PCA_80_FINAL_PATHS_BY_SET[feature_set]),
                "path_exists": PCA_80_FINAL_PATHS_BY_SET[feature_set].exists(),
            },
            {
                "asset_group": "3.2.x representation branch",
                "asset_name": f"{feature_set}_umap_pca_to_umap_10d",
                "path": str(UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET[feature_set]),
                "path_exists": UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET[feature_set].exists(),
            },
        ]
    )

for policy_period, asset_path in TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD.items():
    if_framework_asset_inventory_records.append(
        {
            "asset_group": "3.2.2 taxi split PCA",
            "asset_name": f"taxi_{policy_period}_pca_80pct",
            "path": str(asset_path),
            "path_exists": asset_path.exists(),
        }
    )

for asset_name, asset_path in {
    "selected_dbscan_configurations": DBSCAN_SELECTED_CONFIGURATION_PATH,
    "dbscan_anomaly_export_manifest": DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH,
    "dbscan_final_row_level_dir": DBSCAN_FINAL_ROW_LEVEL_DIR,
}.items():
    if_framework_asset_inventory_records.append(
        {
            "asset_group": "3.3.2 DBSCAN handoff",
            "asset_name": asset_name,
            "path": str(asset_path),
            "path_exists": asset_path.exists(),
        }
    )

if_framework_asset_inventory_df = pd.DataFrame(if_framework_asset_inventory_records)

if_framework_asset_inventory_summary_df = (
    if_framework_asset_inventory_df.groupby("asset_group", dropna=False)
    .agg(
        assets_expected=("asset_name", "size"),
        assets_found=("path_exists", "sum"),
    )
    .reset_index()
)
if_framework_asset_inventory_summary_df["status"] = np.where(
    if_framework_asset_inventory_summary_df["assets_expected"]
    == if_framework_asset_inventory_summary_df["assets_found"],
    "pass",
    "review",
)

display(if_framework_asset_inventory_summary_df)
display(if_framework_asset_inventory_df)

assert if_framework_asset_inventory_df["path_exists"].all(), (
    "One or more inherited anomaly-framework assets is missing."
)

,asset_group,assets_expected,assets_found,status
0,3.2.2 taxi split PCA,2,2,pass
1,3.2.x representation branch,10,10,pass
2,3.3.1 calibration metadata,10,10,pass
3,3.3.1 shared framework,5,5,pass
4,3.3.2 DBSCAN handoff,3,3,pass


,asset_group,asset_name,path,path_exists
0,3.3.1 shared framework,shared_representation_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/shared_representation_defaults.csv,True
1,3.3.1 shared framework,shared_calibration_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/shared_calibration_defaults.csv,True
2,3.3.1 shared framework,shared_comparison_universe_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/shared_comparison_universe_defaults.csv,True
3,3.3.1 shared framework,shared_framework_interpretation_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/shared_framework_interpretation_defaults.csv,True
4,3.3.1 shared framework,shared_framework_asset_manifest,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/shared_framework_asset_manifest.csv,True
5,3.3.1 calibration metadata,bus_anomaly_calibration_ids,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/bus_anomaly_calibration_ids.parquet,True
6,3.3.1 calibration metadata,bus_anomaly_calibration_metadata,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/bus_anomaly_calibration_metadata.parquet,True
7,3.2.x representation branch,bus_pca_80pct,/datasets/_deepnote_work/pipeline_data/3.2.1.final_tables/bus_pca_modeling_scores.parquet,True
8,3.2.x representation branch,bus_umap_pca_to_umap_10d,/datasets/_deepnote_work/pipeline_data/3.2.4.final_tables/bus_umap_pca_to_umap_10d.parquet,True
9,3.3.1 calibration metadata,subway_anomaly_calibration_ids,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/subway_anomaly_calibration_ids.parquet,True


Findings\. This inventory block is just a readiness check, and the expected result is simple: all inherited framework tables, calibration assets, retained representation branches, Taxi split\-PCA assets, and DBSCAN handoff files should be present before Isolation Forest calibration begins\. If the block passes, then 3\.3\.3 is starting from a complete inherited framework rather than silently rebuilding missing pieces\.

Next, load the inherited defaults and prove that the shared calibration panel really lines up with both downstream representation branches\. The main thing to check here is row alignment: every calibration row should resolve cleanly into the PCA branch and the UMAP branch for each modality, with no dropped rows and no branch\-specific missing representation values\.

In [10]:
# ---------------------------------------------------------------------
# Load inherited framework defaults and validate row alignment to PCA and UMAP
# ---------------------------------------------------------------------

shared_representation_defaults_df = load_shared_framework_table(
    "shared_representation_defaults"
)
shared_calibration_defaults_df = load_shared_framework_table(
    "shared_calibration_defaults"
)
shared_comparison_universe_defaults_df = load_shared_framework_table(
    "shared_comparison_universe_defaults"
)
shared_framework_interpretation_defaults_df = load_shared_framework_table(
    "shared_framework_interpretation_defaults"
)
dbscan_selected_configurations_df = pd.read_csv(DBSCAN_SELECTED_CONFIGURATION_PATH)
dbscan_anomaly_export_manifest_df = pd.read_csv(DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH)


def ensure_default_metadata_columns(feature_set, metadata_df):
    missing_default_columns = [
        column for column in DEFAULT_METADATA_COLUMNS if column not in metadata_df.columns
    ]
    if not missing_default_columns:
        return metadata_df

    row_metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set]
    row_metadata_df = pd.read_parquet(
        row_metadata_path,
        columns=[MODEL_ID_COLUMN, *missing_default_columns],
    )
    return metadata_df.merge(
        row_metadata_df,
        on=MODEL_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )


def load_if_branch_rows(feature_set, representation_name):
    metadata_df = load_calibration_metadata(feature_set)
    metadata_df = ensure_default_metadata_columns(feature_set, metadata_df)

    if (
        feature_set == "taxi"
        and representation_name == "pca_80pct"
        and TAXI_PCA_HANDLING_DECISION == "split_prepost_period"
    ):
        if PRE_POST_CP_COLUMN not in metadata_df.columns:
            raise KeyError(
                f"'{PRE_POST_CP_COLUMN}' is still missing for taxi after metadata recovery."
            )

        taxi_branch_parts = []
        expected_numeric_columns = None

        for policy_period, period_metadata_df in metadata_df.groupby(
            PRE_POST_CP_COLUMN,
            dropna=False,
            sort=False,
        ):
            period_path = TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD[policy_period]
            period_numeric_columns = get_numeric_representation_columns(
                feature_set=feature_set,
                representation_name=representation_name,
                representation_path=period_path,
            )
            if expected_numeric_columns is None:
                expected_numeric_columns = period_numeric_columns
            else:
                assert expected_numeric_columns == period_numeric_columns, (
                    "Taxi split PCA exports do not share the same component columns."
                )

            period_representation_df = pd.read_parquet(
                period_path,
                columns=[MODEL_ID_COLUMN, *period_numeric_columns],
            )
            taxi_branch_parts.append(
                period_metadata_df.merge(
                    period_representation_df,
                    on=MODEL_ID_COLUMN,
                    how="left",
                    validate="one_to_one",
                )
            )

        merged_df = pd.concat(taxi_branch_parts, ignore_index=True)
        merged_df = merged_df.sort_values(MODEL_ID_COLUMN).reset_index(drop=True)
        return merged_df, expected_numeric_columns

    merged_df, numeric_columns, _ = load_representation_rows(
        feature_set=feature_set,
        representation_name=representation_name,
        metadata_df=metadata_df,
    )
    merged_df = merged_df.sort_values(MODEL_ID_COLUMN).reset_index(drop=True)
    return merged_df, numeric_columns


if_branch_alignment_records = []

for feature_set in MODEL_FEATURE_SET_NAMES:
    calibration_metadata_df = load_calibration_metadata(feature_set)
    calibration_metadata_df = ensure_default_metadata_columns(
        feature_set,
        calibration_metadata_df,
    )
    expected_modeling_rows = calibration_metadata_df[MODEL_ID_COLUMN].nunique()

    for representation_name in ISOLATION_FOREST_REPRESENTATION_NAMES:
        branch_df, numeric_columns = load_if_branch_rows(feature_set, representation_name)
        rows_with_any_missing = int(branch_df[numeric_columns].isna().any(axis=1).sum())
        missing_branch_rows = int(branch_df[numeric_columns].isna().all(axis=1).sum())
        duplicate_modeling_rows = int(
            branch_df[MODEL_ID_COLUMN].duplicated().sum()
        )

        if_branch_alignment_records.append(
            {
                "feature_set": feature_set,
                "representation_name": representation_name,
                "rows_in_panel": len(branch_df),
                "distinct_modeling_rows": branch_df[MODEL_ID_COLUMN].nunique(),
                "expected_modeling_rows": expected_modeling_rows,
                "representation_columns": len(numeric_columns),
                "rows_with_any_missing": rows_with_any_missing,
                "rows_missing_all_representation_values": missing_branch_rows,
                "duplicate_modeling_rows": duplicate_modeling_rows,
                "status": (
                    "pass"
                    if len(branch_df) == expected_modeling_rows
                    and branch_df[MODEL_ID_COLUMN].nunique() == expected_modeling_rows
                    and rows_with_any_missing == 0
                    and missing_branch_rows == 0
                    and duplicate_modeling_rows == 0
                    else "review"
                ),
            }
        )

if_branch_alignment_df = pd.DataFrame(if_branch_alignment_records)

if_shared_framework_default_summary_df = pd.DataFrame(
    [
        {
            "shared_comparison_universe": SHARED_COMPARISON_UNIVERSE_NAME,
            "comparison_grouping_columns": ", ".join(SHARED_COMPARISON_GROUPING_COLUMNS),
            "taxi_pca_handling_decision": TAXI_PCA_HANDLING_DECISION,
            "representation_branches_retained_for_if": ", ".join(ISOLATION_FOREST_REPRESENTATION_NAMES),
        }
    ]
)

display(if_shared_framework_default_summary_df)
display(
    shared_representation_defaults_df[
        [
            column
            for column in [
                "feature_set",
                "representation_name",
                "advance_downstream",
                "representation_role",
                "downstream_status",
            ]
            if column in shared_representation_defaults_df.columns
        ]
    ]
)
display(
    shared_comparison_universe_defaults_df[
        [
            column
            for column in [
                "feature_set",
                "comparison_universe",
                "downstream_status",
                "downstream_role",
            ]
            if column in shared_comparison_universe_defaults_df.columns
        ]
    ]
)
display(if_branch_alignment_df)

assert if_branch_alignment_df["status"].eq("pass").all(), (
    "One or more inherited representation branches is not aligned to the shared calibration panel."
)

,shared_comparison_universe,comparison_grouping_columns,taxi_pca_handling_decision,representation_branches_retained_for_if
0,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",split_prepost_period,"pca_80pct, umap_pca_to_umap_10d"


,feature_set,representation_name,advance_downstream,representation_role,downstream_status
0,bus,raw_scaled,False,reference_only,retired_as_primary_branch
1,bus,pca_80pct,True,primary_linear_representation,shared_default
2,bus,umap_pca_to_umap_10d,True,nonlinear_complement_representation,shared_default
3,subway,raw_scaled,False,reference_only,retired_as_primary_branch
4,subway,pca_80pct,True,primary_linear_representation,shared_default
5,subway,umap_pca_to_umap_10d,True,nonlinear_complement_representation,shared_default
6,taxi,raw_scaled,False,reference_only,retired_as_primary_branch
7,taxi,pca_80pct,True,primary_linear_representation,shared_default
8,taxi,umap_pca_to_umap_10d,True,nonlinear_complement_representation,shared_default
9,fhvhv,raw_scaled,False,reference_only,retired_as_primary_branch


,feature_set,comparison_universe,downstream_status,downstream_role
0,bus,same_temporal_bucket_and_policy_geography,shared_default,primary_scoring_baseline
1,subway,same_temporal_bucket_and_policy_geography,shared_default,primary_scoring_baseline
2,taxi,same_temporal_bucket_and_policy_geography,shared_default,primary_scoring_baseline
3,taxi,same_temporal_bucket_and_pre_post_cp,targeted_diagnostic,taxi_policy_sanity_check
4,fhvhv,same_temporal_bucket_and_policy_geography,shared_default,primary_scoring_baseline
5,multimodal,same_temporal_bucket_and_policy_geography,shared_default,primary_scoring_baseline


,feature_set,representation_name,rows_in_panel,distinct_modeling_rows,expected_modeling_rows,representation_columns,rows_with_any_missing,rows_missing_all_representation_values,duplicate_modeling_rows,status
0,bus,pca_80pct,50000,50000,50000,14,0,0,0,pass
1,bus,umap_pca_to_umap_10d,50000,50000,50000,10,0,0,0,pass
2,subway,pca_80pct,50000,50000,50000,11,0,0,0,pass
3,subway,umap_pca_to_umap_10d,50000,50000,50000,10,0,0,0,pass
4,taxi,pca_80pct,50000,50000,50000,15,0,0,0,pass
5,taxi,umap_pca_to_umap_10d,50000,50000,50000,10,0,0,0,pass
6,fhvhv,pca_80pct,50000,50000,50000,13,0,0,0,pass
7,fhvhv,umap_pca_to_umap_10d,50000,50000,50000,10,0,0,0,pass
8,multimodal,pca_80pct,50000,50000,50000,46,0,0,0,pass
9,multimodal,umap_pca_to_umap_10d,50000,50000,50000,10,0,0,0,pass


Findings\. The inherited calibration panel is successfully aligned to both retained Isolation Forest representation branches for every modality, including Taxi’s split pre/post PCA handling\. In practice, that means every calibration row resolves cleanly into the downstream PCA or UMAP feature space with no duplicate modeling rows and no branch\-specific missing representation values, so later calibration work is comparing representations on the same shared panel rather than on mismatched samples\.

Finally, confirm that the retained DBSCAN branch is available as a comparison surface\. Isolation Forest does not inherit DBSCAN’s assumptions, but it does inherit DBSCAN’s exported anomaly outputs as an external reference later in the notebook for comparison of results\.

In [11]:
# ---------------------------------------------------------------------
# Validate the retained DBSCAN handoff for later framework comparison
# ---------------------------------------------------------------------

if_dbscan_handoff_validation_df = pd.DataFrame(
    [
        {
            "selected_dbscan_configuration_rows": len(dbscan_selected_configurations_df),
            "selected_dbscan_feature_sets": dbscan_selected_configurations_df["feature_set"].nunique()
            if "feature_set" in dbscan_selected_configurations_df.columns
            else np.nan,
            "dbscan_export_manifest_rows": len(dbscan_anomaly_export_manifest_df),
            "dbscan_export_feature_sets": dbscan_anomaly_export_manifest_df["feature_set"].nunique()
            if "feature_set" in dbscan_anomaly_export_manifest_df.columns
            else np.nan,
            "dbscan_row_level_dir_exists": DBSCAN_FINAL_ROW_LEVEL_DIR.exists(),
            "status": (
                "pass"
                if len(dbscan_selected_configurations_df) > 0
                and len(dbscan_anomaly_export_manifest_df) > 0
                and DBSCAN_FINAL_ROW_LEVEL_DIR.exists()
                and (
                    "feature_set" not in dbscan_selected_configurations_df.columns
                    or dbscan_selected_configurations_df["feature_set"].nunique()
                    == len(MODEL_FEATURE_SET_NAMES)
                )
                and (
                    "feature_set" not in dbscan_anomaly_export_manifest_df.columns
                    or dbscan_anomaly_export_manifest_df["feature_set"].nunique()
                    == len(MODEL_FEATURE_SET_NAMES)
                )
                else "review"
            ),
        }
    ]
)

display(
    dbscan_selected_configurations_df[
        [
            column
            for column in [
                "feature_set",
                "representation_name",
                "min_samples",
                "eps_band",
                "eps_value",
            ]
            if column in dbscan_selected_configurations_df.columns
        ]
    ]
)
display(if_dbscan_handoff_validation_df)

assert if_dbscan_handoff_validation_df["status"].eq("pass").all(), (
    "The retained DBSCAN anomaly handoff is incomplete."
)

,feature_set,representation_name,min_samples,eps_band,eps_value
0,bus,pca_80pct,15,balanced,5.281448
1,fhvhv,pca_80pct,15,balanced,5.116225
2,multimodal,pca_80pct,15,balanced,13.557635
3,subway,pca_80pct,15,balanced,2.744738
4,taxi,pca_80pct,20,tight,4.812092


,selected_dbscan_configuration_rows,selected_dbscan_feature_sets,dbscan_export_manifest_rows,dbscan_export_feature_sets,dbscan_row_level_dir_exists,status
0,5,5,5,5,True,pass


Findings\. The retained DBSCAN branch is fully available as a later comparison surface: one selected DBSCAN configuration per modality, a populated DBSCAN export manifest, and an existing row\-level output directory\. If this block passes, then later Isolation Forest versus DBSCAN comparisons can proceed as inherited framework comparisons rather than as ad hoc reruns\.

## 3\.3\.3\.2 Calibrate Candidate Contamination Rates

Isolation Forest’s main tuning lever is contamination, which controls how much of the calibration panel gets treated as anomalous\. This section starts by defining the contamination ranges we want to test across both retained representation branches, then runs a lightweight sweep to see how anomaly prevalence and anomaly\-score behavior change as contamination increases\. The goal is to narrow the search space to contamination regions that still look sparse, separated, and interpretable enough to carry forward into stability testing\.

### Define candidate contamination grid and output paths

Start by making the sweep explicit\. Before we run anything, we want a compact inventory of the representation branches, contamination values, and output tables that this calibration section will use\. This keeps the sweep auditable and makes it easy to confirm that PCA and UMAP are both being tested across the same shared framework\.

In [12]:
# ---------------------------------------------------------------------
# Define candidate contamination grid and section-specific output paths
# ---------------------------------------------------------------------

IF_CONTAMINATION_SWEEP_CANDIDATES = pd.DataFrame(
    [
        {
            "feature_set": feature_set,
            "representation_name": representation_name,
            "representation_role": REPRESENTATION_ROLE_BY_NAME[representation_name],
            "contamination": contamination,
            "comparison_universe": SHARED_COMPARISON_UNIVERSE_NAME,
            "comparison_grouping_columns": ", ".join(SHARED_COMPARISON_GROUPING_COLUMNS),
        }
        for feature_set in MODEL_FEATURE_SET_NAMES
        for representation_name in ISOLATION_FOREST_REPRESENTATION_NAMES
        for contamination in ISOLATION_FOREST_CONTAMINATION_GRID
    ]
)

IF_CONTAMINATION_SWEEP_OUTPUTS_DF = pd.DataFrame(
    [
        {
            "output_name": "if_input_asset_summary",
            "path": str(IF_INPUT_ASSET_SUMMARY_PATH),
        },
        {
            "output_name": "if_contamination_sweep_summary",
            "path": str(IF_CONTAMINATION_SWEEP_SUMMARY_PATH),
        },
        {
            "output_name": "if_contamination_sweep_detail",
            "path": str(IF_CONTAMINATION_SWEEP_DETAIL_PATH),
        },
        {
            "output_name": "if_score_profile",
            "path": str(IF_SCORE_PROFILE_PATH),
        },
    ]
)

display(
    IF_CONTAMINATION_SWEEP_CANDIDATES.groupby(
        ["feature_set", "representation_name"], dropna=False
    )
    .agg(
        contamination_candidates=("contamination", "size"),
        contamination_min=("contamination", "min"),
        contamination_max=("contamination", "max"),
    )
    .reset_index()
)

display(IF_CONTAMINATION_SWEEP_OUTPUTS_DF)

,feature_set,representation_name,contamination_candidates,contamination_min,contamination_max
0,bus,pca_80pct,7,0.005,0.05
1,bus,umap_pca_to_umap_10d,7,0.005,0.05
2,fhvhv,pca_80pct,7,0.005,0.05
3,fhvhv,umap_pca_to_umap_10d,7,0.005,0.05
4,multimodal,pca_80pct,7,0.005,0.05
5,multimodal,umap_pca_to_umap_10d,7,0.005,0.05
6,subway,pca_80pct,7,0.005,0.05
7,subway,umap_pca_to_umap_10d,7,0.005,0.05
8,taxi,pca_80pct,7,0.005,0.05
9,taxi,umap_pca_to_umap_10d,7,0.005,0.05


,output_name,path
0,if_input_asset_summary,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_input_asset_summary.parquet
1,if_contamination_sweep_summary,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_contamination_sweep_summary.parquet
2,if_contamination_sweep_detail,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_contamination_sweep_detail.parquet
3,if_score_profile,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_score_profile.parquet


Findings\. This setup block defines one shared contamination grid for every modality and for both retained Isolation Forest representation branches, so the sweep will compare PCA and UMAP on the same inherited framework rather than on different candidate ranges or ad hoc outputs\.

Now add a quick validation block so we can verify that the sweep inventory is complete before we run the expensive part\.

In [13]:
# ---------------------------------------------------------------------
# Validate contamination-sweep coverage before execution
# ---------------------------------------------------------------------

if_contamination_grid_validation_df = pd.DataFrame(
    [
        {
            "feature_sets_covered": IF_CONTAMINATION_SWEEP_CANDIDATES["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "representations_covered": IF_CONTAMINATION_SWEEP_CANDIDATES["representation_name"].nunique(),
            "expected_representations": len(ISOLATION_FOREST_REPRESENTATION_NAMES),
            "contamination_values_tested": IF_CONTAMINATION_SWEEP_CANDIDATES["contamination"].nunique(),
            "expected_contamination_values": len(ISOLATION_FOREST_CONTAMINATION_GRID),
            "candidate_rows": len(IF_CONTAMINATION_SWEEP_CANDIDATES),
            "expected_candidate_rows": (
                len(MODEL_FEATURE_SET_NAMES)
                * len(ISOLATION_FOREST_REPRESENTATION_NAMES)
                * len(ISOLATION_FOREST_CONTAMINATION_GRID)
            ),
            "status": (
                "pass"
                if IF_CONTAMINATION_SWEEP_CANDIDATES["feature_set"].nunique()
                == len(MODEL_FEATURE_SET_NAMES)
                and IF_CONTAMINATION_SWEEP_CANDIDATES["representation_name"].nunique()
                == len(ISOLATION_FOREST_REPRESENTATION_NAMES)
                and IF_CONTAMINATION_SWEEP_CANDIDATES["contamination"].nunique()
                == len(ISOLATION_FOREST_CONTAMINATION_GRID)
                and len(IF_CONTAMINATION_SWEEP_CANDIDATES)
                == (
                    len(MODEL_FEATURE_SET_NAMES)
                    * len(ISOLATION_FOREST_REPRESENTATION_NAMES)
                    * len(ISOLATION_FOREST_CONTAMINATION_GRID)
                )
                else "review"
            ),
        }
    ]
)

display(if_contamination_grid_validation_df)

assert if_contamination_grid_validation_df["status"].eq("pass").all(), (
    "The Isolation Forest contamination grid does not cover the expected "
    "feature-set, representation, and contamination combinations."
)

,feature_sets_covered,expected_feature_sets,representations_covered,expected_representations,contamination_values_tested,expected_contamination_values,candidate_rows,expected_candidate_rows,status
0,5,5,2,2,7,7,70,70,pass


Findings\. The contamination grid now fully covers all five modalities, both retained representation branches, and every contamination value in the shared candidate range, so the sweep can proceed as a complete calibration exercise rather than as a partial spot check\.

### Run a lightweight Isolation Forest contamination sweep

With the candidate grid defined, we can now run a lightweight calibration sweep across the shared anomaly framework\. Each feature\_set x representation x contamination combination is fit on the inherited calibration panel, and we record the basic outputs we care about for contamination calibration: anomaly prevalence, score behavior, and a compact summary of how sharply the flagged tail separates from the bulk\.
In this section, contamination is the assumed anomaly fraction passed into Isolation Forest\. Lower values force the model to carve out a smaller anomaly tail; higher values allow the anomaly surface to expand\. The question is not simply whether the model can flag anomalies, but whether the resulting tail remains sparse and well\-separated enough to be analytically useful\.

In [14]:
# ---------------------------------------------------------------------
# Run a lightweight Isolation Forest contamination sweep
# ---------------------------------------------------------------------

def ensure_if_default_metadata_columns(feature_set, metadata_df):
    missing_default_columns = [
        column for column in DEFAULT_METADATA_COLUMNS
        if column not in metadata_df.columns
    ]
    if not missing_default_columns:
        return metadata_df

    row_metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set]
    row_metadata_df = pd.read_parquet(
        row_metadata_path,
        columns=[MODEL_ID_COLUMN, *missing_default_columns],
    )

    return metadata_df.merge(
        row_metadata_df,
        on=MODEL_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )


def load_if_representation_rows(feature_set, representation_name):
    metadata_df = load_calibration_metadata(feature_set)
    metadata_df = ensure_if_default_metadata_columns(feature_set, metadata_df)

    if (
        feature_set == "taxi"
        and representation_name == "pca_80pct"
        and TAXI_PCA_HANDLING_DECISION == "split_prepost_period"
    ):
        if PRE_POST_CP_COLUMN not in metadata_df.columns:
            raise KeyError(
                f"'{PRE_POST_CP_COLUMN}' is still missing for taxi after metadata recovery."
            )

        taxi_branch_parts = []
        expected_numeric_columns = None

        for policy_period, period_metadata_df in metadata_df.groupby(
            PRE_POST_CP_COLUMN,
            dropna=False,
            sort=False,
        ):
            period_path = TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD[policy_period]
            period_numeric_columns = get_numeric_representation_columns(
                feature_set=feature_set,
                representation_name=representation_name,
                representation_path=period_path,
            )

            if expected_numeric_columns is None:
                expected_numeric_columns = period_numeric_columns
            else:
                assert expected_numeric_columns == period_numeric_columns, (
                    "Taxi split PCA exports do not share the same component columns."
                )

            period_representation_df = pd.read_parquet(
                period_path,
                columns=[MODEL_ID_COLUMN, *period_numeric_columns],
            )

            taxi_branch_parts.append(
                period_metadata_df.merge(
                    period_representation_df,
                    on=MODEL_ID_COLUMN,
                    how="left",
                    validate="one_to_one",
                )
            )

        merged_df = (
            pd.concat(taxi_branch_parts, ignore_index=True)
            .sort_values(MODEL_ID_COLUMN)
            .reset_index(drop=True)
        )
        return merged_df, expected_numeric_columns

    merged_df, numeric_columns, _ = load_representation_rows(
        feature_set=feature_set,
        representation_name=representation_name,
        metadata_df=metadata_df,
    )
    merged_df = merged_df.sort_values(MODEL_ID_COLUMN).reset_index(drop=True)
    return merged_df, numeric_columns


def run_isolation_forest_branch(
    feature_set,
    representation_name,
    contamination,
):
    branch_df, numeric_columns = load_if_representation_rows(
        feature_set=feature_set,
        representation_name=representation_name,
    )

    X = branch_df[numeric_columns].to_numpy(dtype=float)

    model = IsolationForest(
        n_estimators=ISOLATION_FOREST_N_ESTIMATORS,
        contamination=contamination,
        max_samples=ISOLATION_FOREST_MAX_SAMPLES,
        max_features=ISOLATION_FOREST_MAX_FEATURES,
        random_state=ISOLATION_FOREST_RANDOM_STATE,
        n_jobs=-1,
    )

    fit_start = perf_counter()
    model.fit(X)
    fit_seconds = perf_counter() - fit_start

    raw_prediction = model.predict(X)
    anomaly_flag = (raw_prediction == -1).astype(int)

    decision_score = model.decision_function(X)
    raw_score = model.score_samples(X)

    branch_result_df = branch_df[
        [
            MODEL_ID_COLUMN,
            DATE_COLUMN,
            TEMPORAL_BUCKET_COLUMN,
            PRE_POST_CP_COLUMN,
            POLICY_GEOGRAPHY_COLUMN,
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            ZONE_ID_COLUMN,
        ]
    ].copy()

    branch_result_df["feature_set"] = feature_set
    branch_result_df["representation_name"] = representation_name
    branch_result_df["contamination"] = contamination
    branch_result_df["if_anomaly_flag"] = anomaly_flag
    branch_result_df["if_decision_score"] = decision_score
    branch_result_df["if_raw_score"] = raw_score

    anomaly_mask = branch_result_df["if_anomaly_flag"].eq(1)
    normal_mask = ~anomaly_mask

    anomaly_share = float(anomaly_mask.mean())
    anomaly_rows = int(anomaly_mask.sum())
    normal_rows = int(normal_mask.sum())

    summary_record = {
        "feature_set": feature_set,
        "representation_name": representation_name,
        "contamination": contamination,
        "rows_evaluated": len(branch_result_df),
        "anomaly_rows": anomaly_rows,
        "normal_rows": normal_rows,
        "anomaly_share": anomaly_share,
        "decision_score_mean_anomaly": float(branch_result_df.loc[anomaly_mask, "if_decision_score"].mean()),
        "decision_score_mean_normal": float(branch_result_df.loc[normal_mask, "if_decision_score"].mean()),
        "decision_score_median_anomaly": float(branch_result_df.loc[anomaly_mask, "if_decision_score"].median()),
        "decision_score_median_normal": float(branch_result_df.loc[normal_mask, "if_decision_score"].median()),
        "raw_score_mean_anomaly": float(branch_result_df.loc[anomaly_mask, "if_raw_score"].mean()),
        "raw_score_mean_normal": float(branch_result_df.loc[normal_mask, "if_raw_score"].mean()),
        "fit_seconds": fit_seconds,
        "status": "pass",
    }

    detail_record = {
        "feature_set": feature_set,
        "representation_name": representation_name,
        "contamination": contamination,
        "decision_score_q01": float(branch_result_df["if_decision_score"].quantile(0.01)),
        "decision_score_q05": float(branch_result_df["if_decision_score"].quantile(0.05)),
        "decision_score_q25": float(branch_result_df["if_decision_score"].quantile(0.25)),
        "decision_score_q50": float(branch_result_df["if_decision_score"].quantile(0.50)),
        "decision_score_q75": float(branch_result_df["if_decision_score"].quantile(0.75)),
        "decision_score_q95": float(branch_result_df["if_decision_score"].quantile(0.95)),
        "decision_score_q99": float(branch_result_df["if_decision_score"].quantile(0.99)),
        "status": "pass",
    }

    return branch_result_df, summary_record, detail_record


if should_rebuild(REBUILD_IF_CONTAMINATION_CALIBRATION, IF_CONTAMINATION_SWEEP_SUMMARY_PATH):
    if_contamination_summary_records = []
    if_contamination_detail_records = []
    if_branch_result_parts = []

    total_runs = len(IF_CONTAMINATION_SWEEP_CANDIDATES)

    for run_number, candidate in enumerate(
        IF_CONTAMINATION_SWEEP_CANDIDATES.to_dict("records"),
        start=1,
    ):
        feature_set = candidate["feature_set"]
        representation_name = candidate["representation_name"]
        contamination = candidate["contamination"]

        print(
            f"[{run_number}/{total_runs}] Running Isolation Forest for "
            f"{feature_set} / {representation_name} / contamination={contamination:.3f}"
        )

        branch_result_df, summary_record, detail_record = run_isolation_forest_branch(
            feature_set=feature_set,
            representation_name=representation_name,
            contamination=contamination,
        )

        if_branch_result_parts.append(branch_result_df)
        if_contamination_summary_records.append(summary_record)
        if_contamination_detail_records.append(detail_record)

    if_contamination_sweep_summary_df = (
        pd.DataFrame(if_contamination_summary_records)
        .sort_values(["feature_set", "representation_name", "contamination"])
        .reset_index(drop=True)
    )
    if_contamination_sweep_detail_df = (
        pd.DataFrame(if_contamination_detail_records)
        .sort_values(["feature_set", "representation_name", "contamination"])
        .reset_index(drop=True)
    )
    if_score_profile_df = (
        pd.concat(if_branch_result_parts, ignore_index=True)
        .sort_values(["feature_set", "representation_name", "contamination", MODEL_ID_COLUMN])
        .reset_index(drop=True)
    )

    if WRITE_333_OUTPUTS:
        if_contamination_sweep_summary_df.to_parquet(
            IF_CONTAMINATION_SWEEP_SUMMARY_PATH,
            index=False,
        )
        if_contamination_sweep_detail_df.to_parquet(
            IF_CONTAMINATION_SWEEP_DETAIL_PATH,
            index=False,
        )
        if_score_profile_df.to_parquet(
            IF_SCORE_PROFILE_PATH,
            index=False,
        )
else:
    if_contamination_sweep_summary_df = pd.read_parquet(IF_CONTAMINATION_SWEEP_SUMMARY_PATH)
    if_contamination_sweep_detail_df = pd.read_parquet(IF_CONTAMINATION_SWEEP_DETAIL_PATH)
    if_score_profile_df = pd.read_parquet(IF_SCORE_PROFILE_PATH)

# Compact execution inventory instead of dumping the full sweep tables
if_contamination_run_inventory_df = (
    if_contamination_sweep_summary_df.groupby(
        ["feature_set", "representation_name"],
        dropna=False,
    )
    .agg(
        contamination_candidates=("contamination", "size"),
        contamination_min=("contamination", "min"),
        contamination_max=("contamination", "max"),
        rows_evaluated=("rows_evaluated", "max"),
        anomaly_rows_min=("anomaly_rows", "min"),
        anomaly_rows_max=("anomaly_rows", "max"),
        status=("status", lambda s: "pass" if s.eq("pass").all() else "review"),
    )
    .reset_index()
)

display(
    if_contamination_run_inventory_df.style.format(
        {
            "contamination_min": "{:.3f}",
            "contamination_max": "{:.3f}",
        }
    )
)

,feature_set,representation_name,contamination_candidates,contamination_min,contamination_max,rows_evaluated,anomaly_rows_min,anomaly_rows_max,status
0,bus,pca_80pct,7,0.005,0.050,50000,250,2500,pass
1,bus,umap_pca_to_umap_10d,7,0.005,0.050,50000,250,2500,pass
2,fhvhv,pca_80pct,7,0.005,0.050,50000,250,2500,pass
3,fhvhv,umap_pca_to_umap_10d,7,0.005,0.050,50000,250,2500,pass
4,multimodal,pca_80pct,7,0.005,0.050,50000,250,2500,pass
5,multimodal,umap_pca_to_umap_10d,7,0.005,0.050,50000,248,2500,pass
6,subway,pca_80pct,7,0.005,0.050,50000,250,2500,pass
7,subway,umap_pca_to_umap_10d,7,0.005,0.050,50000,250,2500,pass
8,taxi,pca_80pct,7,0.005,0.050,50000,250,2500,pass
9,taxi,umap_pca_to_umap_10d,7,0.005,0.050,50000,250,2500,pass


Findings\. The lightweight sweep completed across all five modalities, both retained representation branches, and the full candidate contamination grid\. At this stage the main thing to confirm is coverage, not interpretation: the run inventory shows that every modality\-representation branch was evaluated across the same seven contamination values, with complete outputs written for later prevalence and score\-separation review\.

### Review anomaly\-score separation across contamination rates

Start interpretation with the simplest signal: anomaly prevalence\. Before digging into score separation, we want to see how quickly the anomaly surface expands as contamination rises, whether that growth looks controlled or abrupt, and whether PCA and UMAP are producing meaningfully different anomaly burdens at the same nominal contamination level\.

In [15]:
# ---------------------------------------------------------------------
# Summarize anomaly prevalence across contamination rates
# ---------------------------------------------------------------------

if_anomaly_prevalence_summary_df = (
    if_contamination_sweep_summary_df[
        [
            "feature_set",
            "representation_name",
            "contamination",
            "rows_evaluated",
            "anomaly_rows",
            "anomaly_share",
            "status",
        ]
    ]
    .sort_values(["feature_set", "representation_name", "contamination"])
    .reset_index(drop=True)
)

if_anomaly_prevalence_gap_df = (
    if_anomaly_prevalence_summary_df.pivot_table(
        index=["feature_set", "contamination"],
        columns="representation_name",
        values="anomaly_share",
    )
    .reset_index()
)

if_anomaly_prevalence_gap_df["abs_prevalence_gap"] = (
    if_anomaly_prevalence_gap_df["pca_80pct"]
    - if_anomaly_prevalence_gap_df["umap_pca_to_umap_10d"]
).abs()

if_anomaly_prevalence_gap_summary_df = (
    if_anomaly_prevalence_gap_df.groupby("feature_set", dropna=False)
    .agg(
        max_abs_prevalence_gap=("abs_prevalence_gap", "max"),
        mean_abs_prevalence_gap=("abs_prevalence_gap", "mean"),
    )
    .reset_index()
)

display(
    if_anomaly_prevalence_gap_summary_df.style.format(
        {
            "max_abs_prevalence_gap": "{:.3f}",
            "mean_abs_prevalence_gap": "{:.3f}",
        }
    )
)

# ---------------------------------------------------------------------
# Visualize anomaly prevalence across contamination rates
# ---------------------------------------------------------------------

from plotly.subplots import make_subplots
import plotly.graph_objects as go

plot_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]
panel_titles = ["BUS", "SUBWAY", "TAXI", "FHVHV", "MULTIMODAL"]

plot_df = if_anomaly_prevalence_summary_df.copy()
plot_df["representation_label"] = plot_df["representation_name"].map(
    {
        "pca_80pct": "PCA",
        "umap_pca_to_umap_10d": "UMAP",
    }
)
fig = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{}, {}],
        [{}, {}],
        [{}, {}],
    ],
    subplot_titles=["BUS", "", "SUBWAY", "TAXI", "FHVHV", "MULTIMODAL"],
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.10,
    vertical_spacing=0.1,
)

panel_positions = {
    "bus": (1, 1),
    "subway": (2, 1),
    "taxi": (2, 2),
    "fhvhv": (3, 1),
    "multimodal": (3, 2),
}

color_map = IF_REPRESENTATION_COLOR_MAP
dash_map = {"PCA": "solid", "UMAP": "dot"}

for feature_set in plot_order:
    row, col = panel_positions[feature_set]
    feature_df = plot_df.loc[plot_df["feature_set"].eq(feature_set)].copy()

    for representation_label in ["PCA", "UMAP"]:
        rep_df = feature_df.loc[
            feature_df["representation_label"].eq(representation_label)
        ].sort_values("contamination")

        fig.add_trace(
            go.Scatter(
                x=rep_df["contamination"],
                y=rep_df["anomaly_share"],
                mode="lines+markers",
                name=representation_label,
                legendgroup=representation_label,
                showlegend=(feature_set == "bus"),
                line=dict(
                    color=color_map[representation_label],
                    dash=dash_map[representation_label],
                    width=2,
                ),
                marker=dict(size=6),
            ),
            row=row,
            col=col,
        )

fig.update_yaxes(
    tickformat=".1%",
    title_text="Anomaly share",
    row=1,
    col=1,
)
fig.update_yaxes(tickformat=".1%", row=2, col=1)
fig.update_yaxes(tickformat=".1%", row=2, col=2)
fig.update_yaxes(tickformat=".1%", row=3, col=1)
fig.update_yaxes(tickformat=".1%", row=3, col=2)

fig.update_xaxes(title_text="Contamination rate", tickformat=".3f", row=3, col=1)
fig.update_xaxes(title_text="Contamination rate", tickformat=".3f", row=3, col=2)

fig.update_layout(
    title="Isolation Forest Anomaly Prevalence Across Candidate Contamination Rates",
    width=980,
    height=640,
    margin=dict(l=55, r=25, t=70, b=65),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.08,
        xanchor="center",
        x=0.5,
    ),
)

apply_if_brand_layout(
    fig,
    title="Isolation Forest Anomaly Prevalence Across Contamination Rates",
)

fig.show()

,feature_set,max_abs_prevalence_gap,mean_abs_prevalence_gap
0,bus,0.000,0.000
1,fhvhv,0.000,0.000
2,multimodal,0.000,0.000
3,subway,0.000,0.000
4,taxi,0.000,0.000


Findings\. The table and chart tell the same story very cleanly: anomaly prevalence rises almost one\-for\-one with the requested contamination rate, and it does so in essentially the same way for both PCA and UMAP across every modality\. The zero prevalence\-gap summary confirms what the overlaid lines suggest visually: representation choice is not materially affecting anomaly volume at this stage; contamination is acting as a direct anomaly\-volume lever\. That makes this block a useful calibration sanity check, but not a decision block on its own, because prevalence is behaving exactly as instructed rather than revealing which contamination settings produce the most analytically useful anomaly tail\.

### Review anomaly\-score separation and tail behavior across contamination rates

Now that prevalence has confirmed that contamination is acting as a direct anomaly\-volume lever, the next question is whether the resulting anomaly tail still looks meaningfully distinct from the bulk of observations\. This is the part of the calibration where Isolation Forest becomes more than a bookkeeping exercise\.

For this review, focus on three score\-based quantities:

\* anomaly\_score\_p95: the upper edge of the anomaly score distribution; values closer to 0 mean the anomaly set is absorbing more borderline rows
\* normal\_score\_p05: the lower edge of the normal score distribution; lower values mean the normal mass is bleeding closer to the anomaly threshold
\* tail\_separation\_gap = normal\_score\_p05 \- anomaly\_score\_p95: positive values indicate cleaner separation between the anomaly tail and the bulk, while shrinking or negative values indicate growing overlap

In [16]:
# ---------------------------------------------------------------------
# Summarize anomaly-score separation across contamination rates
# ---------------------------------------------------------------------

if_score_separation_records = []

for (
    feature_set,
    representation_name,
    contamination,
), group_df in if_score_profile_df.groupby(
    ["feature_set", "representation_name", "contamination"],
    dropna=False,
    sort=True,
):
    anomaly_scores = group_df.loc[
        group_df["if_anomaly_flag"].eq(1),
        "if_decision_score",
    ]
    normal_scores = group_df.loc[
        group_df["if_anomaly_flag"].eq(0),
        "if_decision_score",
    ]

    anomaly_score_p50 = float(anomaly_scores.median())
    anomaly_score_p95 = float(anomaly_scores.quantile(0.95))
    normal_score_p05 = float(normal_scores.quantile(0.05))
    normal_score_p50 = float(normal_scores.median())

    if_score_separation_records.append(
        {
            "feature_set": feature_set,
            "representation_name": representation_name,
            "contamination": contamination,
            "anomaly_rows": int(anomaly_scores.shape[0]),
            "normal_rows": int(normal_scores.shape[0]),
            "anomaly_score_p50": anomaly_score_p50,
            "anomaly_score_p95": anomaly_score_p95,
            "normal_score_p05": normal_score_p05,
            "normal_score_p50": normal_score_p50,
            "median_separation": normal_score_p50 - anomaly_score_p50,
            "tail_separation_gap": normal_score_p05 - anomaly_score_p95,
        }
    )

if_score_separation_summary_df = (
    pd.DataFrame(if_score_separation_records)
    .sort_values(["feature_set", "representation_name", "contamination"])
    .reset_index(drop=True)
)

display(
    if_score_separation_summary_df.style.format(
        {
            "contamination": "{:.3f}",
            "anomaly_score_p50": "{:.3f}",
            "anomaly_score_p95": "{:.3f}",
            "normal_score_p05": "{:.3f}",
            "normal_score_p50": "{:.3f}",
            "median_separation": "{:.3f}",
            "tail_separation_gap": "{:.3f}",
        }
    )
)

# ---------------------------------------------------------------------
# Visualize tail separation across contamination rates
# ---------------------------------------------------------------------

if_tail_plot_df = if_score_separation_summary_df.copy()
if_tail_plot_df["representation_label"] = if_tail_plot_df["representation_name"].map(
    {
        "pca_80pct": "PCA",
        "umap_pca_to_umap_10d": "UMAP",
    }
)

feature_set_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]
panel_positions = {
    "bus": (1, 1),
    "subway": (2, 1),
    "taxi": (2, 2),
    "fhvhv": (3, 1),
    "multimodal": (3, 2),
}

fig = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{}, {}],
        [{}, {}],
        [{}, {}],
    ],
    subplot_titles=["BUS", "", "SUBWAY", "TAXI", "FHVHV", "MULTIMODAL"],
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.10,
    vertical_spacing=0.10,
)

color_map = IF_REPRESENTATION_COLOR_MAP
dash_map = {"PCA": "solid", "UMAP": "dot"}

for feature_set in feature_set_order:
    row, col = panel_positions[feature_set]
    feature_df = if_tail_plot_df.loc[
        if_tail_plot_df["feature_set"].eq(feature_set)
    ].copy()

    for representation_label in ["PCA", "UMAP"]:
        rep_df = feature_df.loc[
            feature_df["representation_label"].eq(representation_label)
        ].sort_values("contamination")

        fig.add_trace(
            go.Scatter(
                x=rep_df["contamination"],
                y=rep_df["tail_separation_gap"],
                mode="lines+markers",
                name=representation_label,
                legendgroup=representation_label,
                showlegend=(feature_set == "bus"),
                line=dict(
                    color=color_map[representation_label],
                    dash=dash_map[representation_label],
                    width=2,
                ),
                marker=dict(size=6),
                customdata=np.stack(
                    [
                        rep_df["median_separation"],
                        rep_df["anomaly_score_p95"],
                        rep_df["normal_score_p05"],
                    ],
                    axis=1,
                ),
                hovertemplate=(
                    "Representation=%{fullData.name}<br>"
                    "Contamination=%{x:.3f}<br>"
                    "Tail separation gap=%{y:.3f}<br>"
                    "Median separation=%{customdata[0]:.3f}<br>"
                    "Anomaly score p95=%{customdata[1]:.3f}<br>"
                    "Normal score p05=%{customdata[2]:.3f}<extra></extra>"
                ),
            ),
            row=row,
            col=col,
        )

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
    line_width=1,
)

fig.update_xaxes(visible=False, row=1, col=2)
fig.update_yaxes(visible=False, row=1, col=2)
fig.layout.annotations[1].text = ""

fig.update_yaxes(title_text="Tail separation gap", row=1, col=1)
fig.update_xaxes(title_text="Contamination rate", tickformat=".3f", row=3, col=1)
fig.update_xaxes(title_text="Contamination rate", tickformat=".3f", row=3, col=2)

fig.update_layout(
    title="Isolation Forest Tail Separation Across Candidate Contamination Rates",
    width=930,
    height=630,
    margin=dict(l=60, r=25, t=70, b=65),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.08,
        xanchor="center",
        x=0.5,
    ),
)

fig.show()

,feature_set,representation_name,contamination,anomaly_rows,normal_rows,anomaly_score_p50,anomaly_score_p95,normal_score_p05,normal_score_p50,median_separation,tail_separation_gap
0,bus,pca_80pct,0.005,250,49750,-0.042,-0.003,0.136,0.235,0.277,0.139
1,bus,pca_80pct,0.010,500,49500,-0.044,-0.004,0.095,0.191,0.235,0.099
2,bus,pca_80pct,0.015,750,49250,-0.042,-0.003,0.074,0.167,0.209,0.076
3,bus,pca_80pct,0.020,1000,49000,-0.044,-0.003,0.056,0.148,0.192,0.059
4,bus,pca_80pct,0.030,1500,48500,-0.042,-0.003,0.038,0.126,0.168,0.041
5,bus,pca_80pct,0.040,2000,48000,-0.035,-0.002,0.030,0.114,0.148,0.032
6,bus,pca_80pct,0.050,2500,47500,-0.031,-0.002,0.024,0.105,0.136,0.027
7,bus,umap_pca_to_umap_10d,0.005,250,49750,-0.010,-0.002,0.051,0.152,0.162,0.053
8,bus,umap_pca_to_umap_10d,0.010,500,49500,-0.024,-0.000,0.033,0.128,0.152,0.033
9,bus,umap_pca_to_umap_10d,0.015,749,49251,-0.009,-0.001,0.032,0.125,0.134,0.032


Findings\. This block is asking a simple question: as we tell Isolation Forest to flag more rows, does it still isolate a clearly unusual anomaly group, or does it start pulling in rows that look more and more like the normal bulk? The chart suggests the second thing happens as contamination increases\. Across every modality, the gap between the anomaly tail and the normal mass gets smaller at higher contamination rates, which means the model is gradually moving from “flag the most isolated rows” toward “flag a broader and blurrier slice of the panel\.” PCA usually preserves a cleaner gap than UMAP, especially for Bus, Subway, Taxi, and FHVHV, while Multimodal is the closest case between the two\. So the practical takeaway is that lower contamination settings are doing a better job of isolating a genuinely distinct anomaly tail, while higher settings increasingly look like they are sweeping borderline observations into the anomaly set\.This block is asking a simple question: as we tell Isolation Forest to flag more rows, does it still isolate a clearly unusual anomaly group, or does it start pulling in rows that look more and more like the normal bulk? The chart suggests the second thing happens as contamination increases\. Across every modality, the gap between the anomaly tail and the normal mass gets smaller at higher contamination rates, which means the model is gradually moving from “flag the most isolated rows” toward “flag a broader and blurrier slice of the panel\.” PCA usually preserves a cleaner gap than UMAP, especially for Bus, Subway, Taxi, and FHVHV, while Multimodal is the closest case between the two\. So the practical takeaway is that lower contamination settings are doing a better job of isolating a genuinely distinct anomaly tail, while higher settings increasingly look like they are sweeping borderline observations into the anomaly set\.

### Screen contamination regions using score\-separation behavior

Now let's turn the score\-separation evidence into a transparent screening rule\. The goal here is not to pick final Isolation Forest settings yet\. It is to separate contamination levels that still preserve a reasonably distinct anomaly tail from settings that are already starting to blur the anomaly boundary too aggressively\.

For this screen, we’ll use two simple ideas:

1\) Tail separation gap should stay comfortably positive\. If it gets too small, the anomaly tail is starting to overlap too much with the bulk of the panel\.
2\) Median separation should also stay meaningfully positive\. This acts as a broader check that the anomaly and normal score distributions are not collapsing toward each other even if the extreme tail still looks barely separated\.

We will score each candidate contamination setting within each modality\-representation branch and label it as:
\* advance: still looks plausible enough to carry into stability testing
\* caution: not immediately unusable, but the separation is thinning
\* retire: separation has eroded enough that the anomaly set is starting to look too diffuse for a clean next\-stage calibration branch

Let’s turn the score\-separation evidence into an explicit screening layer that later blocks can reuse\. This block does not display results; it just applies a stricter decision rule so that contamination settings with weak tail separation and weak median separation are no longer preserved too generously as caution cases\. The goal is to make the later scorecards easier to trust by ensuring that the decision label matches what the score values already suggest\.

In [17]:
# ---------------------------------------------------------------------
# Screen contamination regions using score-separation behavior
# ---------------------------------------------------------------------

IF_TAIL_GAP_ADVANCE_MIN = 0.040
IF_TAIL_GAP_CAUTION_MIN = 0.030
IF_MEDIAN_SEPARATION_ADVANCE_MIN = 0.200
IF_MEDIAN_SEPARATION_CAUTION_MIN = 0.150

def score_tail_gap_absolute(value):
    if value >= IF_TAIL_GAP_ADVANCE_MIN:
        return 1.0
    if value >= IF_TAIL_GAP_CAUTION_MIN:
        return 0.5
    return 0.0


def score_median_gap_absolute(value):
    if value >= IF_MEDIAN_SEPARATION_ADVANCE_MIN:
        return 1.0
    if value >= IF_MEDIAN_SEPARATION_CAUTION_MIN:
        return 0.5
    return 0.0


def classify_if_contamination_screen(row):
    tail_gap = row["tail_separation_gap"]
    median_gap = row["median_separation"]

    if (
        tail_gap >= IF_TAIL_GAP_ADVANCE_MIN
        and median_gap >= IF_MEDIAN_SEPARATION_ADVANCE_MIN
    ):
        return "advance"

    if (
        tail_gap >= IF_TAIL_GAP_CAUTION_MIN
        and median_gap >= IF_MEDIAN_SEPARATION_CAUTION_MIN
    ):
        return "caution"

    return "retire"


if_contamination_screen_df = if_score_separation_summary_df.copy()

if_contamination_screen_df["screen_decision"] = (
    if_contamination_screen_df.apply(classify_if_contamination_screen, axis=1)
)

if_contamination_screen_df["screen_rationale"] = np.select(
    [
        if_contamination_screen_df["screen_decision"].eq("advance"),
        if_contamination_screen_df["screen_decision"].eq("caution"),
        if_contamination_screen_df["screen_decision"].eq("retire"),
    ],
    [
        "Tail and median score separation both remain comfortably positive.",
        "Separation is still positive, but the anomaly boundary is thinning.",
        "Score separation has thinned enough that the anomaly tail looks too diffuse.",
    ],
    default="review",
)

if_contamination_screen_df["representation_label"] = if_contamination_screen_df[
    "representation_name"
].map(
    {
        "pca_80pct": "PCA",
        "umap_pca_to_umap_10d": "UMAP",
    }
)

if_contamination_screen_df["representation_label"] = pd.Categorical(
    if_contamination_screen_df["representation_label"],
    categories=["PCA", "UMAP"],
    ordered=True,
)

if_contamination_screen_df["row_label"] = (
    if_contamination_screen_df["representation_label"].astype(str)
    + " | "
    + if_contamination_screen_df["contamination"].map(lambda x: f"{x:.3f}")
)

if_contamination_screen_df["tail_gap_score"] = if_contamination_screen_df[
    "tail_separation_gap"
].map(score_tail_gap_absolute)

if_contamination_screen_df["median_gap_score"] = if_contamination_screen_df[
    "median_separation"
].map(score_median_gap_absolute)

if_contamination_screen_df["decision_score"] = if_contamination_screen_df[
    "screen_decision"
].map(
    {
        "advance": 1.0,
        "caution": 0.5,
        "retire": 0.0,
    }
)

With the stricter screening rule in place, we can now inspect the four modality\-specific branches in a way that is easier to trust visually\. The table gives the exact tail\-gap, median\-gap, and decision values, while the scorecards translate them into a simple screening surface\. Read each row as one contamination setting within one representation: greener rows preserve a cleaner anomaly tail, while red rows indicate settings that should probably be retired before stability testing\.

In [18]:
# ---------------------------------------------------------------------
# Review contamination screening table and scorecards: Bus, Subway, Taxi, FHVHV
# ---------------------------------------------------------------------

feature_sets_top = ["bus", "subway", "taxi", "fhvhv"]
metric_order = ["Tail gap", "Median gap", "Decision"]

scorecard_top_table_df = (
    if_contamination_screen_df.loc[
        if_contamination_screen_df["feature_set"].isin(feature_sets_top)
    ]
    .copy()
    .sort_values(
        ["feature_set", "representation_label", "contamination"],
        ascending=[True, True, True],
    )
    .reset_index(drop=True)
)

display(
    scorecard_top_table_df[
        [
            "feature_set",
            "representation_label",
            "contamination",
            "tail_separation_gap",
            "median_separation",
            "screen_decision",
        ]
    ].style.format(
        {
            "contamination": "{:.3f}",
            "tail_separation_gap": "{:.3f}",
            "median_separation": "{:.3f}",
        }
    )
)

scorecard_long_df = pd.concat(
    [
        if_contamination_screen_df[
            ["feature_set", "row_label", "tail_separation_gap", "tail_gap_score"]
        ].rename(
            columns={
                "tail_separation_gap": "metric_value",
                "tail_gap_score": "relative_score",
            }
        ).assign(metric_label="Tail gap"),
        if_contamination_screen_df[
            ["feature_set", "row_label", "median_separation", "median_gap_score"]
        ].rename(
            columns={
                "median_separation": "metric_value",
                "median_gap_score": "relative_score",
            }
        ).assign(metric_label="Median gap"),
        if_contamination_screen_df[
            ["feature_set", "row_label", "screen_decision", "decision_score"]
        ].rename(
            columns={
                "screen_decision": "metric_value",
                "decision_score": "relative_score",
            }
        ).assign(metric_label="Decision"),
    ],
    ignore_index=True,
)

row_order_df = (
    if_contamination_screen_df.loc[
        if_contamination_screen_df["feature_set"].isin(feature_sets_top)
    ][["feature_set", "representation_label", "contamination", "row_label"]]
    .drop_duplicates()
    .sort_values(
        ["feature_set", "representation_label", "contamination"],
        ascending=[True, True, True],
    )
)

row_orders = {
    feature_set: row_order_df.loc[
        row_order_df["feature_set"].eq(feature_set), "row_label"
    ].tolist()
    for feature_set in feature_sets_top
}

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=["BUS", "SUBWAY", "TAXI", "FHVHV"],
    shared_xaxes=True,
    shared_yaxes=False,
    horizontal_spacing=0.16,
    vertical_spacing=0.10,
)

panel_positions = {
    "bus": (1, 1),
    "subway": (1, 2),
    "taxi": (2, 1),
    "fhvhv": (2, 2),
}

for feature_set in feature_sets_top:
    row, col = panel_positions[feature_set]
    panel_df = scorecard_long_df.loc[
        scorecard_long_df["feature_set"].eq(feature_set)
    ].copy()

    z_matrix = []
    text_matrix = []
    ordered_rows = row_orders[feature_set]

    for row_label in ordered_rows:
        row_df = panel_df.loc[panel_df["row_label"].eq(row_label)].copy()
        row_df["metric_label"] = pd.Categorical(
            row_df["metric_label"],
            categories=metric_order,
            ordered=True,
        )
        row_df = row_df.sort_values("metric_label")

        z_matrix.append(row_df["relative_score"].tolist())

        text_cells = []
        for _, metric_row in row_df.iterrows():
            if metric_row["metric_label"] == "Decision":
                text_cells.append(str(metric_row["metric_value"]))
            else:
                text_cells.append(f"{float(metric_row['metric_value']):.3f}")
        text_matrix.append(text_cells)

    fig.add_trace(
        go.Heatmap(
            z=z_matrix,
            x=metric_order,
            y=ordered_rows,
            text=text_matrix,
            texttemplate="%{text}",
            textfont={"size": 10},
            colorscale=IF_STABILITY_SCALE,
            zmin=0,
            zmax=1,
            showscale=(feature_set == "bus"),
            colorbar=dict(
                title="Relative score",
                len=0.90,
                y=0.50,
            ) if feature_set == "bus" else None,
            hovertemplate=(
                "Feature set=" + feature_set
                + "<br>Setting=%{y}"
                + "<br>Metric=%{x}"
                + "<br>Displayed value=%{text}"
                + "<extra></extra>"
            ),
        ),
        row=row,
        col=col,
    )

fig.update_yaxes(ticklabelstandoff=4)

fig.update_layout(
    title="Isolation Forest Contamination Screening Scorecards: Bus, Subway, Taxi, and FHVHV",
    width=930,
    height=640,
    margin=dict(l=125, r=40, t=75, b=55),
)

fig.show()

,feature_set,representation_label,contamination,tail_separation_gap,median_separation,screen_decision
0,bus,PCA,0.005,0.139,0.277,advance
1,bus,PCA,0.010,0.099,0.235,advance
2,bus,PCA,0.015,0.076,0.209,advance
3,bus,PCA,0.020,0.059,0.192,caution
4,bus,PCA,0.030,0.041,0.168,caution
5,bus,PCA,0.040,0.032,0.148,retire
6,bus,PCA,0.050,0.027,0.136,retire
7,bus,UMAP,0.005,0.053,0.162,caution
8,bus,UMAP,0.010,0.033,0.152,caution
9,bus,UMAP,0.015,0.032,0.134,retire


Findings\. The scorecards make the screening logic much easier to read than the raw score tables\. Across all four modality\-specific branches, the same broad pattern holds: lower contamination settings preserve the strongest tail separation, while higher contamination settings steadily weaken the distinction between anomalous and normal rows\. PCA is consistently stronger than UMAP in these modality\-specific branches, especially for Bus, Subway, and Taxi, where PCA preserves a usable low\-contamination region while UMAP falls into caution or retire much earlier, and in Subway never reaches advance at all\. FHVHV is the softest case: both representations remain more viable there than elsewhere, but PCA still stays cleaner overall and UMAP becomes mixed sooner\. So the practical takeaway is that the next\-stage candidate set should lean toward lower contamination values, with PCA carrying the stronger calibration story for these four modalities\.

Multimodal gets its own readout because it is the one branch where PCA and UMAP stay closer together than they do in the modality\-specific scorecards\. The job here is the same as before: identify which contamination settings still preserve a distinct anomaly tail and which ones are already thinning into a blurrier anomaly boundary\. The table gives the exact score\-separation values, and the scorecard makes it easier to see whether Multimodal still supports a clean retained region or instead pushes most of the candidate settings into caution territory\.

In [19]:
# ---------------------------------------------------------------------
# Review contamination screening table and scorecard: Multimodal
# ---------------------------------------------------------------------

multimodal_screen_df = if_contamination_screen_df.copy()

multimodal_screen_df["representation_label"] = multimodal_screen_df["representation_name"].map(
    {
        "pca_80pct": "PCA",
        "umap_pca_to_umap_10d": "UMAP",
    }
)

multimodal_screen_df["representation_label"] = pd.Categorical(
    multimodal_screen_df["representation_label"],
    categories=["PCA", "UMAP"],
    ordered=True,
)

multimodal_screen_df["row_label"] = (
    multimodal_screen_df["representation_label"].astype(str)
    + " | "
    + multimodal_screen_df["contamination"].map(lambda x: f"{x:.3f}")
)

multimodal_screen_df["tail_gap_score"] = multimodal_screen_df["tail_separation_gap"].map(
    score_tail_gap_absolute
)
multimodal_screen_df["median_gap_score"] = multimodal_screen_df["median_separation"].map(
    score_median_gap_absolute
)
multimodal_screen_df["decision_score"] = multimodal_screen_df["screen_decision"].map(
    {
        "advance": 1.0,
        "caution": 0.5,
        "retire": 0.0,
    }
)

metric_order = ["Tail gap", "Median gap", "Decision"]

multimodal_table_df = (
    multimodal_screen_df.loc[multimodal_screen_df["feature_set"].eq("multimodal")]
    .copy()
    .sort_values(
        ["representation_label", "contamination"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

display(
    multimodal_table_df[
        [
            "feature_set",
            "representation_label",
            "contamination",
            "tail_separation_gap",
            "median_separation",
            "screen_decision",
        ]
    ].style.format(
        {
            "contamination": "{:.3f}",
            "tail_separation_gap": "{:.3f}",
            "median_separation": "{:.3f}",
        }
    )
)

multimodal_long_df = pd.concat(
    [
        multimodal_screen_df.loc[
            multimodal_screen_df["feature_set"].eq("multimodal"),
            ["feature_set", "row_label", "tail_separation_gap", "tail_gap_score"],
        ].rename(
            columns={
                "tail_separation_gap": "metric_value",
                "tail_gap_score": "relative_score",
            }
        ).assign(metric_label="Tail gap"),
        multimodal_screen_df.loc[
            multimodal_screen_df["feature_set"].eq("multimodal"),
            ["feature_set", "row_label", "median_separation", "median_gap_score"],
        ].rename(
            columns={
                "median_separation": "metric_value",
                "median_gap_score": "relative_score",
            }
        ).assign(metric_label="Median gap"),
        multimodal_screen_df.loc[
            multimodal_screen_df["feature_set"].eq("multimodal"),
            ["feature_set", "row_label", "screen_decision", "decision_score"],
        ].rename(
            columns={
                "screen_decision": "metric_value",
                "decision_score": "relative_score",
            }
        ).assign(metric_label="Decision"),
    ],
    ignore_index=True,
)

multimodal_row_order = (
    multimodal_table_df[["representation_label", "contamination", "row_label"]]
    .drop_duplicates()
    .sort_values(
        ["representation_label", "contamination"],
        ascending=[True, True],
    )["row_label"]
    .tolist()
)

z_matrix = []
text_matrix = []

for row_label in multimodal_row_order:
    row_df = multimodal_long_df.loc[
        multimodal_long_df["row_label"].eq(row_label)
    ].copy()
    row_df["metric_label"] = pd.Categorical(
        row_df["metric_label"],
        categories=metric_order,
        ordered=True,
    )
    row_df = row_df.sort_values("metric_label")

    z_matrix.append(row_df["relative_score"].tolist())

    text_cells = []
    for _, metric_row in row_df.iterrows():
        if metric_row["metric_label"] == "Decision":
            text_cells.append(str(metric_row["metric_value"]))
        else:
            text_cells.append(f"{float(metric_row['metric_value']):.3f}")
    text_matrix.append(text_cells)

fig = go.Figure(
    data=[
        go.Heatmap(
            z=z_matrix,
            x=metric_order,
            y=multimodal_row_order,
            text=text_matrix,
            texttemplate="%{text}",
            textfont={"size": 11},
            colorscale=IF_STABILITY_SCALE,
            zmin=0,
            zmax=1,
            colorbar=dict(
                title="Relative score",
                len=0.85,
                y=0.50,
            ),
            hovertemplate=(
                "Feature set=multimodal"
                + "<br>Setting=%{y}"
                + "<br>Metric=%{x}"
                + "<br>Displayed value=%{text}"
                + "<extra></extra>"
            ),
        )
    ]
)

fig.update_yaxes(ticklabelstandoff=4)

fig.update_layout(
    title="Isolation Forest Contamination Screening Scorecard: Multimodal",
    width=930,
    height=420,
    margin=dict(l=140, r=45, t=70, b=50),
)

fig.show()

,feature_set,representation_label,contamination,tail_separation_gap,median_separation,screen_decision
0,multimodal,PCA,0.005,0.132,0.262,advance
1,multimodal,PCA,0.010,0.103,0.240,advance
2,multimodal,PCA,0.015,0.089,0.221,advance
3,multimodal,PCA,0.020,0.079,0.206,advance
4,multimodal,PCA,0.030,0.064,0.188,caution
5,multimodal,PCA,0.040,0.052,0.173,caution
6,multimodal,PCA,0.050,0.044,0.161,caution
7,multimodal,UMAP,0.005,0.095,0.189,caution
8,multimodal,UMAP,0.010,0.082,0.187,caution
9,multimodal,UMAP,0.015,0.071,0.183,caution


Findings\. Multimodal is the closest representation\-sensitivity case in this screening step, but PCA still comes out cleaner than UMAP\. PCA keeps a clearly usable low\-contamination region at 0\.005 through 0\.020, where both tail separation and median separation remain stronger and the screen stays in advance\. UMAP never reaches advance at all: even at the lowest contamination levels it stays in caution, and both separation metrics step down steadily as contamination increases, with the highest setting finally dropping to retire\. So the practical takeaway is that Multimodal does not overturn the broader pattern from the modality\-specific branches, but it does soften it a bit: UMAP remains plausible at the low end, yet PCA still preserves the more convincing anomaly tail and gives us the clearer retained region for downstream stability testing\.

> 🎯 Decision\. The contamination screen is now strong enough to narrow the Isolation Forest search space before stability testing\. Across the notebook, lower contamination settings preserve the cleanest anomaly tail, while higher settings steadily blur the boundary between anomalous and normal rows\. PCA consistently carries the stronger calibration story, especially for Bus, Subway, and Taxi, and Multimodal follows the same general pattern\. UMAP is not being retired outright yet, but it is clearly the weaker branch: it survives only through a much narrower low\-contamination region and should advance on a conditional basis pending the stability checks that follow\. The next section should therefore carry forward the strongest low\-contamination PCA settings as the main candidate region, while retaining only the lowest still\-plausible UMAP settings for further testing\.

### Section Summary

Isolation Forest calibration is now doing what we need it to do\. The prevalence check showed that contamination behaves mechanically as a direct anomaly\-volume lever, so anomaly counts alone are not useful for selection\. The real signal came from score separation: as contamination rises, the anomaly tail becomes less distinct from the normal bulk, and that erosion happens faster in UMAP than in PCA\. After tightening the screening rule, the retained region is now much clearer\. PCA keeps a believable low\-contamination anomaly tail across all five branches, while UMAP survives only in a more limited, lower\-contamination range and in some cases already looks too diffuse to trust further\. That gives the next section a much narrower and more defensible search space for stability testing\.

- Bus: PCA supports a small low\-contamination retained region, while UMAP weakens quickly and is mostly caution or retire\.

- Subway: PCA remains strong across the full tested range, but UMAP is effectively screened out from the start\.

- Taxi: PCA preserves a usable low\-contamination region, while UMAP becomes weak enough to retire at modest contamination levels\.

- FHVHV: Both representations are more viable here than elsewhere, but PCA still provides the cleaner calibration path\.

- Multimodal: PCA again holds the clearer retained region, while UMAP remains plausible only at the low end and never fully reaches advance territory\.

## 3\.3\.3\.3 Evaluate Configuration Stability and Representation Sensitivity

Now that the contamination space has been narrowed, the next question is whether the surviving Isolation Forest branches are actually stable enough to trust\. This section tests how much the anomaly surface moves under small contamination changes and whether PCA and UMAP still produce meaningfully different anomaly behavior once both have been narrowed to their most plausible retained regions\. The goal is to determine whether the surviving configurations remain coherent under local perturbation and whether UMAP is still contributing a distinct anomaly slice or merely adding instability without enough analytical benefit\.

### Materialize retained row\-level anomaly outputs for stability testing

Before we can compare neighboring retained settings or judge whether PCA and UMAP remain stable enough to trust, we need the actual row\-level Isolation Forest outputs for every retained branch\. This step materializes those retained anomaly surfaces once, writes them to reusable parquet parts, and records a manifest so that the stability section can compare branches directly without refitting the same models over and over\.

This is one of the few places where writing intermediate outputs is worth it\. Each retained branch gets its own row\-level file so the notebook can resume cleanly after interruption, and so all later stability and representation\-sensitivity checks are working from the exact same anomaly surfaces\.

In [20]:
# ---------------------------------------------------------------------
# Materialize retained row-level Isolation Forest outputs for stability testing
# ---------------------------------------------------------------------

IF_STABILITY_ROW_LEVEL_DIR = INTERMEDIATE_333_DIR / "if_stability_row_level_parts"
IF_STABILITY_ROW_LEVEL_DIR.mkdir(parents=True, exist_ok=True)

IF_STABILITY_ROW_LEVEL_MANIFEST_PATH = (
    INTERMEDIATE_333_DIR / "if_stability_row_level_manifest.parquet"
)

# If you named the retained-candidate table something else, just point this alias to it.
if "if_retained_contamination_regions_df" in globals():
    if_stability_candidate_regions_df = if_retained_contamination_regions_df.copy()
elif "if_contamination_retained_regions_df" in globals():
    if_stability_candidate_regions_df = if_contamination_retained_regions_df.copy()
elif "if_contamination_screen_df" in globals():
    if_stability_candidate_regions_df = (
        if_contamination_screen_df.loc[
            if_contamination_screen_df["screen_decision"].isin(["advance", "caution"])
        ]
        .copy()
        .reset_index(drop=True)
    )
else:
    raise NameError(
        "No retained Isolation Forest candidate table is in memory. "
        "Please run the retained-contamination screening step first."
    )

required_candidate_columns = [
    "feature_set",
    "representation_name",
    "contamination",
]
missing_candidate_columns = [
    column
    for column in required_candidate_columns
    if column not in if_stability_candidate_regions_df.columns
]
if missing_candidate_columns:
    raise KeyError(
        "Retained candidate table is missing required columns: "
        f"{missing_candidate_columns}"
    )


def load_if_branch_rows_for_stability(feature_set, representation_name):
    metadata_df = load_calibration_metadata(feature_set)
    metadata_df = ensure_default_metadata_columns(feature_set, metadata_df)

    if (
        feature_set == "taxi"
        and representation_name == "pca_80pct"
        and TAXI_PCA_HANDLING_DECISION == "split_prepost_period"
    ):
        if PRE_POST_CP_COLUMN not in metadata_df.columns:
            raise KeyError(
                f"'{PRE_POST_CP_COLUMN}' is still missing for taxi after metadata recovery."
            )

        taxi_branch_parts = []
        expected_numeric_columns = None

        for policy_period, period_metadata_df in metadata_df.groupby(
            PRE_POST_CP_COLUMN,
            dropna=False,
            sort=False,
        ):
            period_path = TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD[policy_period]
            period_numeric_columns = get_numeric_representation_columns(
                feature_set=feature_set,
                representation_name=representation_name,
                representation_path=period_path,
            )

            if expected_numeric_columns is None:
                expected_numeric_columns = period_numeric_columns
            else:
                assert expected_numeric_columns == period_numeric_columns, (
                    "Taxi split PCA exports do not share the same component columns."
                )

            period_representation_df = pd.read_parquet(
                period_path,
                columns=[MODEL_ID_COLUMN, *period_numeric_columns],
            )

            taxi_branch_parts.append(
                period_metadata_df.merge(
                    period_representation_df,
                    on=MODEL_ID_COLUMN,
                    how="left",
                    validate="one_to_one",
                )
            )

        merged_df = (
            pd.concat(taxi_branch_parts, ignore_index=True)
            .sort_values(MODEL_ID_COLUMN)
            .reset_index(drop=True)
        )
        return merged_df, expected_numeric_columns

    merged_df, numeric_columns, _ = load_representation_rows(
        feature_set=feature_set,
        representation_name=representation_name,
        metadata_df=metadata_df,
    )
    merged_df = merged_df.sort_values(MODEL_ID_COLUMN).reset_index(drop=True)
    return merged_df, numeric_columns


def materialize_if_row_level_branch(feature_set, representation_name, contamination):
    branch_df, numeric_columns = load_if_branch_rows_for_stability(
        feature_set=feature_set,
        representation_name=representation_name,
    )

    X = branch_df[numeric_columns].to_numpy(dtype=float)

    model = IsolationForest(
        n_estimators=ISOLATION_FOREST_N_ESTIMATORS,
        contamination=contamination,
        max_samples=ISOLATION_FOREST_MAX_SAMPLES,
        max_features=ISOLATION_FOREST_MAX_FEATURES,
        random_state=ISOLATION_FOREST_RANDOM_STATE,
        n_jobs=-1,
    )

    fit_start = perf_counter()
    model.fit(X)
    fit_seconds = perf_counter() - fit_start

    raw_prediction = model.predict(X)
    anomaly_flag = (raw_prediction == -1).astype(int)
    decision_score = model.decision_function(X)
    raw_score = model.score_samples(X)

    branch_result_df = branch_df[
        [
            MODEL_ID_COLUMN,
            DATE_COLUMN,
            TEMPORAL_BUCKET_COLUMN,
            PRE_POST_CP_COLUMN,
            POLICY_GEOGRAPHY_COLUMN,
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            ZONE_ID_COLUMN,
        ]
    ].copy()

    branch_result_df["feature_set"] = feature_set
    branch_result_df["representation_name"] = representation_name
    branch_result_df["contamination"] = contamination
    branch_result_df["if_anomaly_flag"] = anomaly_flag
    branch_result_df["if_decision_score"] = decision_score
    branch_result_df["if_raw_score"] = raw_score

    candidate_id = (
        f"{feature_set}__{representation_name}__cont_{contamination:.3f}"
        .replace(".", "p")
    )
    output_path = IF_STABILITY_ROW_LEVEL_DIR / f"{candidate_id}.parquet"

    return branch_result_df, candidate_id, output_path, fit_seconds


if should_rebuild(REBUILD_IF_STABILITY_DIAGNOSTICS, IF_STABILITY_ROW_LEVEL_MANIFEST_PATH):
    if_stability_row_level_manifest_records = []

    retained_candidates_to_materialize_df = (
        if_stability_candidate_regions_df[
            ["feature_set", "representation_name", "contamination"]
        ]
        .drop_duplicates()
        .sort_values(["feature_set", "representation_name", "contamination"])
        .reset_index(drop=True)
    )

    total_candidates = len(retained_candidates_to_materialize_df)

    for run_number, candidate in enumerate(
        retained_candidates_to_materialize_df.to_dict("records"),
        start=1,
    ):
        feature_set = candidate["feature_set"]
        representation_name = candidate["representation_name"]
        contamination = float(candidate["contamination"])

        print(
            f"[{run_number}/{total_candidates}] Materializing Isolation Forest branch "
            f"{feature_set} / {representation_name} / contamination={contamination:.3f}"
        )

        branch_result_df, candidate_id, output_path, fit_seconds = (
            materialize_if_row_level_branch(
                feature_set=feature_set,
                representation_name=representation_name,
                contamination=contamination,
            )
        )

        if WRITE_333_OUTPUTS:
            branch_result_df.to_parquet(output_path, index=False)

        if_stability_row_level_manifest_records.append(
            {
                "feature_set": feature_set,
                "representation_name": representation_name,
                "contamination": contamination,
                "candidate_id": candidate_id,
                "row_level_path": str(output_path),
                "rows_written": len(branch_result_df),
                "anomaly_like_rows": int(branch_result_df["if_anomaly_flag"].sum()),
                "fit_seconds": fit_seconds,
                "status": "written",
            }
        )

    if_stability_row_level_manifest_df = (
        pd.DataFrame(if_stability_row_level_manifest_records)
        .sort_values(["feature_set", "representation_name", "contamination"])
        .reset_index(drop=True)
    )

    if WRITE_333_OUTPUTS:
        if_stability_row_level_manifest_df.to_parquet(
            IF_STABILITY_ROW_LEVEL_MANIFEST_PATH,
            index=False,
        )
else:
    if_stability_row_level_manifest_df = pd.read_parquet(
        IF_STABILITY_ROW_LEVEL_MANIFEST_PATH
    )

display(
    if_stability_row_level_manifest_df.style.format(
        {
            "contamination": "{:.3f}",
            "fit_seconds": "{:.3f}",
        }
    )
)

,feature_set,representation_name,contamination,candidate_id,row_level_path,rows_written,anomaly_like_rows,fit_seconds,status
0,bus,pca_80pct,0.005,bus__pca_80pct__cont_0p005,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_stability_row_level_parts/bus__pca_80pct__cont_0p005.parquet,50000,250,1.001,written
1,bus,pca_80pct,0.010,bus__pca_80pct__cont_0p010,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_stability_row_level_parts/bus__pca_80pct__cont_0p010.parquet,50000,500,0.984,written
2,bus,pca_80pct,0.015,bus__pca_80pct__cont_0p015,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_stability_row_level_parts/bus__pca_80pct__cont_0p015.parquet,50000,750,0.987,written
3,bus,pca_80pct,0.020,bus__pca_80pct__cont_0p020,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_stability_row_level_parts/bus__pca_80pct__cont_0p020.parquet,50000,1000,0.996,written
4,bus,pca_80pct,0.030,bus__pca_80pct__cont_0p030,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_stability_row_level_parts/bus__pca_80pct__cont_0p030.parquet,50000,1500,1.002,written
5,bus,umap_pca_to_umap_10d,0.005,bus__umap_pca_to_umap_10d__cont_0p005,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_stability_row_level_parts/bus__umap_pca_to_umap_10d__cont_0p005.parquet,50000,250,1.005,written
6,bus,umap_pca_to_umap_10d,0.010,bus__umap_pca_to_umap_10d__cont_0p010,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_stability_row_level_parts/bus__umap_pca_to_umap_10d__cont_0p010.parquet,50000,500,1.025,written
7,fhvhv,pca_80pct,0.005,fhvhv__pca_80pct__cont_0p005,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_stability_row_level_parts/fhvhv__pca_80pct__cont_0p005.parquet,50000,250,0.976,written
8,fhvhv,pca_80pct,0.010,fhvhv__pca_80pct__cont_0p010,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_stability_row_level_parts/fhvhv__pca_80pct__cont_0p010.parquet,50000,500,1.026,written
9,fhvhv,pca_80pct,0.015,fhvhv__pca_80pct__cont_0p015,/datasets/_deepnote_work/pipeline_data/3.3.3.intermediate_tables/if_stability_row_level_parts/fhvhv__pca_80pct__cont_0p015.parquet,50000,750,0.978,written


Findings\. The retained Isolation Forest candidate branches have now been materialized into reusable row\-level anomaly surfaces, with one parquet part per surviving feature\_set x representation x contamination combination\. That gives the next stability step a fixed set of anomaly outputs to compare directly, rather than forcing the notebook to refit the same retained branches each time\.

Before moving into actual stability comparisons, do one clean handoff check\. We want to confirm that every retained branch wrote successfully, that every row\-level file exists, and that each retained candidate still covers the full shared calibration panel rather than some partial or corrupted subset\.

In [21]:
# ---------------------------------------------------------------------
# Validate retained row-level Isolation Forest outputs for stability testing
# ---------------------------------------------------------------------

if_stability_row_level_validation_records = []

for manifest_row in if_stability_row_level_manifest_df.to_dict("records"):
    row_level_path = Path(manifest_row["row_level_path"])

    if not row_level_path.exists():
        if_stability_row_level_validation_records.append(
            {
                "candidate_id": manifest_row["candidate_id"],
                "feature_set": manifest_row["feature_set"],
                "representation_name": manifest_row["representation_name"],
                "contamination": manifest_row["contamination"],
                "row_level_rows": np.nan,
                "distinct_modeling_rows": np.nan,
                "expected_rows": 50000,
                "missing_rows": np.nan,
                "status": "review",
            }
        )
        continue

    branch_df = pd.read_parquet(row_level_path)

    row_level_rows = len(branch_df)
    distinct_modeling_rows = branch_df[MODEL_ID_COLUMN].nunique()
    expected_rows = 50000
    missing_rows = expected_rows - distinct_modeling_rows

    if_stability_row_level_validation_records.append(
        {
            "candidate_id": manifest_row["candidate_id"],
            "feature_set": manifest_row["feature_set"],
            "representation_name": manifest_row["representation_name"],
            "contamination": manifest_row["contamination"],
            "row_level_rows": row_level_rows,
            "distinct_modeling_rows": distinct_modeling_rows,
            "expected_rows": expected_rows,
            "missing_rows": missing_rows,
            "status": (
                "pass"
                if row_level_rows == expected_rows
                and distinct_modeling_rows == expected_rows
                and missing_rows == 0
                else "review"
            ),
        }
    )

if_stability_row_level_validation_df = pd.DataFrame(
    if_stability_row_level_validation_records
).sort_values(["feature_set", "representation_name", "contamination"]).reset_index(drop=True)

display(
    if_stability_row_level_validation_df.style.format(
        {
            "contamination": "{:.3f}",
        }
    )
)

assert if_stability_row_level_validation_df["status"].eq("pass").all(), (
    "One or more retained Isolation Forest row-level outputs is incomplete or misaligned."
)

,candidate_id,feature_set,representation_name,contamination,row_level_rows,distinct_modeling_rows,expected_rows,missing_rows,status
0,bus__pca_80pct__cont_0p005,bus,pca_80pct,0.005,50000,50000,50000,0,pass
1,bus__pca_80pct__cont_0p010,bus,pca_80pct,0.010,50000,50000,50000,0,pass
2,bus__pca_80pct__cont_0p015,bus,pca_80pct,0.015,50000,50000,50000,0,pass
3,bus__pca_80pct__cont_0p020,bus,pca_80pct,0.020,50000,50000,50000,0,pass
4,bus__pca_80pct__cont_0p030,bus,pca_80pct,0.030,50000,50000,50000,0,pass
5,bus__umap_pca_to_umap_10d__cont_0p005,bus,umap_pca_to_umap_10d,0.005,50000,50000,50000,0,pass
6,bus__umap_pca_to_umap_10d__cont_0p010,bus,umap_pca_to_umap_10d,0.010,50000,50000,50000,0,pass
7,fhvhv__pca_80pct__cont_0p005,fhvhv,pca_80pct,0.005,50000,50000,50000,0,pass
8,fhvhv__pca_80pct__cont_0p010,fhvhv,pca_80pct,0.010,50000,50000,50000,0,pass
9,fhvhv__pca_80pct__cont_0p015,fhvhv,pca_80pct,0.015,50000,50000,50000,0,pass


Findings\. The row\-level handoff is intact if this block passes: every retained candidate wrote successfully, every row\-level file exists, and each surviving branch still covers the full 50,000\-row shared calibration panel\. That means the next section can compare neighboring retained settings directly on the same observation set rather than worrying about missing or inconsistent row\-level outputs\.

### Compare neighboring retained settings within each modality\-representation branch

Now that the retained Isolation Forest branches have been materialized, we can compare neighboring contamination settings directly on the same rows\. This is the core local\-stability check for the framework: if a small contamination change causes only modest movement in anomaly flags and score behavior, that branch looks more trustworthy; if the anomaly surface jumps abruptly, then the retained region may still be too fragile to carry forward confidently\.

This part compares only neighboring retained settings within the same feature set and representation branch\. We are not yet comparing PCA versus UMAP directly here; we are first asking whether each branch is locally stable on its own terms\.

In [22]:
# ---------------------------------------------------------------------
# Compare neighboring retained settings within each modality-representation branch
# ---------------------------------------------------------------------

def load_if_stability_branch(candidate_id):
    manifest_match_df = if_stability_row_level_manifest_df.loc[
        if_stability_row_level_manifest_df["candidate_id"].eq(candidate_id)
    ].copy()

    if len(manifest_match_df) != 1:
        raise ValueError(
            f"Expected exactly one manifest row for candidate_id={candidate_id}, "
            f"found {len(manifest_match_df)}."
        )

    row_level_path = Path(manifest_match_df.iloc[0]["row_level_path"])
    if not row_level_path.exists():
        raise FileNotFoundError(f"Missing row-level file for {candidate_id}: {row_level_path}")

    return pd.read_parquet(row_level_path)


retained_candidates_for_neighbors_df = (
    if_stability_row_level_manifest_df[
        ["feature_set", "representation_name", "contamination", "candidate_id"]
    ]
    .drop_duplicates()
    .sort_values(["feature_set", "representation_name", "contamination"])
    .reset_index(drop=True)
)

neighbor_pair_records = []

for (
    feature_set,
    representation_name,
), branch_candidates_df in retained_candidates_for_neighbors_df.groupby(
    ["feature_set", "representation_name"],
    dropna=False,
    sort=True,
):
    branch_candidates_df = branch_candidates_df.sort_values("contamination").reset_index(drop=True)

    if len(branch_candidates_df) < 2:
        continue

    for left_idx in range(len(branch_candidates_df) - 1):
        left_row = branch_candidates_df.iloc[left_idx]
        right_row = branch_candidates_df.iloc[left_idx + 1]

        neighbor_pair_records.append(
            {
                "feature_set": feature_set,
                "representation_name": representation_name,
                "left_candidate_id": left_row["candidate_id"],
                "right_candidate_id": right_row["candidate_id"],
                "left_contamination": float(left_row["contamination"]),
                "right_contamination": float(right_row["contamination"]),
                "contamination_step": float(right_row["contamination"] - left_row["contamination"]),
            }
        )

if_neighbor_pairs_df = pd.DataFrame(neighbor_pair_records)

display(
    if_neighbor_pairs_df.style.format(
        {
            "left_contamination": "{:.3f}",
            "right_contamination": "{:.3f}",
            "contamination_step": "{:.3f}",
        }
    )
)

,feature_set,representation_name,left_candidate_id,right_candidate_id,left_contamination,right_contamination,contamination_step
0,bus,pca_80pct,bus__pca_80pct__cont_0p005,bus__pca_80pct__cont_0p010,0.005,0.010,0.005
1,bus,pca_80pct,bus__pca_80pct__cont_0p010,bus__pca_80pct__cont_0p015,0.010,0.015,0.005
2,bus,pca_80pct,bus__pca_80pct__cont_0p015,bus__pca_80pct__cont_0p020,0.015,0.020,0.005
3,bus,pca_80pct,bus__pca_80pct__cont_0p020,bus__pca_80pct__cont_0p030,0.020,0.030,0.010
4,bus,umap_pca_to_umap_10d,bus__umap_pca_to_umap_10d__cont_0p005,bus__umap_pca_to_umap_10d__cont_0p010,0.005,0.010,0.005
5,fhvhv,pca_80pct,fhvhv__pca_80pct__cont_0p005,fhvhv__pca_80pct__cont_0p010,0.005,0.010,0.005
6,fhvhv,pca_80pct,fhvhv__pca_80pct__cont_0p010,fhvhv__pca_80pct__cont_0p015,0.010,0.015,0.005
7,fhvhv,pca_80pct,fhvhv__pca_80pct__cont_0p015,fhvhv__pca_80pct__cont_0p020,0.015,0.020,0.005
8,fhvhv,pca_80pct,fhvhv__pca_80pct__cont_0p020,fhvhv__pca_80pct__cont_0p030,0.020,0.030,0.010
9,fhvhv,pca_80pct,fhvhv__pca_80pct__cont_0p030,fhvhv__pca_80pct__cont_0p040,0.030,0.040,0.010


Findings\. This setup block defines the neighboring retained comparisons that matter for local Isolation Forest stability: each pair represents the smallest surviving contamination step within a single modality\-representation branch\. If the retained candidate space is behaving sensibly, these are exactly the comparisons where we should expect the anomaly surface to move smoothly rather than jump abruptly\.
Now actually compare the neighboring retained surfaces row by row\.

In [23]:
# ---------------------------------------------------------------------
# Compare row-level overlap and score drift across neighboring retained settings
# ---------------------------------------------------------------------

if_neighbor_stability_records = []

for pair_row in if_neighbor_pairs_df.to_dict("records"):
    left_df = load_if_stability_branch(pair_row["left_candidate_id"])
    right_df = load_if_stability_branch(pair_row["right_candidate_id"])

    compare_df = left_df.merge(
        right_df[
            [
                MODEL_ID_COLUMN,
                "if_anomaly_flag",
                "if_decision_score",
                "if_raw_score",
            ]
        ].rename(
            columns={
                "if_anomaly_flag": "if_anomaly_flag_right",
                "if_decision_score": "if_decision_score_right",
                "if_raw_score": "if_raw_score_right",
            }
        ),
        on=MODEL_ID_COLUMN,
        how="inner",
        validate="one_to_one",
    )

    compare_df = compare_df.rename(
        columns={
            "if_anomaly_flag": "if_anomaly_flag_left",
            "if_decision_score": "if_decision_score_left",
            "if_raw_score": "if_raw_score_left",
        }
    )

    left_flags = compare_df["if_anomaly_flag_left"].to_numpy(dtype=int)
    right_flags = compare_df["if_anomaly_flag_right"].to_numpy(dtype=int)

    left_positive = left_flags.sum()
    right_positive = right_flags.sum()
    intersection = int(((left_flags == 1) & (right_flags == 1)).sum())
    union = int(((left_flags == 1) | (right_flags == 1)).sum())

    jaccard_similarity = intersection / union if union > 0 else np.nan

    anomaly_share_left = float(left_positive / len(compare_df))
    anomaly_share_right = float(right_positive / len(compare_df))
    anomaly_share_delta = abs(anomaly_share_right - anomaly_share_left)

    mean_abs_decision_score_shift = float(
        np.abs(
            compare_df["if_decision_score_right"] - compare_df["if_decision_score_left"]
        ).mean()
    )
    median_abs_decision_score_shift = float(
        np.abs(
            compare_df["if_decision_score_right"] - compare_df["if_decision_score_left"]
        ).median()
    )

    flipped_rows = int((left_flags != right_flags).sum())
    flipped_row_share = float(flipped_rows / len(compare_df))

    if_neighbor_stability_records.append(
        {
            "feature_set": pair_row["feature_set"],
            "representation_name": pair_row["representation_name"],
            "left_candidate_id": pair_row["left_candidate_id"],
            "right_candidate_id": pair_row["right_candidate_id"],
            "left_contamination": pair_row["left_contamination"],
            "right_contamination": pair_row["right_contamination"],
            "contamination_step": pair_row["contamination_step"],
            "rows_compared": len(compare_df),
            "left_anomaly_rows": int(left_positive),
            "right_anomaly_rows": int(right_positive),
            "flag_intersection_rows": intersection,
            "flag_union_rows": union,
            "flag_jaccard_similarity": jaccard_similarity,
            "anomaly_share_delta": anomaly_share_delta,
            "flipped_rows": flipped_rows,
            "flipped_row_share": flipped_row_share,
            "mean_abs_decision_score_shift": mean_abs_decision_score_shift,
            "median_abs_decision_score_shift": median_abs_decision_score_shift,
            "status": "pass",
        }
    )

if_neighbor_stability_df = (
    pd.DataFrame(if_neighbor_stability_records)
    .sort_values(
        ["feature_set", "representation_name", "left_contamination", "right_contamination"]
    )
    .reset_index(drop=True)
)

display(
    if_neighbor_stability_df.style.format(
        {
            "left_contamination": "{:.3f}",
            "right_contamination": "{:.3f}",
            "contamination_step": "{:.3f}",
            "flag_jaccard_similarity": "{:.3f}",
            "anomaly_share_delta": "{:.3f}",
            "flipped_row_share": "{:.3f}",
            "mean_abs_decision_score_shift": "{:.3f}",
            "median_abs_decision_score_shift": "{:.3f}",
        }
    )
)


,feature_set,representation_name,left_candidate_id,right_candidate_id,left_contamination,right_contamination,contamination_step,rows_compared,left_anomaly_rows,right_anomaly_rows,flag_intersection_rows,flag_union_rows,flag_jaccard_similarity,anomaly_share_delta,flipped_rows,flipped_row_share,mean_abs_decision_score_shift,median_abs_decision_score_shift,status
0,bus,pca_80pct,bus__pca_80pct__cont_0p005,bus__pca_80pct__cont_0p010,0.005,0.010,0.005,50000,250,500,250,500,0.500,0.005,250,0.005,0.044,0.044,pass
1,bus,pca_80pct,bus__pca_80pct__cont_0p010,bus__pca_80pct__cont_0p015,0.010,0.015,0.005,50000,500,750,500,750,0.667,0.005,250,0.005,0.024,0.024,pass
2,bus,pca_80pct,bus__pca_80pct__cont_0p015,bus__pca_80pct__cont_0p020,0.015,0.020,0.005,50000,750,1000,750,1000,0.750,0.005,250,0.005,0.020,0.020,pass
3,bus,pca_80pct,bus__pca_80pct__cont_0p020,bus__pca_80pct__cont_0p030,0.020,0.030,0.010,50000,1000,1500,1000,1500,0.667,0.010,500,0.010,0.022,0.022,pass
4,bus,umap_pca_to_umap_10d,bus__umap_pca_to_umap_10d__cont_0p005,bus__umap_pca_to_umap_10d__cont_0p010,0.005,0.010,0.005,50000,250,500,250,500,0.500,0.005,250,0.005,0.024,0.024,pass
5,fhvhv,pca_80pct,fhvhv__pca_80pct__cont_0p005,fhvhv__pca_80pct__cont_0p010,0.005,0.010,0.005,50000,250,500,250,500,0.500,0.005,250,0.005,0.034,0.034,pass
6,fhvhv,pca_80pct,fhvhv__pca_80pct__cont_0p010,fhvhv__pca_80pct__cont_0p015,0.010,0.015,0.005,50000,500,750,500,750,0.667,0.005,250,0.005,0.026,0.026,pass
7,fhvhv,pca_80pct,fhvhv__pca_80pct__cont_0p015,fhvhv__pca_80pct__cont_0p020,0.015,0.020,0.005,50000,750,1000,750,1000,0.750,0.005,250,0.005,0.019,0.019,pass
8,fhvhv,pca_80pct,fhvhv__pca_80pct__cont_0p020,fhvhv__pca_80pct__cont_0p030,0.020,0.030,0.010,50000,1000,1500,1000,1500,0.667,0.010,500,0.010,0.029,0.029,pass
9,fhvhv,pca_80pct,fhvhv__pca_80pct__cont_0p030,fhvhv__pca_80pct__cont_0p040,0.030,0.040,0.010,50000,1500,2000,1500,2000,0.750,0.010,500,0.010,0.019,0.019,pass


Findings\. The neighboring\-setting comparison block gives us the right unit of stability analysis: each comparison is a local contamination step within the same modality and representation branch, so we are measuring whether small retained\-setting changes behave smoothly or abruptly\. The retained PCA branches generally contribute the richer local comparison surface because they survived screening across more contamination values, while the weaker UMAP branches often have fewer neighboring pairs left to compare\. That is already informative on its own: some of UMAP’s instability story is showing up not just in weaker scores, but in the fact that there is less viable retained region left to test locally\.

To make this easier to absorb than a big table, add a compact branch\-level summary plus a visual\.

In [24]:
# ---------------------------------------------------------------------
# Summarize neighboring-setting stability in a reader-friendly way
# ---------------------------------------------------------------------

if_neighbor_stability_summary_df = (
    if_neighbor_stability_df.groupby(
        ["feature_set", "representation_name"],
        dropna=False,
    )
    .agg(
        neighbor_pairs=("left_candidate_id", "size"),
        mean_jaccard=("flag_jaccard_similarity", "mean"),
        min_jaccard=("flag_jaccard_similarity", "min"),
        max_jaccard=("flag_jaccard_similarity", "max"),
        mean_anomaly_share_delta=("anomaly_share_delta", "mean"),
        mean_flipped_row_share=("flipped_row_share", "mean"),
        mean_abs_score_shift=("mean_abs_decision_score_shift", "mean"),
    )
    .reset_index()
    .sort_values(["feature_set", "representation_name"])
    .reset_index(drop=True)
)

display(
    if_neighbor_stability_summary_df.style.format(
        {
            "mean_jaccard": "{:.3f}",
            "min_jaccard": "{:.3f}",
            "max_jaccard": "{:.3f}",
            "mean_anomaly_share_delta": "{:.3f}",
            "mean_flipped_row_share": "{:.3f}",
            "mean_abs_score_shift": "{:.3f}",
        }
    )
)

# ---------------------------------------------------------------------
# Visualize neighboring-setting stability within retained IF branches
# ---------------------------------------------------------------------

plot_df = if_neighbor_stability_summary_df.copy()
plot_df["representation_label"] = plot_df["representation_name"].map(
    {
        "pca_80pct": "PCA",
        "umap_pca_to_umap_10d": "UMAP",
    }
)

feature_set_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Mean Jaccard overlap",
        "Mean flipped-row share",
        "Mean abs. score shift",
    ],
    shared_yaxes=True,
    horizontal_spacing=0.08,
)

color_map = IF_REPRESENTATION_COLOR_MAP
symbol_map = {"PCA": "circle", "UMAP": "diamond"}

for representation_label in ["PCA", "UMAP"]:
    rep_df = plot_df.loc[plot_df["representation_label"].eq(representation_label)].copy()
    rep_df["feature_set"] = pd.Categorical(
        rep_df["feature_set"],
        categories=feature_set_order,
        ordered=True,
    )
    rep_df = rep_df.sort_values("feature_set")

    common_kwargs = dict(
        y=rep_df["feature_set"],
        mode="markers",
        name=representation_label,
        legendgroup=representation_label,
        showlegend=True,
        marker=dict(
            color=color_map[representation_label],
            symbol=symbol_map[representation_label],
            size=9,
        ),
        hovertemplate=(
            "Representation=" + representation_label
            + "<br>Feature set=%{y}"
            + "<br>Value=%{x:.3f}"
            + "<extra></extra>"
        ),
    )

    fig.add_trace(
        go.Scatter(
            x=rep_df["mean_jaccard"],
            **common_kwargs,
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=rep_df["mean_flipped_row_share"],
            showlegend=False,
            **{k: v for k, v in common_kwargs.items() if k != "showlegend"},
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Scatter(
            x=rep_df["mean_abs_score_shift"],
            showlegend=False,
            **{k: v for k, v in common_kwargs.items() if k != "showlegend"},
        ),
        row=1,
        col=3,
    )

fig.update_yaxes(categoryorder="array", categoryarray=feature_set_order[::-1], ticklabelstandoff=4)

fig.update_xaxes(title_text="Jaccard", row=1, col=1)
fig.update_xaxes(title_text="Flipped-row share", row=1, col=2)
fig.update_xaxes(title_text="Score shift", row=1, col=3)

# Adapt ranges to values
score_shift_min = if_neighbor_stability_summary_df["mean_abs_score_shift"].min()
score_shift_max = if_neighbor_stability_summary_df["mean_abs_score_shift"].max()
score_shift_pad = 0.002

fig.update_xaxes(
    range=[score_shift_min - score_shift_pad, score_shift_max + score_shift_pad],
    row=1,
    col=3,
)

fig.update_layout(
    title="Neighboring Retained Isolation Forest Setting Stability",
    width=980,
    height=420,
    margin=dict(l=90, r=30, t=70, b=55),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.18,
        xanchor="center",
        x=0.5,
    ),
)

fig.show()

,feature_set,representation_name,neighbor_pairs,mean_jaccard,min_jaccard,max_jaccard,mean_anomaly_share_delta,mean_flipped_row_share,mean_abs_score_shift
0,bus,pca_80pct,4,0.646,0.500,0.750,0.006,0.006,0.028
1,bus,umap_pca_to_umap_10d,1,0.500,0.500,0.500,0.005,0.005,0.024
2,fhvhv,pca_80pct,6,0.689,0.500,0.800,0.007,0.007,0.024
3,fhvhv,umap_pca_to_umap_10d,5,0.643,0.500,0.800,0.009,0.009,0.012
4,multimodal,pca_80pct,6,0.689,0.500,0.800,0.007,0.007,0.021
5,multimodal,umap_pca_to_umap_10d,5,0.666,0.496,0.750,0.007,0.007,0.015
6,subway,pca_80pct,6,0.689,0.500,0.800,0.007,0.007,0.025
7,taxi,pca_80pct,6,0.689,0.500,0.800,0.007,0.007,0.020
8,taxi,umap_pca_to_umap_10d,2,0.584,0.500,0.668,0.005,0.005,0.014


Findings\. The summary table and chart tell a fairly coherent stability story\. First, the good news: across the retained branches, neighboring contamination moves are local rather than catastrophic\. Mean anomaly\-share deltas and mean flipped\-row shares stay very small, generally around 0\.005 to 0\.009, which means small retained contamination changes are not causing the anomaly surface to completely reshuffle\. Second, PCA is usually as stable or more stable than UMAP on the main overlap criterion\. Bus, FHVHV, Multimodal, Subway, and Taxi PCA all sit around mean Jaccard values of roughly 0\.646 to 0\.689, while UMAP is weaker where it survives, especially for Bus \(0\.500\) and Taxi \(0\.584\)\. Third, UMAP is a mixed story rather than a total collapse: its score shifts are often smaller than PCA’s, but that smoother score movement does not translate into stronger row\-level overlap\. So the practical takeaway is that the retained Isolation Forest region looks locally stable enough to keep going, but PCA still carries the stronger stability case overall, while UMAP remains a narrower and less convincing branch that will need to justify itself in the representation\-sensitivity comparison\.

### Compare PCA and UMAP within the retained Isolation Forest region

Now that neighboring\-setting stability has been checked within each retained branch, the next question is whether PCA and UMAP are still telling meaningfully different anomaly stories inside Isolation Forest\. This part compares the strongest surviving PCA and UMAP settings within each modality and asks two things: how much they overlap on the same rows, and whether they concentrate anomalies in the same kinds of temporal and policy\-geography slices\. If UMAP is mostly reproducing PCA while remaining weaker on the earlier screening and stability checks, then its case for survival gets much thinner\.

Start by choosing one retained PCA branch and one retained UMAP branch per modality\. The comparison should be fair but conservative: prefer advance over caution, and within a decision level prefer the lowest surviving contamination value\.

In [25]:
# ---------------------------------------------------------------------
# Select representative retained PCA and UMAP branches for comparison
# ---------------------------------------------------------------------

if_representation_compare_candidates_df = if_contamination_screen_df.copy()

if_representation_compare_candidates_df["decision_rank"] = (
    if_representation_compare_candidates_df["screen_decision"].map(
        {
            "advance": 0,
            "caution": 1,
            "retire": 2,
        }
    )
)

if_representative_branch_selection_df = (
    if_representation_compare_candidates_df.loc[
        if_representation_compare_candidates_df["screen_decision"].isin(["advance", "caution"])
    ]
    .sort_values(
        ["feature_set", "representation_name", "decision_rank", "contamination"],
        ascending=[True, True, True, True],
    )
    .groupby(["feature_set", "representation_name"], dropna=False)
    .head(1)
    .reset_index(drop=True)
)

if_representative_branch_selection_df["selection_rationale"] = np.where(
    if_representative_branch_selection_df["screen_decision"].eq("advance"),
    "Lowest surviving advance setting for this modality-representation branch.",
    "Lowest surviving caution setting for this modality-representation branch.",
)

display(
    if_representative_branch_selection_df[
        [
            "feature_set",
            "representation_name",
            "contamination",
            "screen_decision",
            "tail_separation_gap",
            "median_separation",
            "selection_rationale",
        ]
    ].style.format(
        {
            "contamination": "{:.3f}",
            "tail_separation_gap": "{:.3f}",
            "median_separation": "{:.3f}",
        }
    )
)

,feature_set,representation_name,contamination,screen_decision,tail_separation_gap,median_separation,selection_rationale
0,bus,pca_80pct,0.005,advance,0.139,0.277,Lowest surviving advance setting for this modality-representation branch.
1,bus,umap_pca_to_umap_10d,0.005,caution,0.053,0.162,Lowest surviving caution setting for this modality-representation branch.
2,fhvhv,pca_80pct,0.005,advance,0.150,0.299,Lowest surviving advance setting for this modality-representation branch.
3,fhvhv,umap_pca_to_umap_10d,0.005,advance,0.065,0.218,Lowest surviving advance setting for this modality-representation branch.
4,multimodal,pca_80pct,0.005,advance,0.132,0.262,Lowest surviving advance setting for this modality-representation branch.
5,multimodal,umap_pca_to_umap_10d,0.005,caution,0.095,0.189,Lowest surviving caution setting for this modality-representation branch.
6,subway,pca_80pct,0.005,advance,0.156,0.333,Lowest surviving advance setting for this modality-representation branch.
7,taxi,pca_80pct,0.005,advance,0.130,0.272,Lowest surviving advance setting for this modality-representation branch.
8,taxi,umap_pca_to_umap_10d,0.005,caution,0.053,0.174,Lowest surviving caution setting for this modality-representation branch.


Findings\. This block selects one representative retained PCA branch and one representative retained UMAP branch per modality, always preferring the lowest surviving advance setting when available and otherwise falling back to the lowest surviving caution setting\. That gives the representation comparison a clean like\-for\-like basis without letting weaker high\-contamination settings distort the PCA\-versus\-UMAP readout\.

Compare PCA and UMAP row\-level anomaly overlap within each modality\. Now that each modality has a retained PCA branch and, where available, a retained UMAP branch, we can ask whether the two representations are flagging mostly the same rows or mostly different ones\. This block compares the retained branches at the row level using four simple quantities: shared\_flag\_rows is the number of rows both branches call anomalous; pca\_only\_rows and umap\_only\_rows are the rows unique to each representation; jaccard\_similarity summarizes overall overlap as shared rows divided by the full union of flagged rows; and umap\_only\_share\_of\_union helps us see how much of the combined anomaly surface is being contributed uniquely by UMAP\. anomaly\_share\_gap is included as a quick check that any difference is about anomaly identity rather than just anomaly volume\.

In [26]:
# ---------------------------------------------------------------------
# Compare PCA and UMAP row-level anomaly overlap within each modality
# ---------------------------------------------------------------------

def load_if_row_level_candidate(candidate_id):
    manifest_match_df = if_stability_row_level_manifest_df.loc[
        if_stability_row_level_manifest_df["candidate_id"].eq(candidate_id)
    ].copy()

    if len(manifest_match_df) != 1:
        raise ValueError(
            f"Expected exactly one manifest row for candidate_id={candidate_id}, "
            f"found {len(manifest_match_df)}."
        )

    row_level_path = Path(manifest_match_df.iloc[0]["row_level_path"])
    if not row_level_path.exists():
        raise FileNotFoundError(f"Missing row-level file for {candidate_id}: {row_level_path}")

    return pd.read_parquet(row_level_path)


representative_branch_manifest_df = if_stability_row_level_manifest_df.merge(
    if_representative_branch_selection_df[
        ["feature_set", "representation_name", "contamination"]
    ],
    on=["feature_set", "representation_name", "contamination"],
    how="inner",
    validate="one_to_one",
)

if_representation_overlap_records = []

for feature_set in sorted(if_representative_branch_selection_df["feature_set"].unique()):
    branch_pair_df = representative_branch_manifest_df.loc[
        representative_branch_manifest_df["feature_set"].eq(feature_set)
    ].copy()

    if branch_pair_df["representation_name"].nunique() != 2:
        continue

    pca_row = branch_pair_df.loc[
        branch_pair_df["representation_name"].eq("pca_80pct")
    ].iloc[0]
    umap_row = branch_pair_df.loc[
        branch_pair_df["representation_name"].eq("umap_pca_to_umap_10d")
    ].iloc[0]

    pca_df = load_if_row_level_candidate(pca_row["candidate_id"]).rename(
        columns={
            "if_anomaly_flag": "if_anomaly_flag_pca",
            "if_decision_score": "if_decision_score_pca",
            "if_raw_score": "if_raw_score_pca",
        }
    )
    umap_df = load_if_row_level_candidate(umap_row["candidate_id"]).rename(
        columns={
            "if_anomaly_flag": "if_anomaly_flag_umap",
            "if_decision_score": "if_decision_score_umap",
            "if_raw_score": "if_raw_score_umap",
        }
    )

    compare_df = pca_df.merge(
        umap_df[
            [
                MODEL_ID_COLUMN,
                "if_anomaly_flag_umap",
                "if_decision_score_umap",
                "if_raw_score_umap",
            ]
        ],
        on=MODEL_ID_COLUMN,
        how="inner",
        validate="one_to_one",
    )

    pca_flags = compare_df["if_anomaly_flag_pca"].to_numpy(dtype=int)
    umap_flags = compare_df["if_anomaly_flag_umap"].to_numpy(dtype=int)

    pca_count = int(pca_flags.sum())
    umap_count = int(umap_flags.sum())
    intersection = int(((pca_flags == 1) & (umap_flags == 1)).sum())
    union = int(((pca_flags == 1) | (umap_flags == 1)).sum())

    if_representation_overlap_records.append(
        {
            "feature_set": feature_set,
            "pca_contamination": float(pca_row["contamination"]),
            "umap_contamination": float(umap_row["contamination"]),
            "rows_compared": len(compare_df),
            "pca_flag_count": pca_count,
            "umap_flag_count": umap_count,
            "shared_flag_rows": intersection,
            "pca_only_rows": int(((pca_flags == 1) & (umap_flags == 0)).sum()),
            "umap_only_rows": int(((pca_flags == 0) & (umap_flags == 1)).sum()),
            "union_flag_rows": union,
            "jaccard_similarity": intersection / union if union > 0 else np.nan,
            "pca_anomaly_share": pca_count / len(compare_df),
            "umap_anomaly_share": umap_count / len(compare_df),
            "anomaly_share_gap": abs((pca_count / len(compare_df)) - (umap_count / len(compare_df))),
        }
    )

if_representation_overlap_df = (
    pd.DataFrame(if_representation_overlap_records)
    .sort_values("feature_set")
    .reset_index(drop=True)
)

display(
    if_representation_overlap_df.style.format(
        {
            "pca_contamination": "{:.3f}",
            "umap_contamination": "{:.3f}",
            "jaccard_similarity": "{:.3f}",
            "pca_anomaly_share": "{:.3f}",
            "umap_anomaly_share": "{:.3f}",
            "anomaly_share_gap": "{:.3f}",
        }
    )
)

# ---------------------------------------------------------------------
# Visualize PCA-versus-UMAP overlap within retained IF branches
# ---------------------------------------------------------------------

overlap_plot_df = if_representation_overlap_df.copy()

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Jaccard overlap",
        "Anomaly-share gap",
        "UMAP-only share of union",
    ],
    shared_yaxes=True,
    horizontal_spacing=0.10,
)

umap_only_share = overlap_plot_df["umap_only_rows"] / overlap_plot_df["union_flag_rows"]

fig.add_trace(
    go.Bar(
        x=overlap_plot_df["jaccard_similarity"],
        y=overlap_plot_df["feature_set"],
        orientation="h",
        marker_color=BRAND_COLORS["dark_teal"],
        text=overlap_plot_df["jaccard_similarity"].map(lambda x: f"{x:.3f}"),
        textposition="outside",
        showlegend=False,
        hovertemplate="Feature set=%{y}<br>Jaccard=%{x:.3f}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=overlap_plot_df["anomaly_share_gap"],
        y=overlap_plot_df["feature_set"],
        orientation="h",
        marker_color=BRAND_COLORS["terracotta"],
        text=overlap_plot_df["anomaly_share_gap"].map(lambda x: f"{x:.5f}"),
        textposition="outside",
        showlegend=False,
        hovertemplate="Feature set=%{y}<br>Anomaly-share gap=%{x:.3f}<extra></extra>",
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Bar(
        x=umap_only_share,
        y=overlap_plot_df["feature_set"],
        orientation="h",
        marker_color=BRAND_COLORS["seafoam"],
        text=umap_only_share.map(lambda x: f"{x:.3f}"),
        textposition="outside",
        showlegend=False,
        hovertemplate="Feature set=%{y}<br>UMAP-only share of union=%{x:.3f}<extra></extra>",
    ),
    row=1,
    col=3,
)

fig.update_yaxes(
    categoryorder="array",
    categoryarray=sorted(overlap_plot_df["feature_set"].tolist())[::-1],
    ticklabelstandoff=4,
)

fig.update_xaxes(title_text="Jaccard", row=1, col=1)
fig.update_xaxes(title_text="Share gap", row=1, col=2)
fig.update_xaxes(title_text="UMAP-only union share", row=1, col=3)

fig.update_layout(
    title="Retained Isolation Forest PCA-versus-UMAP Overlap",
    width=980,
    height=420,
    margin=dict(l=90, r=30, t=70, b=55),
)
fig.update_xaxes(range=[0, 0.15], row=1, col=1)
fig.update_xaxes(range=[0, 0.00006], row=1, col=2)
fig.update_xaxes(range=[0, 0.70], row=1, col=3)

apply_if_brand_layout(
    fig,
    title="PCA-versus-UMAP Overlap Within Retained Isolation Forest Branches",
)

fig.show()

,feature_set,pca_contamination,umap_contamination,rows_compared,pca_flag_count,umap_flag_count,shared_flag_rows,pca_only_rows,umap_only_rows,union_flag_rows,jaccard_similarity,pca_anomaly_share,umap_anomaly_share,anomaly_share_gap
0,bus,0.005,0.005,50000,250,250,51,199,199,449,0.114,0.005,0.005,0.000
1,fhvhv,0.005,0.005,50000,250,250,0,250,250,500,0.000,0.005,0.005,0.000
2,multimodal,0.005,0.005,50000,250,248,7,243,241,491,0.014,0.005,0.005,0.000
3,taxi,0.005,0.005,50000,250,250,0,250,250,500,0.000,0.005,0.005,0.000


Findings\. PCA and UMAP are producing almost completely different anomaly identities, even when they are calibrated to the same anomaly volume\. In Bus, both branches flag 250 rows but share only 51 of them, for a Jaccard overlap of 0\.114; in FHVHV and Taxi they share none at all, and in Multimodal they share only 7 rows, for a Jaccard of 0\.014\. The anomaly\-share gap is effectively zero everywhere, which means this is not a “one branch is broader than the other” story; it is a “same anomaly budget, different rows” story\. The UMAP\-only share of the union is also very large across all four modalities, which tells us that UMAP is not just reproducing PCA with a little noise around the edges\. So the practical takeaway is that retained UMAP branches are contributing a genuinely different anomaly surface, but one that now needs to justify itself on interpretability and concentration behavior, because distinctness alone is not the same thing as usefulness\.

Compare PCA and UMAP anomaly concentration across local comparison slices\. Overlap tells us whether PCA and UMAP flag the same rows, but it does not tell us whether those anomalies are distributed similarly across local operating regimes\. This block summarizes anomaly concentration across the 30 shared temporal\_bucket x policy\_geography\_class slices for each retained branch\. min\_slice\_anomaly\_share shows the quietest local slice, median\_slice\_anomaly\_share shows the typical slice, and max\_slice\_anomaly\_share shows the most anomaly\-heavy slice\. Read these as a compact concentration profile: lower medians suggest a flatter, more diffuse anomaly surface across slices, while higher maxima suggest that anomalies are piling up more sharply in a few specific local regimes\.

In [27]:
# ---------------------------------------------------------------------
# Compare PCA and UMAP anomaly concentration across local comparison slices
# ---------------------------------------------------------------------

if_representation_concentration_records = []

for feature_set in sorted(if_representative_branch_selection_df["feature_set"].unique()):
    branch_pair_df = representative_branch_manifest_df.loc[
        representative_branch_manifest_df["feature_set"].eq(feature_set)
    ].copy()

    if branch_pair_df["representation_name"].nunique() != 2:
        continue

    pca_row = branch_pair_df.loc[
        branch_pair_df["representation_name"].eq("pca_80pct")
    ].iloc[0]
    umap_row = branch_pair_df.loc[
        branch_pair_df["representation_name"].eq("umap_pca_to_umap_10d")
    ].iloc[0]

    pca_df = load_if_row_level_candidate(pca_row["candidate_id"]).copy()
    umap_df = load_if_row_level_candidate(umap_row["candidate_id"]).copy()

    for representation_label, branch_df in [("PCA", pca_df), ("UMAP", umap_df)]:
        slice_summary_df = (
            branch_df.groupby(
                [TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN],
                dropna=False,
            )
            .agg(
                total_rows=(MODEL_ID_COLUMN, "size"),
                anomaly_rows=("if_anomaly_flag", "sum"),
            )
            .reset_index()
        )
        slice_summary_df["anomaly_share"] = (
            slice_summary_df["anomaly_rows"] / slice_summary_df["total_rows"]
        )

        if_representation_concentration_records.append(
            {
                "feature_set": feature_set,
                "representation_label": representation_label,
                "comparison_slices": len(slice_summary_df),
                "min_slice_anomaly_share": float(slice_summary_df["anomaly_share"].min()),
                "median_slice_anomaly_share": float(slice_summary_df["anomaly_share"].median()),
                "max_slice_anomaly_share": float(slice_summary_df["anomaly_share"].max()),
            }
        )

if_representation_concentration_summary_df = (
    pd.DataFrame(if_representation_concentration_records)
    .sort_values(["feature_set", "representation_label"])
    .reset_index(drop=True)
)

display(
    if_representation_concentration_summary_df.style.format(
        {
            "min_slice_anomaly_share": "{:.3f}",
            "median_slice_anomaly_share": "{:.3f}",
            "max_slice_anomaly_share": "{:.3f}",
        }
    )
)
# ---------------------------------------------------------------------
# Visualize PCA-versus-UMAP slice concentration across retained branches
# ---------------------------------------------------------------------

concentration_plot_df = (
    if_representation_concentration_summary_df.copy()
    .assign(
        representation_label=lambda df: df["representation_label"].astype(str),
        feature_set=lambda df: pd.Categorical(
            df["feature_set"],
            categories=["bus", "subway", "taxi", "fhvhv", "multimodal"],
            ordered=True,
        ),
    )
    .sort_values(["feature_set", "representation_label"])
    .reset_index(drop=True)
)

plot_rows = []
for _, row in concentration_plot_df.iterrows():
    plot_rows.extend(
        [
            {
                "feature_set": row["feature_set"],
                "representation_label": row["representation_label"],
                "metric_label": "Median slice share",
                "metric_value": row["median_slice_anomaly_share"],
            },
            {
                "feature_set": row["feature_set"],
                "representation_label": row["representation_label"],
                "metric_label": "Max slice share",
                "metric_value": row["max_slice_anomaly_share"],
            },
        ]
    )

concentration_long_df = pd.DataFrame(plot_rows)

fig = px.bar(
    concentration_long_df,
    x="feature_set",
    y="metric_value",
    color="representation_label",
    barmode="group",
    facet_col="metric_label",
    category_orders={
        "feature_set": ["bus", "subway", "taxi", "fhvhv", "multimodal"],
        "representation_label": ["PCA", "UMAP"],
        "metric_label": ["Median slice share", "Max slice share"],
    },
    color_discrete_map=IF_REPRESENTATION_COLOR_MAP,
    labels={
        "feature_set": "Feature set",
        "metric_value": "Anomaly share",
        "representation_label": "Representation",
        "metric_label": "",
    },
    title="Retained Isolation Forest Slice Concentration: PCA vs UMAP",
)

fig.update_traces(
    texttemplate="%{y:.3f}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_layout(
    width=930,
    height=460,
    margin=dict(l=60, r=40, t=80, b=60),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.18,
        xanchor="center",
        x=0.5,
        title_text="",
    ),
)

fig.update_yaxes(tickformat=".3f", rangemode="tozero")
fig.update_xaxes(title_text="Feature set")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

apply_if_brand_layout(
    fig,
    title="PCA-versus-UMAP Slice Concentration Across Retained Branches",
)

fig.show()

,feature_set,representation_label,comparison_slices,min_slice_anomaly_share,median_slice_anomaly_share,max_slice_anomaly_share
0,bus,PCA,30,0.000,0.001,0.074
1,bus,UMAP,30,0.000,0.000,0.010
2,fhvhv,PCA,30,0.000,0.003,0.037
3,fhvhv,UMAP,30,0.000,0.000,0.010
4,multimodal,PCA,30,0.000,0.003,0.039
5,multimodal,UMAP,30,0.000,0.000,0.012
6,taxi,PCA,30,0.000,0.005,0.056
7,taxi,UMAP,30,0.000,0.000,0.014


Findings\. This chart compares how concentrated the retained PCA and UMAP anomaly surfaces are across the 30 local temporal\_bucket x policy\_geography\_class slices\. Median slice share is the typical anomaly burden of a local slice, while max slice share is the heaviest anomaly burden found in any single slice\. On both metrics, PCA is consistently above UMAP in Bus, Taxi, FHVHV, and Multimodal\. For example, Bus reaches a maximum slice share of 0\.074 under PCA versus 0\.010 under UMAP, and Taxi reaches 0\.056 versus 0\.014\. The median values are also higher for PCA, though they are small in absolute terms because most local slices remain fairly quiet\.

In plain language, that means PCA is more willing to say, “this specific kind of place at this specific kind of time looks unusual,” while UMAP spreads its anomaly flags more thinly across many slices\. So the takeaway is that PCA is producing sharper, more situation\-specific anomaly pockets, whereas UMAP is producing a flatter and less decisive anomaly pattern\. If our goal is to surface anomalies that can be tied back to recognizable operating contexts, this chart supports carrying PCA forward as the more interpretable Isolation Forest representation\.

> 🎯 Decision\. Retire UMAP from the shared Isolation Forest downstream path and carry PCA forward as the sole primary representation for this framework\. PCA preserves stronger anomaly\-tail separation, produces more locally concentrated and interpretable anomaly surfaces, and gives a cleaner calibration story across modalities\. Retained UMAP branches do show that representation choice can materially change which rows are flagged, but the resulting UMAP anomaly surfaces are generally weaker, flatter, and less operationally legible\. For this project stage, that makes UMAP better understood as a sensitivity check we have now completed, rather than a representation branch that still needs to advance into full downstream anomaly generation\.

### Select final contamination setting per modality

We now have enough calibration evidence to stop treating contamination as an open tuning range\. The screening step told us which settings still preserve a clean anomaly tail, the stability step showed how sensitive neighboring retained settings are, and the representation review told us to stay on PCA for downstream Isolation Forest generation\.

This block converts that evidence into one final contamination choice per modality\. The selection rule is intentionally conservative: prefer the lowest retained PCA contamination that still survived screening, because lower contamination preserves the sharpest anomaly tail and keeps the resulting anomaly surface sparse enough to remain interpretable\.

In [28]:
# ---------------------------------------------------------------------
# Select final contamination setting per modality
# ---------------------------------------------------------------------

IF_FINAL_CONTAMINATION_SELECTION_PATH = (
    INTERMEDIATE_333_DIR / "if_final_contamination_selection.parquet"
)

assert "if_contamination_screen_df" in globals(), (
    "if_contamination_screen_df is not in memory. Please run the contamination screening block first."
)

screen_df = if_contamination_screen_df.copy()

# Restrict to retained PCA candidates only.
pca_retained_df = (
    screen_df.loc[
        screen_df["representation_name"].eq("pca_80pct")
        & screen_df["screen_decision"].eq("advance")
    ]
    .copy()
)

assert not pca_retained_df.empty, (
    "No retained PCA contamination settings were found. Please review the contamination screening block."
)

# Conservative rule:
# choose the lowest retained PCA contamination for each modality.
if_final_contamination_selection_df = (
    pca_retained_df.sort_values(
        ["feature_set", "contamination"],
        ascending=[True, True],
    )
    .groupby("feature_set", as_index=False)
    .first()
    .reset_index(drop=True)
)

if_final_contamination_selection_df = if_final_contamination_selection_df[
    [
        "feature_set",
        "representation_name",
        "representation_label",
        "contamination",
        "tail_separation_gap",
        "median_separation",
        "screen_decision",
    ]
].copy()

selection_rationale_map = {
    "bus": (
        "Selected the lowest retained PCA contamination to preserve the sharpest "
        "anomaly tail and keep the bus anomaly surface sparse and interpretable."
    ),
    "subway": (
        "Selected the lowest retained PCA contamination because PCA remained strong "
        "throughout screening, and the lowest setting preserves the cleanest tail."
    ),
    "taxi": (
        "Selected the lowest retained PCA contamination to keep Taxi anomalies narrow "
        "and clearly separated rather than drifting into a broader anomaly slice."
    ),
    "fhvhv": (
        "Selected the lowest retained PCA contamination because it keeps the strongest "
        "tail separation while avoiding unnecessary anomaly expansion."
    ),
    "multimodal": (
        "Selected the lowest retained PCA contamination because multimodal anomalies "
        "are most interpretable when the tail remains sparse and sharply separated."
    ),
}

if_final_contamination_selection_df["selection_rationale"] = (
    if_final_contamination_selection_df["feature_set"].map(selection_rationale_map)
)

if should_rebuild(REBUILD_IF_CONTAMINATION_CALIBRATION, IF_FINAL_CONTAMINATION_SELECTION_PATH):
    if_final_contamination_selection_df.to_parquet(
        IF_FINAL_CONTAMINATION_SELECTION_PATH,
        index=False,
    )

display(
    if_final_contamination_selection_df[
        [
            "feature_set",
            "representation_label",
            "contamination",
            "tail_separation_gap",
            "median_separation",
            "selection_rationale",
        ]
    ].style.format(
        {
            "contamination": "{:.3f}",
            "tail_separation_gap": "{:.3f}",
            "median_separation": "{:.3f}",
        }
    )
)

,feature_set,representation_label,contamination,tail_separation_gap,median_separation,selection_rationale
0,bus,PCA,0.005,0.139,0.277,Selected the lowest retained PCA contamination to preserve the sharpest anomaly tail and keep the bus anomaly surface sparse and interpretable.
1,fhvhv,PCA,0.005,0.150,0.299,Selected the lowest retained PCA contamination because it keeps the strongest tail separation while avoiding unnecessary anomaly expansion.
2,multimodal,PCA,0.005,0.132,0.262,Selected the lowest retained PCA contamination because multimodal anomalies are most interpretable when the tail remains sparse and sharply separated.
3,subway,PCA,0.005,0.156,0.333,"Selected the lowest retained PCA contamination because PCA remained strong throughout screening, and the lowest setting preserves the cleanest tail."
4,taxi,PCA,0.005,0.130,0.272,Selected the lowest retained PCA contamination to keep Taxi anomalies narrow and clearly separated rather than drifting into a broader anomaly slice.


Findings\. The final contamination choice is deliberately conservative: every modality retains PCA @ 0\.005\. That is the lowest PCA setting that survived screening across the board, and it keeps the anomaly tail as sharp and sparse as possible while still leaving a viable downstream Isolation Forest branch in every modality\.

In [29]:
# ---------------------------------------------------------------------
# Validate final contamination selection handoff
# ---------------------------------------------------------------------

if_final_contamination_selection_validation_df = pd.DataFrame(
    [
        {
            "selected_feature_sets": if_final_contamination_selection_df["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "all_pca": bool(
                if_final_contamination_selection_df["representation_name"].eq("pca_80pct").all()
            ),
            "all_same_contamination": bool(
                if_final_contamination_selection_df["contamination"].eq(0.005).all()
            ),
            "status": (
                "pass"
                if (
                    if_final_contamination_selection_df["feature_set"].nunique()
                    == len(MODEL_FEATURE_SET_NAMES)
                    and if_final_contamination_selection_df["representation_name"].eq("pca_80pct").all()
                    and if_final_contamination_selection_df["contamination"].eq(0.005).all()
                )
                else "review"
            ),
        }
    ]
)

display(if_final_contamination_selection_validation_df)

assert if_final_contamination_selection_validation_df["status"].eq("pass").all(), (
    "Final Isolation Forest contamination selection is incomplete or inconsistent."
)

,selected_feature_sets,expected_feature_sets,all_pca,all_same_contamination,status
0,5,5,True,True,pass


Findings\. The final contamination handoff is complete and consistent: all five modalities now have one selected Isolation Forest branch, every selected branch is PCA\-based, and every selected contamination level is 0\.005\. That gives the downstream recording and export steps a single clear configuration per modality rather than a retained range\.

### Record the branches advancing beyond stability review

We now have a complete Isolation Forest calibration decision\. PCA has emerged as the retained representation, and the final contamination choice has been fixed at the most conservative retained setting for each modality\.

This block records the exact branches advancing downstream so the next section can generate final Isolation Forest outputs from an explicit handoff table rather than from a loose set of calibration conclusions\.

In [30]:
# ---------------------------------------------------------------------
# Record the branches advancing beyond stability review
# ---------------------------------------------------------------------

IF_RETAINED_BRANCHES_PATH = (
    INTERMEDIATE_333_DIR / "if_retained_branches_after_stability_review.parquet"
)

assert "if_final_contamination_selection_df" in globals(), (
    "if_final_contamination_selection_df is not in memory. Please run the final contamination selection block first."
)

if_retained_branches_df = if_final_contamination_selection_df.copy()

if_retained_branches_df["downstream_status"] = "shared_default"
if_retained_branches_df["downstream_role"] = "primary_if_branch"

recording_rationale_map = {
    "bus": (
        "Advance PCA at 0.005 contamination as the retained Bus Isolation Forest branch; "
        "this keeps the anomaly tail sparse, well-separated, and more interpretable than UMAP."
    ),
    "subway": (
        "Advance PCA at 0.005 contamination as the retained Subway Isolation Forest branch; "
        "PCA preserved the strongest anomaly-tail separation while UMAP was screened out."
    ),
    "taxi": (
        "Advance PCA at 0.005 contamination as the retained Taxi Isolation Forest branch; "
        "this keeps the anomaly surface narrow and better localized than the retained UMAP alternatives."
    ),
    "fhvhv": (
        "Advance PCA at 0.005 contamination as the retained FHVHV Isolation Forest branch; "
        "both branches were somewhat viable, but PCA carried the cleaner screening and stability story."
    ),
    "multimodal": (
        "Advance PCA at 0.005 contamination as the retained Multimodal Isolation Forest branch; "
        "PCA preserved stronger local anomaly pockets, while UMAP stayed flatter and less decision-useful."
    ),
}

if_retained_branches_df["recording_rationale"] = (
    if_retained_branches_df["feature_set"].map(recording_rationale_map)
)

if_retained_branch_retirement_reference_df = pd.DataFrame(
    [
        {
            "representation_name": "umap_pca_to_umap_10d",
            "downstream_status": "retired_for_if",
            "retirement_reason": (
                "UMAP did not carry a strong enough retained anomaly-tail story for Isolation Forest. "
                "Where it survived screening, it overlapped only weakly with PCA and produced flatter, "
                "less locally concentrated anomaly surfaces."
            ),
        }
    ]
)

if should_rebuild(REBUILD_IF_CONTAMINATION_CALIBRATION, IF_RETAINED_BRANCHES_PATH):
    if_retained_branches_df.to_parquet(
        IF_RETAINED_BRANCHES_PATH,
        index=False,
    )

display(
    if_retained_branches_df[
        [
            "feature_set",
            "representation_label",
            "contamination",
            "tail_separation_gap",
            "median_separation",
            "downstream_status",
            "downstream_role",
            "recording_rationale",
        ]
    ].style.format(
        {
            "contamination": "{:.3f}",
            "tail_separation_gap": "{:.3f}",
            "median_separation": "{:.3f}",
        }
    )
)

display(if_retained_branch_retirement_reference_df)

,feature_set,representation_label,contamination,tail_separation_gap,median_separation,downstream_status,downstream_role,recording_rationale
0,bus,PCA,0.005,0.139,0.277,shared_default,primary_if_branch,"Advance PCA at 0.005 contamination as the retained Bus Isolation Forest branch; this keeps the anomaly tail sparse, well-separated, and more interpretable than UMAP."
1,fhvhv,PCA,0.005,0.150,0.299,shared_default,primary_if_branch,"Advance PCA at 0.005 contamination as the retained FHVHV Isolation Forest branch; both branches were somewhat viable, but PCA carried the cleaner screening and stability story."
2,multimodal,PCA,0.005,0.132,0.262,shared_default,primary_if_branch,"Advance PCA at 0.005 contamination as the retained Multimodal Isolation Forest branch; PCA preserved stronger local anomaly pockets, while UMAP stayed flatter and less decision-useful."
3,subway,PCA,0.005,0.156,0.333,shared_default,primary_if_branch,Advance PCA at 0.005 contamination as the retained Subway Isolation Forest branch; PCA preserved the strongest anomaly-tail separation while UMAP was screened out.
4,taxi,PCA,0.005,0.130,0.272,shared_default,primary_if_branch,Advance PCA at 0.005 contamination as the retained Taxi Isolation Forest branch; this keeps the anomaly surface narrow and better localized than the retained UMAP alternatives.


,representation_name,downstream_status,retirement_reason
0,umap_pca_to_umap_10d,retired_for_if,"UMAP did not carry a strong enough retained anomaly-tail story for Isolation Forest. Where it survived screening, it overlapped only weakly with PCA and produced flatter, less locally concentrated anomaly surfaces."


Findings\. The retained Isolation Forest handoff is now explicit: every modality advances through a single shared\-default PCA branch at 0\.005 contamination\. In practical terms, that means the downstream notebook inherits one sparse, well\-separated anomaly surface per modality rather than a menu of still\-open calibration options\.

In [31]:
# ---------------------------------------------------------------------
# Validate retained-branch handoff after stability review
# ---------------------------------------------------------------------

if_retained_branch_validation_df = pd.DataFrame(
    [
        {
            "retained_feature_sets": if_retained_branches_df["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "retained_rows": len(if_retained_branches_df),
            "all_pca": bool(
                if_retained_branches_df["representation_name"].eq("pca_80pct").all()
            ),
            "all_contamination_0005": bool(
                if_retained_branches_df["contamination"].eq(0.005).all()
            ),
            "status": (
                "pass"
                if (
                    if_retained_branches_df["feature_set"].nunique()
                    == len(MODEL_FEATURE_SET_NAMES)
                    and if_retained_branches_df["representation_name"].eq("pca_80pct").all()
                    and if_retained_branches_df["contamination"].eq(0.005).all()
                )
                else "review"
            ),
        }
    ]
)

display(if_retained_branch_validation_df)

assert if_retained_branch_validation_df["status"].eq("pass").all(), (
    "Isolation Forest retained-branch handoff is incomplete or inconsistent."
)

,retained_feature_sets,expected_feature_sets,retained_rows,all_pca,all_contamination_0005,status
0,5,5,5,True,True,pass


Findings\. The retained\-branch validation passes cleanly: all five feature sets now have exactly one advancing Isolation Forest branch, every advancing branch is PCA\-based, and every selected contamination level is 0\.005\. That gives the export section a complete and fully resolved downstream specification\.

### Section Summary

This section converted Isolation Forest from a broad candidate framework into a single clean downstream configuration per modality\. The contamination sweep confirmed that anomaly prevalence behaves mechanically, so the real calibration signal came from anomaly\-tail separation, local stability, and representation sensitivity rather than from anomaly counts alone\. On those tests, PCA consistently carried the stronger story: it preserved sharper separation between anomalous and normal rows, survived screening more often and more cleanly than UMAP, and produced anomaly surfaces that were more locally concentrated and easier to interpret\. UMAP did demonstrate that representation choice can materially change which rows are flagged, but the resulting anomaly surfaces were usually flatter, weaker, and less operationally legible\. The practical outcome is straightforward: Isolation Forest advances as a viable candidate anomaly framework, but it advances through a single PCA\-based branch in every modality, using a conservative 0\.005 contamination setting to keep the anomaly tail sparse, stable, and interpretable for downstream framework comparison\.

## 3\.3\.3\.4 Compare Isolation Forest and DBSCAN Anomaly Surfaces

Isolation Forest and DBSCAN identify anomalous observations using fundamentally different assumptions, so this section performs a lightweight complementarity check rather than a full framework bakeoff\. The goal is to understand whether Isolation Forest mostly recreates the same anomaly surface already captured by DBSCAN or whether it contributes a meaningfully different slice of mobility\-state irregularity that is worth carrying forward\.

### Prepare the retained Isolation Forest and DBSCAN comparison surfaces

Prepare the retained framework comparison surfaces\. Before comparing the two methods, we need one explicit row\-level branch per modality from each framework\. This block resolves the retained Isolation Forest outputs and the retained DBSCAN outputs into a single comparison manifest that the rest of the section can reuse\.

In [32]:
# ---------------------------------------------------------------------
# Prepare retained Isolation Forest and DBSCAN comparison surfaces
# Calibration-universe only
# ---------------------------------------------------------------------

IF_DBSCAN_COMPARISON_MANIFEST_PATH = (
    INTERMEDIATE_333_DIR / "if_dbscan_comparison_manifest.parquet"
)

assert "if_final_contamination_selection_df" in globals(), (
    "if_final_contamination_selection_df is not in memory. Please run the final Isolation Forest contamination-selection block first."
)
assert "if_stability_row_level_manifest_df" in globals(), (
    "if_stability_row_level_manifest_df is not in memory. Please run the retained Isolation Forest row-level materialization block first."
)
assert DBSCAN_SELECTED_CONFIGURATION_PATH.exists(), (
    f"Missing retained DBSCAN configuration table: {DBSCAN_SELECTED_CONFIGURATION_PATH}"
)
assert DBSCAN_FINAL_ROW_LEVEL_MANIFEST_PATH.exists(), (
    f"Missing DBSCAN export manifest: {DBSCAN_FINAL_ROW_LEVEL_MANIFEST_PATH}"
)
assert DBSCAN_FINAL_ROW_LEVEL_DIR.exists(), (
    f"Missing DBSCAN final row-level directory: {DBSCAN_FINAL_ROW_LEVEL_DIR}"
)

def resolve_if_manifest_path_column(manifest_df: pd.DataFrame) -> str:
    path_col_candidates = [
        "row_level_output_path",
        "row_level_path",
        "output_path",
        "path",
    ]
    path_col = next((c for c in path_col_candidates if c in manifest_df.columns), None)
    assert path_col is not None, (
        "Could not identify the Isolation Forest row-level path column in if_stability_row_level_manifest_df."
    )
    return path_col


def resolve_dbscan_row_level_path(feature_set: str, representation_name: str = None) -> str:
    candidate_paths = []

    if representation_name:
        candidate_paths.append(
            DBSCAN_FINAL_ROW_LEVEL_DIR / f"{feature_set}_{representation_name}_dbscan_candidate_anomalies.parquet"
        )
        candidate_paths.append(
            DBSCAN_FINAL_ROW_LEVEL_DIR / f"{feature_set}_{representation_name}_dbscan_candidate_anomaly_rows.parquet"
        )

    candidate_paths.append(
        DBSCAN_FINAL_ROW_LEVEL_DIR / f"{feature_set}_dbscan_candidate_anomalies.parquet"
    )
    candidate_paths.append(
        DBSCAN_FINAL_ROW_LEVEL_DIR / f"{feature_set}_dbscan_candidate_anomaly_rows.parquet"
    )

    for candidate_path in candidate_paths:
        if candidate_path.exists():
            return str(candidate_path)

    matching_paths = sorted(DBSCAN_FINAL_ROW_LEVEL_DIR.glob(f"*{feature_set}*.parquet"))
    if representation_name:
        representation_matches = [
            path for path in matching_paths if representation_name in path.stem
        ]
        if len(representation_matches) == 1:
            return str(representation_matches[0])

    assert len(matching_paths) == 1, (
        f"Could not resolve a unique DBSCAN row-level parquet for {feature_set}. "
        f"Candidate matches: {[str(path) for path in matching_paths]}"
    )
    return str(matching_paths[0])


def resolve_if_row_level_path(feature_set: str, manifest_path: str, representation_name: str = None) -> str:
    manifest_path = Path(manifest_path)
    if manifest_path.exists():
        return str(manifest_path)

    candidate_paths = []

    if representation_name:
        candidate_paths.append(
            IF_FINAL_ROW_LEVEL_DIR / f"{feature_set}_{representation_name}_if_candidate_anomalies.parquet"
        )

    candidate_paths.append(
        IF_FINAL_ROW_LEVEL_DIR / f"{feature_set}_if_candidate_anomalies.parquet"
    )

    for candidate_path in candidate_paths:
        if candidate_path.exists():
            return str(candidate_path)

    matching_paths = sorted(IF_FINAL_ROW_LEVEL_DIR.glob(f"*{feature_set}*.parquet"))
    if representation_name:
        representation_matches = [
            path for path in matching_paths if representation_name in path.stem
        ]
        if len(representation_matches) == 1:
            return str(representation_matches[0])

    assert len(matching_paths) == 1, (
        f"Could not resolve a unique Isolation Forest row-level parquet for {feature_set}. "
        f"Manifest path tried: {manifest_path}. "
        f"Candidate matches: {[str(path) for path in matching_paths]}"
    )
    return str(matching_paths[0])


def _ensure_metadata_columns(df, feature_set, required_cols=("modeling_row_id", "temporal_bucket", "policy_geography_class")):
    missing_cols = [c for c in required_cols if c not in df.columns]
    if not missing_cols:
        return df

    metadata_df = pd.read_parquet(ROW_METADATA_PATHS_BY_SET[feature_set])[
        ["modeling_row_id"] + [c for c in missing_cols if c != "modeling_row_id"]
    ].drop_duplicates(subset=["modeling_row_id"])

    return df.merge(metadata_df, on="modeling_row_id", how="left")


def _detect_anomaly_flag(df, framework_name):
    candidate_cols = [
        "is_anomaly",
        "anomaly_flag",
        "anomaly_like_flag",
        "if_anomaly_flag",
        "dbscan_anomaly_flag",
    ]
    for c in candidate_cols:
        if c in df.columns:
            return df[c].fillna(False).astype(bool)

    if framework_name == "dbscan" and "cluster_label" in df.columns:
        return df["cluster_label"].eq(-1)

    raise AssertionError(
        f"Could not identify anomaly flag column for framework={framework_name}. "
        f"Available columns: {list(df.columns)}"
    )


def _filter_to_calibration_universe(df, feature_set):
    calibration_ids_df = pd.read_parquet(
        ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set],
        columns=[MODEL_ID_COLUMN],
    ).drop_duplicates(subset=[MODEL_ID_COLUMN])

    return df.merge(
        calibration_ids_df,
        on=MODEL_ID_COLUMN,
        how="inner",
        validate="many_to_one",
    )


if (
    not should_rebuild(REBUILD_IF_DBSCAN_COMPARISONS, IF_DBSCAN_COMPARISON_PATH)
    and IF_DBSCAN_COMPARISON_MANIFEST_PATH.exists()
    and IF_DBSCAN_COMPARISON_PATH.exists()
):
    if_dbscan_comparison_manifest_df = pd.read_parquet(IF_DBSCAN_COMPARISON_MANIFEST_PATH)
    if_dbscan_comparison_row_level_df = pd.read_parquet(IF_DBSCAN_COMPARISON_PATH)

else:
    if_path_col = resolve_if_manifest_path_column(if_stability_row_level_manifest_df)

    dbscan_selected_configurations_df = pd.read_csv(DBSCAN_SELECTED_CONFIGURATION_PATH)
    dbscan_final_row_level_manifest_df = pd.read_csv(DBSCAN_FINAL_ROW_LEVEL_MANIFEST_PATH)

    if_manifest_match_df = if_stability_row_level_manifest_df.merge(
        if_final_contamination_selection_df[
            ["feature_set", "representation_name", "contamination"]
        ].drop_duplicates(),
        on=["feature_set", "representation_name", "contamination"],
        how="inner",
    )

    assert not if_manifest_match_df.empty, (
        "No retained Isolation Forest row-level branches matched the final contamination selection."
    )

    dbscan_join_keys = [
        column_name
        for column_name in ["feature_set", "representation_name", "min_samples", "eps_band", "eps_value"]
        if column_name in dbscan_selected_configurations_df.columns
        and column_name in dbscan_final_row_level_manifest_df.columns
    ]

    assert dbscan_join_keys, (
        "Could not find shared DBSCAN join keys between the selected-configuration table and export manifest."
    )

    dbscan_manifest_match_df = dbscan_final_row_level_manifest_df.merge(
        dbscan_selected_configurations_df[dbscan_join_keys].drop_duplicates(),
        on=dbscan_join_keys,
        how="inner",
    )

    assert not dbscan_manifest_match_df.empty, (
        "No retained DBSCAN row-level branches matched the selected DBSCAN configurations."
    )

    if_dbscan_comparison_manifest_df = (
        if_manifest_match_df[
            ["feature_set", "representation_name", "contamination", if_path_col]
        ]
        .rename(
            columns={
                "representation_name": "if_representation_name",
                if_path_col: "if_row_level_path",
            }
        )
        .merge(
            dbscan_manifest_match_df[
                [
                    "feature_set",
                    "representation_name",
                    "min_samples",
                    "eps_band",
                    "eps_value",
                ]
            ].rename(
                columns={
                    "representation_name": "dbscan_representation_name",
                }
            ),
            on="feature_set",
            how="inner",
        )
        .sort_values("feature_set")
        .drop_duplicates(subset=["feature_set"])
        .reset_index(drop=True)
    )

    if_dbscan_comparison_manifest_df["if_row_level_path"] = if_dbscan_comparison_manifest_df.apply(
        lambda row: resolve_if_row_level_path(
            row["feature_set"],
            row["if_row_level_path"],
            row["if_representation_name"],
        ),
        axis=1,
    )

    if_dbscan_comparison_manifest_df["dbscan_row_level_path"] = if_dbscan_comparison_manifest_df.apply(
        lambda row: resolve_dbscan_row_level_path(
            row["feature_set"],
            row["dbscan_representation_name"],
        ),
        axis=1,
    )

    if_dbscan_comparison_manifest_df["if_path_exists"] = if_dbscan_comparison_manifest_df[
        "if_row_level_path"
    ].map(lambda path: Path(path).exists())
    if_dbscan_comparison_manifest_df["dbscan_path_exists"] = if_dbscan_comparison_manifest_df[
        "dbscan_row_level_path"
    ].map(lambda path: Path(path).exists())

    display(if_dbscan_comparison_manifest_df)

    assert if_dbscan_comparison_manifest_df["if_path_exists"].all(), (
        "One or more retained Isolation Forest row-level outputs is missing."
    )
    assert if_dbscan_comparison_manifest_df["dbscan_path_exists"].all(), (
        "One or more retained DBSCAN row-level outputs is missing."
    )

    comparison_frames = []

    for _, row in if_dbscan_comparison_manifest_df.iterrows():
        feature_set = row["feature_set"]

        if_df = pd.read_parquet(row["if_row_level_path"])
        if_df = _ensure_metadata_columns(if_df, feature_set)
        if_df = _filter_to_calibration_universe(if_df, feature_set)
        if_df["anomaly_flag"] = _detect_anomaly_flag(if_df, "if")
        if_df["framework_label"] = "Isolation Forest"

        dbscan_df = pd.read_parquet(row["dbscan_row_level_path"])
        dbscan_df = _ensure_metadata_columns(dbscan_df, feature_set)
        dbscan_df = _filter_to_calibration_universe(dbscan_df, feature_set)
        dbscan_df["anomaly_flag"] = _detect_anomaly_flag(dbscan_df, "dbscan")
        dbscan_df["framework_label"] = "DBSCAN"

        keep_cols = [
            MODEL_ID_COLUMN,
            TEMPORAL_BUCKET_COLUMN,
            POLICY_GEOGRAPHY_COLUMN,
            ZONE_ID_COLUMN,
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            "anomaly_flag",
            "framework_label",
        ]
        keep_cols = [c for c in keep_cols if c in if_df.columns]
        comparison_frames.append(if_df[keep_cols].assign(feature_set=feature_set).copy())

        keep_cols = [
            MODEL_ID_COLUMN,
            TEMPORAL_BUCKET_COLUMN,
            POLICY_GEOGRAPHY_COLUMN,
            ZONE_ID_COLUMN,
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            "anomaly_flag",
            "framework_label",
        ]
        keep_cols = [c for c in keep_cols if c in dbscan_df.columns]
        comparison_frames.append(dbscan_df[keep_cols].assign(feature_set=feature_set).copy())

    if_dbscan_comparison_row_level_df = (
        pd.concat(comparison_frames, ignore_index=True)
        .sort_values(["feature_set", "framework_label", MODEL_ID_COLUMN])
        .reset_index(drop=True)
    )

    if should_rebuild(REBUILD_IF_DBSCAN_COMPARISONS, IF_DBSCAN_COMPARISON_MANIFEST_PATH):
        if_dbscan_comparison_manifest_df.to_parquet(
            IF_DBSCAN_COMPARISON_MANIFEST_PATH,
            index=False,
        )

    if WRITE_333_OUTPUTS:
        if_dbscan_comparison_row_level_df.to_parquet(
            IF_DBSCAN_COMPARISON_PATH,
            index=False,
        )

display(
    pd.DataFrame(
        [
            {
                "frameworks_loaded": if_dbscan_comparison_row_level_df["framework_label"].nunique(),
                "expected_frameworks": 2,
                "feature_sets_covered": if_dbscan_comparison_row_level_df["feature_set"].nunique(),
                "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
                "rows_loaded": len(if_dbscan_comparison_row_level_df),
                "status": (
                    "pass"
                    if (
                        if_dbscan_comparison_row_level_df["framework_label"].nunique() == 2
                        and if_dbscan_comparison_row_level_df["feature_set"].nunique() == len(MODEL_FEATURE_SET_NAMES)
                    )
                    else "review"
                ),
            }
        ]
    )
)

,frameworks_loaded,expected_frameworks,feature_sets_covered,expected_feature_sets,rows_loaded,status
0,2,2,5,5,500000,pass


Findings\. The comparison manifest resolves one retained Isolation Forest branch and one retained DBSCAN branch for each modality, so the rest of this section can compare inherited framework outputs directly rather than rerunning either method ad hoc\.

### Compare anomaly prevalence across frameworks

This first readout is a volume check: how large is each retained anomaly surface under Isolation Forest versus DBSCAN? Prevalence alone will not tell us whether the frameworks are redundant, but it does tell us whether one framework is operating on a meaningfully larger or smaller anomaly budget before we compare row identities or concentration patterns\.

In [33]:
# ---------------------------------------------------------------------
# Compare anomaly prevalence across retained framework branches
# ---------------------------------------------------------------------

assert "if_dbscan_comparison_row_level_df" in globals(), (
    "if_dbscan_comparison_row_level_df is not in memory. "
    "Please run the retained IF-vs-DBSCAN comparison-surface prep block first."
)

if_dbscan_prevalence_df = (
    if_dbscan_comparison_row_level_df.groupby(
        ["feature_set", "framework_label"],
        dropna=False,
        observed=False,
    )
    .agg(
        rows_evaluated=(MODEL_ID_COLUMN, "size"),
        anomaly_rows=("anomaly_flag", lambda s: s.fillna(False).astype(bool).sum()),
    )
    .reset_index()
    .sort_values(["feature_set", "framework_label"])
    .reset_index(drop=True)
)

if_dbscan_prevalence_df["anomaly_share"] = (
    if_dbscan_prevalence_df["anomaly_rows"] / if_dbscan_prevalence_df["rows_evaluated"]
)

display(
    if_dbscan_prevalence_df.style.format(
        {
            "anomaly_share": "{:.3f}",
        }
    )
)

prevalence_plot_df = if_dbscan_prevalence_df.copy()
prevalence_plot_df["feature_set"] = pd.Categorical(
    prevalence_plot_df["feature_set"],
    categories=["bus", "subway", "taxi", "fhvhv", "multimodal"],
    ordered=True,
)

fig = px.bar(
    prevalence_plot_df.sort_values("feature_set"),
    x="feature_set",
    y="anomaly_share",
    color="framework_label",
    barmode="group",
    text="anomaly_share",
    color_discrete_map={
        "Isolation Forest": BRAND_COLORS["dark_teal"],
        "DBSCAN": BRAND_COLORS["terracotta"],
    },
    labels={
        "feature_set": "Feature set",
        "anomaly_share": "Anomaly share",
        "framework_label": "Framework",
    },
    title="Retained Isolation Forest vs DBSCAN Anomaly Prevalence",
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_layout(
    width=900,
    height=420,
    margin=dict(l=50, r=40, t=70, b=60),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.18,
        xanchor="center",
        x=0.5,
        title_text="",
    ),
)

fig.update_yaxes(tickformat=".3f", rangemode="tozero")
fig.show()

,feature_set,framework_label,rows_evaluated,anomaly_rows,anomaly_share
0,bus,DBSCAN,50000,129,0.003
1,bus,Isolation Forest,50000,250,0.005
2,fhvhv,DBSCAN,50000,175,0.004
3,fhvhv,Isolation Forest,50000,250,0.005
4,multimodal,DBSCAN,50000,117,0.002
5,multimodal,Isolation Forest,50000,250,0.005
6,subway,DBSCAN,50000,304,0.006
7,subway,Isolation Forest,50000,250,0.005
8,taxi,DBSCAN,50000,216,0.004
9,taxi,Isolation Forest,50000,250,0.005


Findings\. With the comparison is restricted to the shared 50,000\-row calibration panel in each modality, the retained Isolation Forest branch behaves exactly as designed: it contributes 250 anomalies in every case, or 0\.005 of the calibration universe\. DBSCAN is more variable on that same panel, ranging from 0\.002 in Multimodal and 0\.003 in Bus up to 0\.006 in Subway\. So even on a matched calibration universe, Isolation Forest remains the tighter and more uniform anomaly budget, while DBSCAN expands or contracts depending on modality\-specific structure\.

Particular attention should be paid to whether Isolation Forest mostly overlaps the retained DBSCAN outputs or whether it preserves a distinct anomaly surface that is worth carrying forward into later cross\-framework comparison\.

### Compare row\-level anomaly overlap and framework\-specific anomaly share

This is the direct complementarity test\. jaccard\_similarity tells us how much the two retained frameworks overlap on the same rows, while shared\_flag\_rows, if\_only\_rows, and dbscan\_only\_rows tell us how the combined anomaly surface breaks apart\. If overlap is low and framework\-specific shares are large, then Isolation Forest is not simply replaying the DBSCAN anomaly surface\.

In [34]:
# ---------------------------------------------------------------------
# Compare row-level anomaly overlap and framework-specific anomaly share
# ---------------------------------------------------------------------

assert "if_dbscan_comparison_row_level_df" in globals(), (
    "if_dbscan_comparison_row_level_df is not in memory. "
    "Please run the retained IF-vs-DBSCAN comparison-surface prep block first."
)

overlap_rows = []

for feature_set in MODEL_FEATURE_SET_NAMES:
    feature_df = if_dbscan_comparison_row_level_df.loc[
        if_dbscan_comparison_row_level_df["feature_set"].eq(feature_set)
    ].copy()

    if_df = feature_df.loc[
        feature_df["framework_label"].eq("Isolation Forest"),
        ["modeling_row_id", "anomaly_flag"],
    ].rename(columns={"anomaly_flag": "if_flag"})

    dbscan_df = feature_df.loc[
        feature_df["framework_label"].eq("DBSCAN"),
        ["modeling_row_id", "anomaly_flag"],
    ].rename(columns={"anomaly_flag": "dbscan_flag"})

    merged_df = if_df.merge(dbscan_df, on="modeling_row_id", how="inner", validate="one_to_one")
    merged_df["if_flag"] = merged_df["if_flag"].fillna(False).astype(bool)
    merged_df["dbscan_flag"] = merged_df["dbscan_flag"].fillna(False).astype(bool)
    merged_df["shared_flag"] = merged_df["if_flag"] & merged_df["dbscan_flag"]
    merged_df["if_only_flag"] = merged_df["if_flag"] & ~merged_df["dbscan_flag"]
    merged_df["dbscan_only_flag"] = ~merged_df["if_flag"] & merged_df["dbscan_flag"]
    merged_df["union_flag"] = merged_df["if_flag"] | merged_df["dbscan_flag"]

    shared_rows = int(merged_df["shared_flag"].sum())
    if_only_rows = int(merged_df["if_only_flag"].sum())
    dbscan_only_rows = int(merged_df["dbscan_only_flag"].sum())
    union_rows = int(merged_df["union_flag"].sum())

    overlap_rows.append(
        {
            "feature_set": feature_set,
            "rows_compared": len(merged_df),
            "if_flag_count": int(merged_df["if_flag"].sum()),
            "dbscan_flag_count": int(merged_df["dbscan_flag"].sum()),
            "shared_flag_rows": shared_rows,
            "if_only_rows": if_only_rows,
            "dbscan_only_rows": dbscan_only_rows,
            "union_flag_rows": union_rows,
            "jaccard_similarity": (shared_rows / union_rows) if union_rows else np.nan,
            "if_only_share_of_union": (if_only_rows / union_rows) if union_rows else np.nan,
            "dbscan_only_share_of_union": (dbscan_only_rows / union_rows) if union_rows else np.nan,
            "shared_share_of_union": (shared_rows / union_rows) if union_rows else np.nan,
        }
    )

if_dbscan_overlap_df = pd.DataFrame(overlap_rows).sort_values("feature_set").reset_index(drop=True)

display(
    if_dbscan_overlap_df.style.format(
        {
            "jaccard_similarity": "{:.3f}",
            "if_only_share_of_union": "{:.3f}",
            "dbscan_only_share_of_union": "{:.3f}",
            "shared_share_of_union": "{:.3f}",
        }
    )
)

composition_plot_df = if_dbscan_overlap_df[
    [
        "feature_set",
        "shared_share_of_union",
        "if_only_share_of_union",
        "dbscan_only_share_of_union",
    ]
].melt(
    id_vars="feature_set",
    var_name="segment",
    value_name="share_of_union",
)

segment_label_map = {
    "shared_share_of_union": "Shared",
    "if_only_share_of_union": "IF only",
    "dbscan_only_share_of_union": "DBSCAN only",
}
composition_plot_df["segment"] = composition_plot_df["segment"].map(segment_label_map)

jaccard_plot_df = if_dbscan_overlap_df.copy()

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Jaccard overlap", "Framework composition of union"),
    horizontal_spacing=0.12,
)

fig.add_trace(
    go.Bar(
        x=jaccard_plot_df["jaccard_similarity"],
        y=jaccard_plot_df["feature_set"],
        orientation="h",
        text=jaccard_plot_df["jaccard_similarity"],
        texttemplate="%{text:.3f}",
        marker_color=BRAND_COLORS["dark_teal"],
        showlegend=False,
    ),
    row=1,
    col=1,
)

for segment, color in [
    ("Shared", "#72B7B2"),
    ("IF only", "#4C78A8"),
    ("DBSCAN only", "#F58518"),
]:
    segment_df = composition_plot_df.loc[composition_plot_df["segment"].eq(segment)].copy()
    fig.add_trace(
        go.Bar(
            x=segment_df["share_of_union"],
            y=segment_df["feature_set"],
            orientation="h",
            name=segment,
            marker_color=color,
            text=segment_df["share_of_union"],
            texttemplate="%{text:.3f}",
        ),
        row=1,
        col=2,
    )

fig.update_layout(
    width=930,
    height=430,
    margin=dict(l=70, r=40, t=70, b=70),
    barmode="stack",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.18,
        xanchor="center",
        x=0.5,
        title_text="",
    ),
    title="Retained Isolation Forest vs DBSCAN Overlap and Framework-Specific Share",
)

fig.update_xaxes(title_text="Jaccard", row=1, col=1, tickformat=".3f")
fig.update_xaxes(title_text="Share of union", row=1, col=2, tickformat=".3f")
fig.update_yaxes(title_text="Feature set", row=1, col=1)
fig.update_yaxes(title_text="Feature set", row=1, col=2)

fig.show()

,feature_set,rows_compared,if_flag_count,dbscan_flag_count,shared_flag_rows,if_only_rows,dbscan_only_rows,union_flag_rows,jaccard_similarity,if_only_share_of_union,dbscan_only_share_of_union,shared_share_of_union
0,bus,50000,250,129,88,162,41,291,0.302,0.557,0.141,0.302
1,fhvhv,50000,250,175,100,150,75,325,0.308,0.462,0.231,0.308
2,multimodal,50000,250,117,77,173,40,290,0.266,0.597,0.138,0.266
3,subway,50000,250,304,84,166,220,470,0.179,0.353,0.468,0.179
4,taxi,50000,250,216,114,136,102,352,0.324,0.386,0.290,0.324


Findings\. On the shared 50,000\-row calibration panels, Isolation Forest and DBSCAN overlap meaningfully but remain far from redundant\. Bus, FHVHV, and Taxi show the strongest agreement, with Jaccard overlap around 0\.30 to 0\.32, while Multimodal is a bit lower at 0\.266 and Subway is the clearest divergence case at 0\.179\. In most modalities, the largest framework\-specific slice of the union is IF\-only rather than DBSCAN\-only, especially in Bus and Multimodal, which means the retained IF branch is not just extracting a tiny shared core from DBSCAN\. Subway is the exception: there DBSCAN\-only rows make up 46\.8% of the union, so DBSCAN is clearly seeing a broader anomaly field than IF on that calibration panel\. Overall, the practical takeaway is that the two frameworks share a real core but still preserve distinct anomaly surfaces, with the sharpest disagreement concentrated in Subway\.

### Compare concentration across local operating slices

Overlap tells us whether the two frameworks flag the same rows; this block tells us whether they concentrate those rows into the same kinds of local operating regimes\. min\_slice\_anomaly\_share, median\_slice\_anomaly\_share, and max\_slice\_anomaly\_share summarize anomaly concentration across the 30 shared temporal\_bucket x policy\_geography\_class slices\. If one framework has much higher maxima or medians, it is producing sharper local anomaly pockets rather than a flatter anomaly surface\.

In [35]:
# ---------------------------------------------------------------------
# Compare Isolation Forest and DBSCAN concentration across local slices
# ---------------------------------------------------------------------

assert "if_dbscan_comparison_row_level_df" in globals(), (
    "if_dbscan_comparison_row_level_df is not in memory. "
    "Please run the retained IF-vs-DBSCAN comparison-surface prep block first."
)

slice_rows = []

for (feature_set, framework_label), df in (
    if_dbscan_comparison_row_level_df.groupby(
        ["feature_set", "framework_label"],
        dropna=False,
        observed=False,
    )
):
    slice_summary_df = (
        df.groupby(
            ["temporal_bucket", "policy_geography_class"],
            dropna=False,
            observed=False,
        )["anomaly_flag"]
        .mean()
        .reset_index(name="slice_anomaly_share")
    )

    slice_rows.append(
        {
            "feature_set": feature_set,
            "framework_label": framework_label,
            "comparison_slices": len(slice_summary_df),
            "min_slice_anomaly_share": float(slice_summary_df["slice_anomaly_share"].min()),
            "median_slice_anomaly_share": float(slice_summary_df["slice_anomaly_share"].median()),
            "max_slice_anomaly_share": float(slice_summary_df["slice_anomaly_share"].max()),
        }
    )

if_dbscan_slice_concentration_df = (
    pd.DataFrame(slice_rows)
    .sort_values(["feature_set", "framework_label"])
    .reset_index(drop=True)
)

display(
    if_dbscan_slice_concentration_df.style.format(
        {
            "min_slice_anomaly_share": "{:.3f}",
            "median_slice_anomaly_share": "{:.3f}",
            "max_slice_anomaly_share": "{:.3f}",
        }
    )
)

slice_plot_rows = []
for _, row in if_dbscan_slice_concentration_df.iterrows():
    slice_plot_rows.extend(
        [
            {
                "feature_set": row["feature_set"],
                "framework_label": row["framework_label"],
                "metric_label": "Median slice share",
                "metric_value": row["median_slice_anomaly_share"],
            },
            {
                "feature_set": row["feature_set"],
                "framework_label": row["framework_label"],
                "metric_label": "Max slice share",
                "metric_value": row["max_slice_anomaly_share"],
            },
        ]
    )

slice_plot_df = pd.DataFrame(slice_plot_rows)

fig = px.bar(
    slice_plot_df,
    x="feature_set",
    y="metric_value",
    color="framework_label",
    barmode="group",
    facet_col="metric_label",
    category_orders={
        "feature_set": ["bus", "subway", "taxi", "fhvhv", "multimodal"],
        "framework_label": ["Isolation Forest", "DBSCAN"],
        "metric_label": ["Median slice share", "Max slice share"],
    },
    color_discrete_map={
        "Isolation Forest": BRAND_COLORS["dark_teal"],
        "DBSCAN": BRAND_COLORS["terracotta"],
    },
    labels={
        "feature_set": "Feature set",
        "metric_value": "Anomaly share",
        "framework_label": "Framework",
        "metric_label": "",
    },
    title="Retained Isolation Forest vs DBSCAN Slice Concentration",
)

fig.update_traces(
    texttemplate="%{y:.3f}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_layout(
    width=930,
    height=440,
    margin=dict(l=60, r=40, t=80, b=60),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.18,
        xanchor="center",
        x=0.5,
        title_text="",
    ),
)

fig.update_yaxes(tickformat=".3f", rangemode="tozero")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

,feature_set,framework_label,comparison_slices,min_slice_anomaly_share,median_slice_anomaly_share,max_slice_anomaly_share
0,bus,DBSCAN,30,0.000,0.001,0.018
1,bus,Isolation Forest,30,0.000,0.001,0.074
2,fhvhv,DBSCAN,30,0.000,0.002,0.010
3,fhvhv,Isolation Forest,30,0.000,0.003,0.037
4,multimodal,DBSCAN,30,0.000,0.002,0.014
5,multimodal,Isolation Forest,30,0.000,0.003,0.039
6,subway,DBSCAN,30,0.000,0.003,0.058
7,subway,Isolation Forest,30,0.000,0.000,0.077
8,taxi,DBSCAN,30,0.000,0.006,0.028
9,taxi,Isolation Forest,30,0.000,0.005,0.056


Findings\. On the shared 30\-slice calibration universe, the typical slice\-level anomaly burden is fairly similar across the two frameworks: median slice shares stay near zero to low single\-digit percentages in every modality, with Taxi the highest for both\. The sharper difference appears in the most anomaly\-heavy slice\. Isolation Forest reaches higher max slice shares than DBSCAN in Bus, Subway, Taxi, FHVHV, and Multimodal, peaking at 0\.077 in Subway and 0\.074 in Bus versus 0\.058 and 0\.018 for DBSCAN\. So the practical takeaway is that, on a matched calibration panel, Isolation Forest is not more diffuse than DBSCAN\. If anything, it tends to load its smaller anomaly budget into sharper local slice hotspots\.

In plain language, when both frameworks are restricted to the same calibration panel, DBSCAN does not look like the sharper local\-regime detector\. Isolation Forest still uses a smaller, fixed anomaly budget, but it often concentrates that budget into hotter individual slices, while DBSCAN spreads its anomalies more evenly across the 30 local operating contexts\. So the practical takeaway is that Isolation Forest is not merely a weaker copy of DBSCAN here; it is a more selective framework that often produces sharper slice\-level hotspots even while flagging fewer rows overall\.

### Compare spatial concentration across zones

This last check asks whether the retained frameworks pile anomalies into a small number of taxi zones or whether they spread them more broadly across space\. top\_zone\_share shows how much of each framework’s anomaly surface lives in its single most dominant zone, top5\_zone\_share broadens that to the top five zones, and distinct\_anomaly\_zones tells us how spatially broad the anomaly surface is overall\.

In [36]:
# ---------------------------------------------------------------------
# Compare Isolation Forest and DBSCAN spatial concentration across zones
# ---------------------------------------------------------------------

assert "if_dbscan_comparison_row_level_df" in globals(), (
    "if_dbscan_comparison_row_level_df is not in memory. "
    "Please run the retained IF-vs-DBSCAN comparison-surface prep block first."
)

zone_rows = []

for (feature_set, framework_label), df in (
    if_dbscan_comparison_row_level_df.groupby(
        ["feature_set", "framework_label"],
        dropna=False,
        observed=False,
    )
):
    if ZONE_ID_COLUMN not in df.columns:
        metadata_df = pd.read_parquet(ROW_METADATA_PATHS_BY_SET[feature_set])[
            [MODEL_ID_COLUMN, ZONE_ID_COLUMN, BOROUGH_COLUMN, ZONE_COLUMN]
        ].drop_duplicates(subset=[MODEL_ID_COLUMN])
        df = df.merge(metadata_df, on=MODEL_ID_COLUMN, how="left")

    anomaly_zone_df = (
        df.loc[df["anomaly_flag"].fillna(False).astype(bool)]
        .groupby([ZONE_ID_COLUMN, BOROUGH_COLUMN, ZONE_COLUMN], dropna=False, observed=False)
        .size()
        .reset_index(name="anomaly_rows")
        .sort_values("anomaly_rows", ascending=False)
        .reset_index(drop=True)
    )

    total_anomaly_rows = int(anomaly_zone_df["anomaly_rows"].sum())

    if total_anomaly_rows == 0:
        top_zone_share = np.nan
        top5_zone_share = np.nan
        distinct_anomaly_zones = 0
    else:
        top_zone_share = anomaly_zone_df["anomaly_rows"].iloc[0] / total_anomaly_rows
        top5_zone_share = anomaly_zone_df["anomaly_rows"].head(5).sum() / total_anomaly_rows
        distinct_anomaly_zones = anomaly_zone_df[ZONE_ID_COLUMN].nunique()

    zone_rows.append(
        {
            "feature_set": feature_set,
            "framework_label": framework_label,
            "anomaly_rows": total_anomaly_rows,
            "top_zone_share": top_zone_share,
            "top5_zone_share": top5_zone_share,
            "distinct_anomaly_zones": distinct_anomaly_zones,
        }
    )

if_dbscan_zone_concentration_df = (
    pd.DataFrame(zone_rows)
    .sort_values(["feature_set", "framework_label"])
    .reset_index(drop=True)
)

display(
    if_dbscan_zone_concentration_df.style.format(
        {
            "top_zone_share": "{:.3f}",
            "top5_zone_share": "{:.3f}",
        }
    )
)

# ---------------------------------------------------------------------
# Visualize Isolation Forest and DBSCAN spatial concentration across zones
# ---------------------------------------------------------------------

zone_plot_df = if_dbscan_zone_concentration_df.copy()
zone_plot_df["feature_set"] = pd.Categorical(
    zone_plot_df["feature_set"],
    categories=["bus", "subway", "taxi", "fhvhv", "multimodal"],
    ordered=True,
)
zone_plot_df = zone_plot_df.sort_values(["feature_set", "framework_label"]).reset_index(drop=True)

framework_color_map = IF_FRAMEWORK_COLOR_MAP

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{"colspan": 2}, None],
        [{}, {}],
    ],
    subplot_titles=(
        "Distinct anomaly zones",
        "Top zone share",
        "Top-5 zone share",
    ),
    horizontal_spacing=0.16,
    vertical_spacing=0.22,
    row_heights=[0.42, 0.58],
)

for framework_label in ["Isolation Forest", "DBSCAN"]:
    framework_df = zone_plot_df.loc[
        zone_plot_df["framework_label"].eq(framework_label)
    ].copy()

    fig.add_trace(
        go.Bar(
            x=framework_df["feature_set"],
            y=framework_df["distinct_anomaly_zones"],
            name=framework_label,
            marker_color=framework_color_map[framework_label],
            text=framework_df["distinct_anomaly_zones"],
            texttemplate="%{text:.0f}",
            textposition="outside",
            cliponaxis=False,
            legendgroup=framework_label,
            showlegend=True,
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Bar(
            x=framework_df["feature_set"],
            y=framework_df["top_zone_share"],
            name=framework_label,
            marker_color=framework_color_map[framework_label],
            text=framework_df["top_zone_share"],
            texttemplate="%{text:.3f}",
            textposition="outside",
            cliponaxis=False,
            legendgroup=framework_label,
            showlegend=False,
        ),
        row=2,
        col=1,
    )

    fig.add_trace(
        go.Bar(
            x=framework_df["feature_set"],
            y=framework_df["top5_zone_share"],
            name=framework_label,
            marker_color=framework_color_map[framework_label],
            text=framework_df["top5_zone_share"],
            texttemplate="%{text:.3f}",
            textposition="outside",
            cliponaxis=False,
            legendgroup=framework_label,
            showlegend=False,
        ),
        row=2,
        col=2,
    )

fig.update_layout(
    title="Retained Isolation Forest vs DBSCAN Zone Concentration",
    width=930,
    height=520,
    barmode="group",
    margin=dict(l=60, r=40, t=80, b=70),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.10,
        xanchor="center",
        x=0.5,
        title_text="",
    ),
)

fig.update_xaxes(title_text="Feature set", row=1, col=1)
fig.update_xaxes(title_text="Feature set", row=2, col=1)
fig.update_xaxes(title_text="Feature set", row=2, col=2)

fig.update_yaxes(title_text="Zone count", row=1, col=1)
fig.update_yaxes(title_text="Share of anomaly rows", tickformat=".3f", rangemode="tozero", row=2, col=1)
fig.update_yaxes(title_text="Share of anomaly rows", tickformat=".3f", rangemode="tozero", row=2, col=2)

fig.update_yaxes(
    title_text="Zone count",
    range=[0, 230],
    row=1,
    col=1,
)

fig.show()

,feature_set,framework_label,anomaly_rows,top_zone_share,top5_zone_share,distinct_anomaly_zones
0,bus,DBSCAN,129,0.248,0.612,36
1,bus,Isolation Forest,250,0.380,0.848,23
2,fhvhv,DBSCAN,175,0.137,0.520,40
3,fhvhv,Isolation Forest,250,0.168,0.464,59
4,multimodal,DBSCAN,117,0.111,0.350,50
5,multimodal,Isolation Forest,250,0.288,0.628,43
6,subway,DBSCAN,304,0.128,0.414,64
7,subway,Isolation Forest,250,0.216,0.824,16
8,taxi,DBSCAN,216,0.111,0.310,85
9,taxi,Isolation Forest,250,0.220,0.512,47


Findings\. This spatial\-concentration check still shows Isolation Forest as the more spatially selective framework in most modalities\. Its anomaly surfaces are more tightly concentrated into a handful of zones in Bus, Subway, Taxi, and Multimodal: for example, the top five zones absorb 84\.8% of Bus IF anomalies versus 61\.2% for DBSCAN, 82\.4% versus 41\.4% in Subway, and 62\.8% versus 35\.0% in Multimodal\. Isolation Forest also touches fewer distinct zones in those same modalities, especially in Subway, where it spans just 16 zones versus 64 for DBSCAN\. FHVHV is the main exception: DBSCAN is actually slightly more concentrated in its top five zones there, while Isolation Forest spreads across more distinct zones\. So the broad read still stands, but with one nuance: Isolation Forest is usually the tighter hotspot\-oriented surface, though that advantage is weaker in FHVHV\.

In plain language, Isolation Forest is generally not scattering its anomalies broadly across the city\. In most modalities it returns to a smaller set of hotspot zones more aggressively than DBSCAN does, even on the matched calibration panel\. DBSCAN still spreads its anomaly mass more broadly across space, while Isolation Forest usually extracts a tighter hotspot core\. The main caveat is FHVHV, where the difference is less clean and DBSCAN is not obviously the broader spatial surface\.

> 🎯 Decision\. Isolation Forest still survives into downstream framework comparison because it contributes a meaningfully different anomaly slice than DBSCAN\. On the shared calibration panel, that difference shows up less as a weaker version of DBSCAN and more as a generally tighter hotspot\-oriented surface\. DBSCAN contributes broader spatial coverage, while Isolation Forest usually contributes a smaller and more spatially concentrated anomaly core, with FHVHV as the least decisive modality\.

### Section Summary

> This section showed that the retained Isolation Forest branches do not simply recreate the retained DBSCAN outputs\. On the shared calibration panel, Isolation Forest uses a smaller fixed anomaly budget, overlaps only partially with DBSCAN at the row level, and in most modalities produces a more spatially concentrated hotspot surface\. The slice\-level comparison also suggests that Isolation Forest is not merely a weaker copy of DBSCAN: even while flagging fewer rows, it often concentrates those rows into sharper local hotspots\. The practical takeaway is that Isolation Forest is worth carrying forward, not because it replaces DBSCAN, but because it contributes a distinct and more selective anomaly perspective for later cross\-framework comparison\.

## 3\.3\.3\.5 Generate and Export Candidate Isolation Forest Anomalies

This part promotes Isolation Forest from a calibration\-stage candidate into a full\-universe anomaly branch\. We are no longer asking how the model behaves on the shared 50K calibration panel; we are applying the retained PCA\-based Isolation Forest configuration to the full modeling universe for each modality, caching the prepared full\-universe panels, and exporting one metadata\-enriched row\-level anomaly surface per branch\.

The export is designed to be rerunnable and crash\-tolerant\. Each modality writes its own temporary full\-universe row\-level parquet first, and only then contributes to the final export manifest\. That way, if the notebook is interrupted mid\-run, completed modalities can be loaded from disk rather than recomputed\.

Before writing the final row\-level anomaly files, define one small helper layer that makes the export safer and easier to rerun\. The key idea is simple: for each modality, load the full retained PCA branch, cache a prepared modeling panel on disk, score the full universe with the retained Isolation Forest configuration, and skip finished modality exports when they already exist and the rebuild toggle remains off\.

In [37]:
# ---------------------------------------------------------------------
# Define helper logic for full-universe final Isolation Forest export
# ---------------------------------------------------------------------

def get_if_final_prepared_panel_path(feature_set: str) -> Path:
    return IF_FINAL_PREPARED_PANELS_DIR / f"{feature_set}_if_prepared_panel.parquet"


def get_if_final_temp_output_path(feature_set: str) -> Path:
    return IF_FINAL_ROW_LEVEL_TEMP_DIR / f"{feature_set}_if_candidate_anomalies.parquet"


def get_if_final_output_path(feature_set: str) -> Path:
    return IF_FINAL_ROW_LEVEL_DIR / f"{feature_set}_if_candidate_anomalies.parquet"


def build_if_full_universe_prepared_panel(
    feature_set: str,
    *,
    rebuild: bool = False,
) -> Path:
    """
    Build one full-universe prepared PCA panel per modality for final
    Isolation Forest export.

    This intentionally uses the full modeling metadata universe rather than the
    50K calibration panel.
    """
    prepared_panel_path = get_if_final_prepared_panel_path(feature_set)

    if prepared_panel_path.exists() and not rebuild:
        return prepared_panel_path

    metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set]
    assert metadata_path.exists(), f"Missing full row metadata for {feature_set}: {metadata_path}"

    if feature_set == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
        pre_pca_path = get_representation_path(
            feature_set="taxi",
            representation_name="pca_80pct",
            row_metadata=pd.DataFrame({PRE_POST_CP_COLUMN: ["pre_cp"]}),
        )
        post_pca_path = get_representation_path(
            feature_set="taxi",
            representation_name="pca_80pct",
            row_metadata=pd.DataFrame({PRE_POST_CP_COLUMN: ["post_cp"]}),
        )

        assert pre_pca_path.exists(), f"Missing Taxi pre-CP PCA path: {pre_pca_path}"
        assert post_pca_path.exists(), f"Missing Taxi post-CP PCA path: {post_pca_path}"

        representation_columns = get_numeric_representation_columns(
            feature_set,
            "pca_80pct",
            pre_pca_path,
        )

        pca_sql = f"""
            SELECT {", ".join(duckdb_identifier(col) for col in [MODEL_ID_COLUMN, *representation_columns])}
            FROM read_parquet('{sql_path(pre_pca_path)}')
            UNION ALL
            SELECT {", ".join(duckdb_identifier(col) for col in [MODEL_ID_COLUMN, *representation_columns])}
            FROM read_parquet('{sql_path(post_pca_path)}')
        """
    else:
        pca_path = get_representation_path(
            feature_set=feature_set,
            representation_name="pca_80pct",
            row_metadata=None,
        )
        assert pca_path.exists(), f"Missing PCA path for {feature_set}: {pca_path}"

        representation_columns = get_numeric_representation_columns(
            feature_set,
            "pca_80pct",
            pca_path,
        )

        pca_sql = f"""
            SELECT {", ".join(duckdb_identifier(col) for col in [MODEL_ID_COLUMN, *representation_columns])}
            FROM read_parquet('{sql_path(pca_path)}')
        """

    with duckdb.connect() as con:
        con.execute(f"""
            COPY (
                WITH metadata_rows AS (
                    SELECT
                        {", ".join(duckdb_identifier(col) for col in DEFAULT_METADATA_COLUMNS)}
                    FROM read_parquet('{sql_path(metadata_path)}')
                ),
                pca_rows AS (
                    {pca_sql}
                )
                SELECT
                    m.{duckdb_identifier(MODEL_ID_COLUMN)},
                    m.{duckdb_identifier(DATE_COLUMN)},
                    m.{duckdb_identifier(TEMPORAL_BUCKET_COLUMN)},
                    m.{duckdb_identifier(PRE_POST_CP_COLUMN)},
                    m.{duckdb_identifier(ZONE_ID_COLUMN)},
                    m.{duckdb_identifier(ZONE_COLUMN)},
                    m.{duckdb_identifier(BOROUGH_COLUMN)},
                    m.{duckdb_identifier(POLICY_GEOGRAPHY_COLUMN)},
                    {", ".join(
                        f"CAST(p.{duckdb_identifier(col)} AS FLOAT) AS {duckdb_identifier(col)}"
                        for col in representation_columns
                    )}
                FROM metadata_rows AS m
                INNER JOIN pca_rows AS p
                    ON m.{duckdb_identifier(MODEL_ID_COLUMN)} = p.{duckdb_identifier(MODEL_ID_COLUMN)}
            )
            TO '{sql_path(prepared_panel_path)}' (FORMAT PARQUET)
        """)

    assert prepared_panel_path.exists(), (
        f"Failed to build final IF prepared panel for {feature_set}: {prepared_panel_path}"
    )

    return prepared_panel_path


def load_final_if_export_manifest() -> pd.DataFrame:
    if "if_final_row_level_manifest_df" in globals():
        return if_final_row_level_manifest_df.copy()

    assert IF_ANOMALY_EXPORT_MANIFEST_PATH.exists(), (
        f"Missing final IF export manifest: {IF_ANOMALY_EXPORT_MANIFEST_PATH}"
    )
    return pd.read_csv(IF_ANOMALY_EXPORT_MANIFEST_PATH)


def load_final_if_export_branch(feature_set: str) -> pd.DataFrame:
    manifest_df = load_final_if_export_manifest()

    path_column_candidates = [
        "row_level_output_path",
        "row_level_path",
        "output_path",
        "path",
    ]
    manifest_path_col = next(
        (col for col in path_column_candidates if col in manifest_df.columns),
        None,
    )

    candidate_paths = []

    if manifest_path_col is not None:
        matched_manifest_rows = manifest_df.loc[
            manifest_df["feature_set"].eq(feature_set)
        ].copy()
        if len(matched_manifest_rows) == 1:
            candidate_paths.append(Path(matched_manifest_rows.iloc[0][manifest_path_col]))

    candidate_paths.extend(
        [
            IF_FINAL_ROW_LEVEL_DIR / f"{feature_set}_if_candidate_anomalies.parquet",
            IF_FINAL_ROW_LEVEL_DIR / f"{feature_set}_isolation_forest_candidate_anomalies.parquet",
        ]
    )

    branch_path = next((path for path in candidate_paths if path.exists()), None)
    assert branch_path is not None, (
        f"Could not locate final IF export branch for {feature_set}. "
        f"Tried: {[str(path) for path in candidate_paths]}"
    )

    return pd.read_parquet(branch_path)


def detect_final_if_anomaly_flag(df: pd.DataFrame) -> pd.Series:
    candidate_cols = [
        "is_if_anomaly",
        "is_anomaly",
        "anomaly_flag",
        "anomaly_like_flag",
        "if_anomaly_flag",
    ]
    for col in candidate_cols:
        if col in df.columns:
            return df[col].fillna(False).astype(bool)

    score_cols = [
        "decision_function_score",
        "if_decision_function_score",
        "anomaly_score",
    ]
    for col in score_cols:
        if col in df.columns:
            return df[col].lt(0)

    raise AssertionError(
        f"Could not identify a final Isolation Forest anomaly flag column. "
        f"Available columns: {df.columns.tolist()}"
    )


def ensure_if_final_map_metadata(df: pd.DataFrame, feature_set: str) -> pd.DataFrame:
    required_cols = [
        MODEL_ID_COLUMN,
        ZONE_ID_COLUMN,
        BOROUGH_COLUMN,
        ZONE_COLUMN,
        TEMPORAL_BUCKET_COLUMN,
        POLICY_GEOGRAPHY_COLUMN,
        DATE_COLUMN,
        PRE_POST_CP_COLUMN,
    ]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if not missing_cols:
        return df

    row_metadata_df = pd.read_parquet(
        ROW_METADATA_PATHS_BY_SET[feature_set],
        columns=[MODEL_ID_COLUMN, *[col for col in missing_cols if col != MODEL_ID_COLUMN]],
    ).drop_duplicates(subset=[MODEL_ID_COLUMN])

    return df.merge(
        row_metadata_df,
        on=MODEL_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )

Now write the final row\-level anomaly outputs for the retained branches only\. These exports should preserve anomaly flags, anomaly scores, retained framework settings, and the row\-level metadata needed for downstream framework comparison\. The important distinction here is scale: this export should score the full retained PCA universe for each modality, not the smaller calibration panel used earlier in the notebook\.

In [38]:
# ---------------------------------------------------------------------
# Generate and export candidate Isolation Forest anomalies
# Full-universe, rerunnable at the modality level
# ---------------------------------------------------------------------

assert "if_final_contamination_selection_df" in globals(), (
    "if_final_contamination_selection_df is not in memory. "
    "Please run the final Isolation Forest contamination-selection block first."
)

assert "build_if_full_universe_prepared_panel" in globals(), (
    "build_if_full_universe_prepared_panel is not in memory. "
    "Please run the full-universe IF helper block first."
)

retained_if_export_settings_df = (
    if_final_contamination_selection_df.copy()
    .sort_values(["feature_set"])
    .reset_index(drop=True)
)

final_if_manifest_records = []
overall_start = perf_counter()

for export_index, export_row in enumerate(
    retained_if_export_settings_df.itertuples(index=False),
    start=1,
):
    feature_start = perf_counter()

    feature_set = export_row.feature_set
    representation_name = export_row.representation_name
    contamination = float(export_row.contamination)

    assert representation_name == "pca_80pct", (
        f"Expected retained IF representation to be pca_80pct for final export, got {representation_name}"
    )

    prepared_panel_path = build_if_full_universe_prepared_panel(
        feature_set,
        rebuild=REBUILD_IF_FINAL_OUTPUTS,
    )

    temp_output_path = get_if_final_temp_output_path(feature_set)
    final_output_path = get_if_final_output_path(feature_set)

    if REBUILD_IF_FINAL_OUTPUTS:
        if temp_output_path.exists():
            temp_output_path.unlink()
        if final_output_path.exists():
            final_output_path.unlink()

    if final_output_path.exists() and not REBUILD_IF_FINAL_OUTPUTS:
        branch_df = pd.read_parquet(final_output_path)

        anomaly_flag = detect_final_if_anomaly_flag(branch_df)
        comparison_groups_present = (
            branch_df[[TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN]]
            .drop_duplicates()
            .shape[0]
            if TEMPORAL_BUCKET_COLUMN in branch_df.columns and POLICY_GEOGRAPHY_COLUMN in branch_df.columns
            else np.nan
        )

        final_if_manifest_records.append(
            {
                "feature_set": feature_set,
                "representation_name": representation_name,
                "contamination": contamination,
                "row_level_output_path": str(final_output_path),
                "rows_written": int(len(branch_df)),
                "anomaly_rows": int(anomaly_flag.sum()),
                "comparison_groups_present": comparison_groups_present,
                "status": "loaded_existing",
            }
        )

        print(
            f"[{export_index}/{len(retained_if_export_settings_df)}] "
            f"Loaded existing final IF export for {feature_set} "
            f"in {perf_counter() - feature_start:,.1f}s."
        )
        continue

    print(
        f"[{export_index}/{len(retained_if_export_settings_df)}] "
        f"Generating full-universe IF anomalies for {feature_set}..."
    )

    with duckdb.connect() as con:
        prepared_panel_cols_df = con.execute(f"""
            DESCRIBE SELECT * FROM read_parquet('{sql_path(prepared_panel_path)}')
        """).fetchdf()

    prepared_panel_columns = prepared_panel_cols_df["column_name"].tolist()
    representation_columns = [
        col for col in prepared_panel_columns if str(col).startswith("PC")
    ]

    assert representation_columns, (
        f"No PCA columns found in prepared IF panel for {feature_set}: {prepared_panel_path}"
    )

    load_start = perf_counter()
    prepared_df = pd.read_parquet(prepared_panel_path)
    X = prepared_df[representation_columns].to_numpy(dtype=np.float32, copy=True)

    print(
        f"  Loaded {feature_set}: {len(prepared_df):,} rows and "
        f"{len(representation_columns):,} PCA columns in {perf_counter() - load_start:,.1f}s."
    )

    fit_start = perf_counter()
    if_model = IsolationForest(
        n_estimators=ISOLATION_FOREST_N_ESTIMATORS,
        contamination=contamination,
        max_samples=ISOLATION_FOREST_MAX_SAMPLES,
        max_features=ISOLATION_FOREST_MAX_FEATURES,
        random_state=ISOLATION_FOREST_RANDOM_STATE,
        n_jobs=-1,
    )
    if_model.fit(X)

    predicted_labels = if_model.predict(X)
    decision_scores = if_model.decision_function(X)

    prepared_df["feature_set"] = feature_set
    prepared_df["representation_name"] = representation_name
    prepared_df["contamination"] = contamination
    prepared_df["if_prediction"] = predicted_labels.astype(np.int16, copy=False)
    prepared_df["decision_function_score"] = decision_scores.astype(np.float32, copy=False)
    prepared_df["is_if_anomaly"] = prepared_df["if_prediction"].eq(-1)

    print(
        f"  Fit and scored {feature_set} in {perf_counter() - fit_start:,.1f}s "
        f"with {int(prepared_df['is_if_anomaly'].sum()):,} anomaly rows."
    )

    write_start = perf_counter()
    prepared_df.to_parquet(temp_output_path, index=False)
    shutil.copy2(temp_output_path, final_output_path)

    comparison_groups_present = (
        prepared_df[[TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN]]
        .drop_duplicates()
        .shape[0]
    )

    final_if_manifest_records.append(
        {
            "feature_set": feature_set,
            "representation_name": representation_name,
            "contamination": contamination,
            "row_level_output_path": str(final_output_path),
            "rows_written": int(len(prepared_df)),
            "anomaly_rows": int(prepared_df["is_if_anomaly"].sum()),
            "comparison_groups_present": comparison_groups_present,
            "status": "written",
        }
    )

    print(
        f"  Wrote {feature_set} final IF export in {perf_counter() - write_start:,.1f}s | "
        f"feature_total={perf_counter() - feature_start:,.1f}s"
    )

    del prepared_df
    del X
    del if_model
    del predicted_labels
    del decision_scores
    gc.collect()

if_final_row_level_manifest_df = (
    pd.DataFrame(final_if_manifest_records)
    .sort_values(["feature_set"])
    .reset_index(drop=True)
)

if WRITE_333_OUTPUTS:
    if_final_row_level_manifest_df.to_csv(
        IF_ANOMALY_EXPORT_MANIFEST_PATH,
        index=False,
    )

print(f"Completed full-universe IF export in {perf_counter() - overall_start:,.1f}s.")

display(if_final_row_level_manifest_df)

[1/5] Loaded existing final IF export for bus in 1.5s.
[2/5] Loaded existing final IF export for fhvhv in 2.0s.
[3/5] Loaded existing final IF export for multimodal in 1.2s.
[4/5] Loaded existing final IF export for subway in 0.7s.
[5/5] Loaded existing final IF export for taxi in 0.8s.
Completed full-universe IF export in 6.6s.


,feature_set,representation_name,contamination,row_level_output_path,rows_written,anomaly_rows,comparison_groups_present,status
0,bus,pca_80pct,0.005,/datasets/_deepnote_work/pipeline_data/3.3.3.final_tables/isolation_forest_candidate_anomaly_row_level_parts/bus_if_candidate_anomalies.parquet,1472947,7365,30,loaded_existing
1,fhvhv,pca_80pct,0.005,/datasets/_deepnote_work/pipeline_data/3.3.3.final_tables/isolation_forest_candidate_anomaly_row_level_parts/fhvhv_if_candidate_anomalies.parquet,1541800,7709,30,loaded_existing
2,multimodal,pca_80pct,0.005,/datasets/_deepnote_work/pipeline_data/3.3.3.final_tables/isolation_forest_candidate_anomaly_row_level_parts/multimodal_if_candidate_anomalies.parquet,1541800,7709,30,loaded_existing
3,subway,pca_80pct,0.005,/datasets/_deepnote_work/pipeline_data/3.3.3.final_tables/isolation_forest_candidate_anomaly_row_level_parts/subway_if_candidate_anomalies.parquet,911455,4554,30,loaded_existing
4,taxi,pca_80pct,0.005,/datasets/_deepnote_work/pipeline_data/3.3.3.final_tables/isolation_forest_candidate_anomaly_row_level_parts/taxi_if_candidate_anomalies.parquet,1541800,7709,30,loaded_existing


Findings\. The final retained Isolation Forest exports cover the full modeling universe for all five modalities, with one clean row\-level output per branch and complete coverage of all 30 comparison groups in every case\. Bus writes 1,472,947 rows with 7,365 anomalies, Subway writes 911,455 rows with 4,554 anomalies, and Taxi, FHVHV, and Multimodal each write 1,541,800 rows with 7,709 anomalies\. Because the retained contamination setting is 0\.005 for every branch, the anomaly volumes are sparse and highly consistent across comparable row universes\. So the practical takeaway is that Isolation Forest is contributing a full\-universe, modality\-complete anomaly surface that is selective enough to be interpretable but broad enough to carry forward into downstream framework comparison\.

### Validate the final Isolation Forest export handoff

This last block is a handoff check\. Before leaving the notebook, verify that the retained Isolation Forest package is complete: one selected configuration per modality, one final row\-level export per modality, one manifest entry per modality, and the expected row\-level fields needed for downstream synthesis and comparison\.

In [39]:
# ---------------------------------------------------------------------
# Validate the final Isolation Forest export handoff
# ---------------------------------------------------------------------

assert "if_final_contamination_selection_df" in globals(), (
    "if_final_contamination_selection_df is not in memory. "
    "Please run the final contamination-selection block first."
)
assert "if_final_row_level_manifest_df" in globals(), (
    "if_final_row_level_manifest_df is not in memory. "
    "Please run the final row-level export block first."
)

required_export_columns = [
    MODEL_ID_COLUMN,
    DATE_COLUMN,
    TEMPORAL_BUCKET_COLUMN,
    PRE_POST_CP_COLUMN,
    ZONE_ID_COLUMN,
    ZONE_COLUMN,
    BOROUGH_COLUMN,
    POLICY_GEOGRAPHY_COLUMN,
    "is_if_anomaly",
    "decision_function_score",
    "if_prediction",
    "feature_set",
    "representation_name",
    "contamination",
]

if_export_file_check_records = []

for manifest_row in if_final_row_level_manifest_df.itertuples(index=False):
    export_path = Path(manifest_row.row_level_output_path)
    assert export_path.exists(), f"Missing final IF export file: {export_path}"

    export_df = pd.read_parquet(export_path)
    missing_columns = [col for col in required_export_columns if col not in export_df.columns]

    if_export_file_check_records.append(
        {
            "feature_set": manifest_row.feature_set,
            "rows_written": int(len(export_df)),
            "anomaly_rows": int(export_df["is_if_anomaly"].fillna(False).sum()),
            "missing_required_columns": ", ".join(missing_columns) if missing_columns else "",
            "status": (
                "pass"
                if (len(export_df) > 50000 and not missing_columns)
                else "review"
            ),
        }
    )

if_export_file_check_df = (
    pd.DataFrame(if_export_file_check_records)
    .sort_values(["feature_set"])
    .reset_index(drop=True)
)

if_export_validation_df = pd.DataFrame(
    [
        {
            "retained_feature_sets": if_final_contamination_selection_df["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "final_configuration_rows": len(if_final_contamination_selection_df),
            "manifest_rows": len(if_final_row_level_manifest_df),
            "all_export_paths_exist": if_final_row_level_manifest_df["row_level_output_path"]
            .map(lambda x: Path(x).exists())
            .all()
            if WRITE_333_OUTPUTS
            else True,
            "all_file_checks_pass": if_export_file_check_df["status"].eq("pass").all(),
            "status": (
                "pass"
                if (
                    if_final_contamination_selection_df["feature_set"].nunique()
                    == len(MODEL_FEATURE_SET_NAMES)
                    and len(if_final_contamination_selection_df) == len(MODEL_FEATURE_SET_NAMES)
                    and len(if_final_row_level_manifest_df) == len(MODEL_FEATURE_SET_NAMES)
                    and if_export_file_check_df["status"].eq("pass").all()
                )
                else "review"
            ),
        }
    ]
)

display(if_export_file_check_df)
display(if_export_validation_df)

assert if_export_validation_df["status"].eq("pass").all(), (
    "Isolation Forest export handoff is incomplete."
)

,feature_set,rows_written,anomaly_rows,missing_required_columns,status
0,bus,1472947,7365,,pass
1,fhvhv,1541800,7709,,pass
2,multimodal,1541800,7709,,pass
3,subway,911455,4554,,pass
4,taxi,1541800,7709,,pass


,retained_feature_sets,expected_feature_sets,final_configuration_rows,manifest_rows,all_export_paths_exist,all_file_checks_pass,status
0,5,5,5,5,True,True,pass


Findings\. The final Isolation Forest handoff is complete and internally consistent\. All five modalities are present in the retained configuration layer and the export manifest, every final export path resolves successfully on disk, and every row\-level export passes the required schema check with no missing downstream columns\. The exported row counts and anomaly counts also line up cleanly across modalities, with sparse anomaly volumes relative to the full branch sizes\. So the practical takeaway is that Isolation Forest is now packaged as a complete downstream\-ready anomaly framework, not just as a set of calibration outputs\.

In [40]:
# # ---------------------------------------------------------------------
# # Throwaway cleanup: reset only final IF export artifacts
# # ---------------------------------------------------------------------

# if_manifest_path = FINAL_333_DIR / "isolation_forest_anomaly_export_manifest.csv"
# if_final_row_level_dir = FINAL_333_DIR / "isolation_forest_candidate_anomaly_row_level_parts"
# if_temp_row_level_dir = INTERMEDIATE_333_DIR / "if_final_row_level_temp_parts"

# if if_manifest_path.exists():
#     if_manifest_path.unlink()

# if if_final_row_level_dir.exists():
#     shutil.rmtree(if_final_row_level_dir)
# if_final_row_level_dir.mkdir(parents=True, exist_ok=True)

# if if_temp_row_level_dir.exists():
#     shutil.rmtree(if_temp_row_level_dir)
# if_temp_row_level_dir.mkdir(parents=True, exist_ok=True)

# print("Reset final Isolation Forest export artifacts only.")

## 3\.3\.3\.6 Review Aggregate Isolation Forest Spatial Footprint

Now that the retained Isolation Forest branches have been exported across the full modeling universe, this part steps back and checks whether those full anomaly surfaces behave sensibly in aggregate\. The goal is to confirm that the exported outputs are selective, geographically interpretable, and coherent across temporal buckets and policy geographies before we carry them into downstream framework comparison\.

This is a full\-surface validation step, not another calibration exercise\. At this point, every summary in the section should be reading from the final Isolation Forest row\-level exports rather than from the 50K calibration panel or the retained stability subsets\.

> 🎯 Decision\.  We are intentionally not repeating the row\-by\-row human anomaly review that we used for DBSCAN\. By this point, the retained Isolation Forest branches have already passed contamination screening, stability review, and a direct comparison against DBSCAN\. That comparison showed that Isolation Forest is mostly extracting a smaller, more selective anomaly core from within a broader DBSCAN anomaly field rather than introducing a completely unfamiliar anomaly surface\. Because of that, the most useful remaining sanity check is aggregate spatial coherence, not a second deep inspection of individual rows\.

### Summarize aggregate spatial concentration of the retained Isolation Forest outputs

This block gives us the compact statistical readout for the final anomaly surfaces: how many anomaly rows each modality has, how many distinct taxi zones those anomalies touch, and how concentrated they are in the top one and top five zones\.

In [41]:
# ---------------------------------------------------------------------
# Helper: load final exported retained Isolation Forest row-level outputs
# ---------------------------------------------------------------------

def resolve_if_final_export_manifest():
    if "if_final_row_level_manifest_df" in globals():
        return if_final_row_level_manifest_df.copy()

    assert IF_ANOMALY_EXPORT_MANIFEST_PATH.exists(), (
        f"Missing final Isolation Forest export manifest: {IF_ANOMALY_EXPORT_MANIFEST_PATH}"
    )
    return pd.read_csv(IF_ANOMALY_EXPORT_MANIFEST_PATH)


def resolve_if_final_row_level_path_column(manifest_df):
    path_col_candidates = [
        "row_level_output_path",
        "row_level_path",
        "output_path",
        "path",
    ]
    path_col = next((c for c in path_col_candidates if c in manifest_df.columns), None)
    assert path_col is not None, (
        "Could not identify the final Isolation Forest row-level path column "
        f"in manifest. Available columns: {manifest_df.columns.tolist()}"
    )
    return path_col


def load_final_if_export_branch(feature_set):
    manifest_df = resolve_if_final_export_manifest()
    path_col = resolve_if_final_row_level_path_column(manifest_df)

    branch_manifest_df = (
        manifest_df.loc[manifest_df["feature_set"].eq(feature_set)]
        .copy()
        .reset_index(drop=True)
    )

    assert len(branch_manifest_df) == 1, (
        f"Expected exactly one final exported IF branch for {feature_set}, "
        f"found {len(branch_manifest_df)}."
    )

    export_path = Path(branch_manifest_df.loc[0, path_col])
    assert export_path.exists(), f"Missing final IF export file: {export_path}"

    return pd.read_parquet(export_path)


def ensure_if_final_map_metadata(df, feature_set):
    required_cols = [
        MODEL_ID_COLUMN,
        ZONE_ID_COLUMN,
        BOROUGH_COLUMN,
        ZONE_COLUMN,
        TEMPORAL_BUCKET_COLUMN,
    ]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if not missing_cols:
        return df

    row_metadata_df = pd.read_parquet(
        ROW_METADATA_PATHS_BY_SET[feature_set],
        columns=[MODEL_ID_COLUMN, *[c for c in missing_cols if c != MODEL_ID_COLUMN]],
    ).drop_duplicates(subset=[MODEL_ID_COLUMN])

    return df.merge(
        row_metadata_df,
        on=MODEL_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )


def detect_final_if_anomaly_flag(df):
    candidate_cols = [
        "is_if_anomaly",
        "is_anomaly",
        "anomaly_flag",
        "anomaly_like_flag",
        "if_anomaly_flag",
    ]
    for c in candidate_cols:
        if c in df.columns:
            return df[c].fillna(False).astype(bool)

    if "decision_function_score" in df.columns:
        return df["decision_function_score"].lt(0)

    raise AssertionError(
        "Could not identify an Isolation Forest anomaly flag column in the final export. "
        f"Available columns: {list(df.columns)}"
    )

In [42]:
# ---------------------------------------------------------------------
# Summarize aggregate spatial concentration of retained IF outputs
# ---------------------------------------------------------------------

assert "if_final_contamination_selection_df" in globals(), (
    "if_final_contamination_selection_df is not in memory. Please run the final Isolation Forest contamination-selection block first."
)
assert "load_final_if_export_branch" in globals(), (
    "load_final_if_export_branch is not in memory. Please run the final IF export helper block first."
)

zone_summary_records = []
zone_rank_frames = []

for feature_set in MODEL_FEATURE_SET_NAMES:
    branch_df = load_final_if_export_branch(feature_set)
    branch_df = ensure_if_final_map_metadata(branch_df, feature_set)
    branch_df["is_if_anomaly"] = detect_final_if_anomaly_flag(branch_df)

    anomaly_df = branch_df.loc[branch_df["is_if_anomaly"]].copy()

    zone_counts_df = (
        anomaly_df.groupby([ZONE_ID_COLUMN, BOROUGH_COLUMN, ZONE_COLUMN], dropna=False)
        .size()
        .rename("anomaly_rows")
        .reset_index()
        .sort_values("anomaly_rows", ascending=False)
        .reset_index(drop=True)
    )

    total_anomaly_rows = int(zone_counts_df["anomaly_rows"].sum())
    distinct_anomaly_zones = int(zone_counts_df[ZONE_ID_COLUMN].nunique())

    if total_anomaly_rows > 0:
        zone_counts_df["anomaly_share_within_modality"] = (
            zone_counts_df["anomaly_rows"] / total_anomaly_rows
        )
        top_zone_share = float(zone_counts_df["anomaly_share_within_modality"].iloc[0])
        top5_zone_share = float(
            zone_counts_df["anomaly_share_within_modality"].head(5).sum()
        )
    else:
        zone_counts_df["anomaly_share_within_modality"] = np.nan
        top_zone_share = np.nan
        top5_zone_share = np.nan

    zone_counts_df["feature_set"] = feature_set
    zone_counts_df["zone_rank"] = np.arange(1, len(zone_counts_df) + 1)

    zone_rank_frames.append(zone_counts_df)

    zone_summary_records.append(
        {
            "feature_set": feature_set,
            "anomaly_rows": total_anomaly_rows,
            "distinct_anomaly_zones": distinct_anomaly_zones,
            "top_zone_share": top_zone_share,
            "top5_zone_share": top5_zone_share,
        }
    )

if_spatial_footprint_summary_df = (
    pd.DataFrame(zone_summary_records)
    .sort_values("feature_set")
    .reset_index(drop=True)
)

if_zone_concentration_rank_df = (
    pd.concat(zone_rank_frames, ignore_index=True)
    .sort_values(["feature_set", "zone_rank"])
    .reset_index(drop=True)
)

if should_rebuild(REBUILD_IF_HUMAN_REVIEW, IF_SPATIAL_FOOTPRINT_SUMMARY_PATH):
    if_spatial_footprint_summary_df.to_parquet(
        IF_SPATIAL_FOOTPRINT_SUMMARY_PATH,
        index=False,
    )

if should_rebuild(REBUILD_IF_HUMAN_REVIEW, IF_ZONE_MAP_SUMMARY_PATH):
    if_zone_concentration_rank_df.to_parquet(
        IF_ZONE_MAP_SUMMARY_PATH,
        index=False,
    )

display(
    if_spatial_footprint_summary_df.style.format(
        {
            "top_zone_share": "{:.3f}",
            "top5_zone_share": "{:.3f}",
        }
    )
)

,feature_set,anomaly_rows,distinct_anomaly_zones,top_zone_share,top5_zone_share
0,bus,7365,81,0.355,0.801
1,fhvhv,7709,238,0.104,0.402
2,multimodal,7709,197,0.164,0.556
3,subway,4554,40,0.338,0.856
4,taxi,7709,236,0.180,0.547


Findings\. Bus and Subway are the most spatially concentrated Isolation Forest branches by far\. Bus anomalies appear in 81 zones and Subway in just 40, and in both cases the top five zones account for about 80% or more of all anomaly rows\. That means those two branches are behaving like sharp hotspot detectors rather than broad citywide anomaly surfaces\. FHVHV, Taxi, and Multimodal are much more distributed\. FHVHV and Taxi each span more than 230 anomaly zones, while Multimodal covers 197 and is somewhat more top\-heavy than FHVHV\. So the main takeaway is simple: Bus and Subway are narrow and concentrated, while Taxi, FHVHV, and Multimodal spread anomaly burden much more broadly across the network\.

Map overall anomaly concentration by modality\. This first choropleth answers a simple question: once we aggregate across dates and local comparison groups, where does each modality’s retained Isolation Forest anomaly burden actually land? The toggle switches modalities, and the color scale shows the share of that modality’s anomaly rows attributed to each taxi zone\.

In [43]:
# ---------------------------------------------------------------------
# Visualize anomaly concentration for one modality at a time
# ---------------------------------------------------------------------

IF_MODALITY_TO_MAP = "bus"  # bus | subway | taxi | fhvhv | multimodal

assert IF_MODALITY_TO_MAP in MODEL_FEATURE_SET_NAMES, (
    f"Unsupported modality: {IF_MODALITY_TO_MAP}"
)

assert "if_zone_concentration_rank_df" in globals(), (
    "if_zone_concentration_rank_df is not in memory. "
    "Please run the rewired aggregate hotspot-summary block first so this map "
    "uses the final full-universe Isolation Forest exports."
)

if "TAXI_ZONE_GEOMETRY_PATH" not in globals():
    TAXI_ZONE_GEOMETRY_PATH = PIPELINE_DATA_DIR / "1.1.6.nyc_taxi_zones.parquet"
if "HARMONIZED_TAXI_ZONES_PATH" not in globals():
    HARMONIZED_TAXI_ZONES_PATH = (
        PIPELINE_DATA_DIR / "1.3.1.final_tables" / "nyc_taxi_zones_harmonized.parquet"
    )

def load_harmonized_taxi_zone_geometry_for_if():
    candidate_paths = [
        HARMONIZED_TAXI_ZONES_PATH,
        TAXI_ZONE_GEOMETRY_PATH,
    ]

    last_error = None
    for candidate_path in candidate_paths:
        candidate_path = Path(candidate_path)
        if not candidate_path.exists():
            continue

        try:
            gdf = gpd.read_parquet(candidate_path)
        except Exception as exc:
            last_error = exc
            continue

        gdf = gdf.copy()

        rename_map = {}
        if "taxi_zone_id" not in gdf.columns:
            for candidate in ["LocationID", "location_id", "locationid", "OBJECTID", "objectid"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = "taxi_zone_id"
                    break
        if "borough" not in gdf.columns:
            for candidate in ["Borough", "borough_name"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = "borough"
                    break
        if "zone" not in gdf.columns:
            for candidate in ["Zone", "zone_name", "taxi_zone"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = "zone"
                    break

        if rename_map:
            gdf = gdf.rename(columns=rename_map)

        if "taxi_zone_id" not in gdf.columns:
            last_error = AssertionError(
                f"Geometry file {candidate_path} does not contain a usable taxi-zone id column."
            )
            continue

        if "borough" not in gdf.columns:
            gdf["borough"] = ""
        if "zone" not in gdf.columns:
            gdf["zone"] = ""

        gdf["taxi_zone_id"] = gdf["taxi_zone_id"].astype(str)
        return gdf

    raise AssertionError(
        f"Could not load taxi-zone geometry from any candidate path. Last error: {last_error}"
    )

taxi_zone_gdf = load_harmonized_taxi_zone_geometry_for_if()

# Match the 3.3.2 simplification flow.
taxi_zone_plot_gdf = taxi_zone_gdf.copy().to_crs("EPSG:2263")
taxi_zone_plot_gdf["geometry"] = taxi_zone_plot_gdf["geometry"].simplify(
    tolerance=150,
    preserve_topology=True,
)
taxi_zone_plot_gdf = taxi_zone_plot_gdf.to_crs("EPSG:4326")
taxi_zone_plot_gdf["taxi_zone_id"] = taxi_zone_plot_gdf["taxi_zone_id"].astype(str)

# Build geojson and explicitly coerce feature property ids to string.
taxi_zone_geojson = json.loads(
    taxi_zone_plot_gdf[["taxi_zone_id", "geometry"]].to_json()
)
for feature in taxi_zone_geojson["features"]:
    feature["properties"]["taxi_zone_id"] = str(feature["properties"]["taxi_zone_id"])

minx, miny, maxx, maxy = taxi_zone_plot_gdf.total_bounds
map_center = {
    "lon": float((minx + maxx) / 2.0),
    "lat": float((miny + maxy) / 2.0),
}

# Source rewiring only:
# this table should now be the rebuilt full-universe hotspot table, not the
# earlier calibration-scale retained branch summary.
zone_rank_subset_df = (
    if_zone_concentration_rank_df.loc[
        if_zone_concentration_rank_df["feature_set"].eq(IF_MODALITY_TO_MAP),
        [
            "taxi_zone_id",
            "borough",
            "zone",
            "anomaly_rows",
            "anomaly_share_within_modality",
        ],
    ]
    .copy()
)
zone_rank_subset_df["taxi_zone_id"] = zone_rank_subset_df["taxi_zone_id"].astype(str)

feature_zone_df = (
    taxi_zone_plot_gdf[["taxi_zone_id", "borough", "zone"]]
    .merge(
        zone_rank_subset_df,
        on="taxi_zone_id",
        how="left",
        suffixes=("_shape", ""),
    )
)

feature_zone_df["borough"] = feature_zone_df["borough"].fillna(feature_zone_df["borough_shape"])
feature_zone_df["zone"] = feature_zone_df["zone"].fillna(feature_zone_df["zone_shape"])

feature_zone_df = feature_zone_df.drop(
    columns=[c for c in ["borough_shape", "zone_shape"] if c in feature_zone_df.columns]
).fillna(
    {
        "anomaly_rows": 0,
        "anomaly_share_within_modality": 0.0,
        "borough": "",
        "zone": "",
    }
)

feature_zone_df["anomaly_share_pct_within_modality"] = (
    100.0 * feature_zone_df["anomaly_share_within_modality"]
)

fig = go.Figure(
    go.Choroplethmapbox(
        geojson=taxi_zone_geojson,
        locations=feature_zone_df["taxi_zone_id"],
        z=feature_zone_df["anomaly_share_pct_within_modality"],
        featureidkey="properties.taxi_zone_id",
        colorscale=IF_MAP_SCALE,
        zmin=0,
        zmax=float(100.0 * if_zone_concentration_rank_df["anomaly_share_within_modality"].max()),
        marker_opacity=0.85,
        marker_line_width=0.25,
        colorbar=dict(
            title=dict(
                text="Share of modality anomalies (%)",
                font=dict(color=BRAND_COLORS["dark_teal"]),
            ),
            tickfont=dict(color=BRAND_COLORS["dark_teal"]),
            x=1.01,
        ),
        customdata=np.column_stack(
            [
                feature_zone_df["borough"],
                feature_zone_df["zone"],
                feature_zone_df["anomaly_rows"],
                feature_zone_df["anomaly_share_pct_within_modality"],
            ]
        ),
        hovertemplate=(
            "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
            "Anomaly rows: %{customdata[2]:,.0f}<br>"
            "Share of modality anomalies: %{customdata[3]:.3f}%<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=f"Final Isolation Forest Anomaly Concentration by Taxi Zone: {IF_MODALITY_TO_MAP.upper()}",
    width=980,
    height=760,
    margin=dict(l=20, r=20, t=80, b=20),
    mapbox=dict(
        style="carto-positron",
        zoom=9.3,
        center=map_center,
    ),
)

fig.show()

print(f"{IF_MODALITY_TO_MAP.upper()} choropleth rendered.")

/tmp/ipykernel_5444/277764865.py:153: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Choroplethmapbox(


BUS choropleth rendered.


Findings\. The retained Bus Isolation Forest surface remains concentrated outside the Manhattan core, with one dominant hotspot in eastern Queens and a smaller secondary cluster across the Jamaica Bay and Rockaway\-adjacent zones\. That spatial pattern is highly localized rather than citywide, and it lines up cleanly with the aggregate zone\-concentration results\.

Let's map Taxi anomaly concentration by temporal bucket\. Taxi remained one of the clearest local\-regime cases in the framework comparison, so this second choropleth keeps the spatial lens fixed on Taxi and rotates the temporal bucket instead\. Each view is normalized within that Taxi temporal bucket, so darker zones mean that a larger share of that bucket’s retained Isolation Forest anomalies landed there\.

In [44]:
# ---------------------------------------------------------------------
# Visualize Taxi anomaly concentration for one temporal bucket at a time
# ---------------------------------------------------------------------

IF_TAXI_TEMPORAL_BUCKET_TO_MAP = "weekday_pm_peak"

TEMPORAL_BUCKET_ORDER = [
    "weekday_am_peak",
    "weekday_midday",
    "weekday_evening",
    "weekday_pm_peak",
    "weekday_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_evening",
    "weekend_pm_peak",
    "weekend_overnight",
]

assert IF_TAXI_TEMPORAL_BUCKET_TO_MAP in TEMPORAL_BUCKET_ORDER, (
    f"Unsupported temporal bucket: {IF_TAXI_TEMPORAL_BUCKET_TO_MAP}"
)

if "TAXI_ZONE_GEOMETRY_PATH" not in globals():
    TAXI_ZONE_GEOMETRY_PATH = PIPELINE_DATA_DIR / "1.1.6.nyc_taxi_zones.parquet"
if "HARMONIZED_TAXI_ZONES_PATH" not in globals():
    HARMONIZED_TAXI_ZONES_PATH = (
        PIPELINE_DATA_DIR / "1.3.1.final_tables" / "nyc_taxi_zones_harmonized.parquet"
    )

def load_harmonized_taxi_zone_geometry_for_if():
    candidate_paths = [
        HARMONIZED_TAXI_ZONES_PATH,
        TAXI_ZONE_GEOMETRY_PATH,
    ]

    last_error = None
    for candidate_path in candidate_paths:
        candidate_path = Path(candidate_path)
        if not candidate_path.exists():
            continue

        try:
            gdf = gpd.read_parquet(candidate_path)
        except Exception as exc:
            last_error = exc
            continue

        gdf = gdf.copy()

        rename_map = {}
        if "taxi_zone_id" not in gdf.columns:
            for candidate in ["LocationID", "location_id", "locationid", "OBJECTID", "objectid"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = "taxi_zone_id"
                    break
        if "borough" not in gdf.columns:
            for candidate in ["Borough", "borough_name"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = "borough"
                    break
        if "zone" not in gdf.columns:
            for candidate in ["Zone", "zone_name", "taxi_zone"]:
                if candidate in gdf.columns:
                    rename_map[candidate] = "zone"
                    break

        if rename_map:
            gdf = gdf.rename(columns=rename_map)

        if "taxi_zone_id" not in gdf.columns:
            last_error = AssertionError(
                f"Geometry file {candidate_path} does not contain a usable taxi-zone id column."
            )
            continue

        if "borough" not in gdf.columns:
            gdf["borough"] = ""
        if "zone" not in gdf.columns:
            gdf["zone"] = ""

        gdf["taxi_zone_id"] = gdf["taxi_zone_id"].astype(str)
        return gdf

    raise AssertionError(
        f"Could not load taxi-zone geometry from any candidate path. Last error: {last_error}"
    )

# Source rewiring only:
# use the final Taxi Isolation Forest export rather than the earlier
# calibration-scale retained row-level manifest.
assert "load_final_if_export_branch" in globals(), (
    "load_final_if_export_branch is not in memory. "
    "Please run the final-export helper block first."
)
assert "ensure_if_final_map_metadata" in globals(), (
    "ensure_if_final_map_metadata is not in memory. "
    "Please run the final-export helper block first."
)
assert "detect_final_if_anomaly_flag" in globals(), (
    "detect_final_if_anomaly_flag is not in memory. "
    "Please run the final-export helper block first."
)

taxi_branch_df = load_final_if_export_branch("taxi")
taxi_branch_df = ensure_if_final_map_metadata(taxi_branch_df, "taxi")
taxi_branch_df["is_if_anomaly"] = detect_final_if_anomaly_flag(taxi_branch_df)
taxi_branch_df["taxi_zone_id"] = taxi_branch_df["taxi_zone_id"].astype(str)

taxi_anomaly_df = taxi_branch_df.loc[taxi_branch_df["is_if_anomaly"]].copy()

if_taxi_temporal_zone_map_df = (
    taxi_anomaly_df.groupby(["temporal_bucket", "taxi_zone_id", "borough", "zone"], dropna=False)
    .size()
    .rename("anomaly_rows")
    .reset_index()
)

if_taxi_temporal_totals_df = (
    if_taxi_temporal_zone_map_df.groupby("temporal_bucket", dropna=False)["anomaly_rows"]
    .sum()
    .rename("temporal_bucket_anomaly_rows")
    .reset_index()
)

if_taxi_temporal_zone_map_df = (
    if_taxi_temporal_zone_map_df.merge(
        if_taxi_temporal_totals_df,
        on="temporal_bucket",
        how="left",
    )
    .assign(
        anomaly_share_within_taxi_bucket=lambda df: (
            df["anomaly_rows"] / df["temporal_bucket_anomaly_rows"]
        ),
        anomaly_share_pct_within_taxi_bucket=lambda df: (
            100.0 * df["anomaly_share_within_taxi_bucket"]
        ),
    )
)

taxi_zone_gdf = load_harmonized_taxi_zone_geometry_for_if()

taxi_zone_plot_gdf = taxi_zone_gdf.copy().to_crs("EPSG:2263")
taxi_zone_plot_gdf["geometry"] = taxi_zone_plot_gdf["geometry"].simplify(
    tolerance=150,
    preserve_topology=True,
)
taxi_zone_plot_gdf = taxi_zone_plot_gdf.to_crs("EPSG:4326")
taxi_zone_plot_gdf["taxi_zone_id"] = taxi_zone_plot_gdf["taxi_zone_id"].astype(str)

taxi_zone_geojson = json.loads(
    taxi_zone_plot_gdf[["taxi_zone_id", "geometry"]].to_json()
)
for feature in taxi_zone_geojson["features"]:
    feature["properties"]["taxi_zone_id"] = str(feature["properties"]["taxi_zone_id"])

minx, miny, maxx, maxy = taxi_zone_plot_gdf.total_bounds
map_center = {
    "lon": float((minx + maxx) / 2.0),
    "lat": float((miny + maxy) / 2.0),
}

bucket_zone_subset_df = (
    if_taxi_temporal_zone_map_df.loc[
        if_taxi_temporal_zone_map_df["temporal_bucket"].eq(IF_TAXI_TEMPORAL_BUCKET_TO_MAP),
        [
            "taxi_zone_id",
            "borough",
            "zone",
            "anomaly_rows",
            "anomaly_share_pct_within_taxi_bucket",
        ],
    ]
    .copy()
)
bucket_zone_subset_df["taxi_zone_id"] = bucket_zone_subset_df["taxi_zone_id"].astype(str)

bucket_zone_df = (
    taxi_zone_plot_gdf[["taxi_zone_id", "borough", "zone"]]
    .merge(
        bucket_zone_subset_df,
        on="taxi_zone_id",
        how="left",
        suffixes=("_shape", ""),
    )
)

bucket_zone_df["borough"] = bucket_zone_df["borough"].fillna(bucket_zone_df["borough_shape"])
bucket_zone_df["zone"] = bucket_zone_df["zone"].fillna(bucket_zone_df["zone_shape"])

bucket_zone_df = bucket_zone_df.drop(
    columns=[c for c in ["borough_shape", "zone_shape"] if c in bucket_zone_df.columns]
).fillna(
    {
        "anomaly_rows": 0,
        "anomaly_share_pct_within_taxi_bucket": 0.0,
        "borough": "",
        "zone": "",
    }
)

fig = go.Figure(
    go.Choroplethmapbox(
        geojson=taxi_zone_geojson,
        locations=bucket_zone_df["taxi_zone_id"],
        z=bucket_zone_df["anomaly_share_pct_within_taxi_bucket"],
        featureidkey="properties.taxi_zone_id",
        colorscale=IF_MAP_SCALE,
        zmin=0,
        zmax=float(if_taxi_temporal_zone_map_df["anomaly_share_pct_within_taxi_bucket"].max()),
        marker_opacity=0.85,
        marker_line_color=BRAND_COLORS["dark_teal"],
        marker_line_width=0.25,
        colorbar=dict(
            title=dict(
                text="Share of Taxi bucket anomalies (%)",
                font=dict(color=BRAND_COLORS["dark_teal"]),
            ),
            tickfont=dict(color=BRAND_COLORS["dark_teal"]),
            x=1.01,
        ),
        customdata=np.column_stack(
            [
                bucket_zone_df["borough"],
                bucket_zone_df["zone"],
                bucket_zone_df["anomaly_rows"],
                bucket_zone_df["anomaly_share_pct_within_taxi_bucket"],
            ]
        ),
        hovertemplate=(
            "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
            "Taxi anomaly rows in bucket: %{customdata[2]:,.0f}<br>"
            "Share of bucket anomalies: %{customdata[3]:.3f}%<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=f"Taxi Isolation Forest Anomaly Concentration by Taxi Zone: {IF_TAXI_TEMPORAL_BUCKET_TO_MAP}",
    width=980,
    height=760,
    margin=dict(l=20, r=20, t=80, b=20),
    mapbox=dict(
        style="carto-positron",
        zoom=9.3,
        center=map_center,
    ),
)

fig.show()

print(f"Taxi Isolation Forest choropleth rendered for {IF_TAXI_TEMPORAL_BUCKET_TO_MAP}.")

/tmp/ipykernel_5444/3350709677.py:204: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Choroplethmapbox(


Taxi Isolation Forest choropleth rendered for weekday_pm_peak.


Findings\. The retained Taxi Isolation Forest surface remains concentrated in a small number of zones rather than broadly diffused across the city, and the weekday\_pm\_peak view makes the Manhattan\-core pattern especially clear\. The anomaly burden is dominated by a tight cluster in central Manhattan, with very little comparable intensity elsewhere on the map\.

### Review Aggregate Spatial Footprint and Directional Meaning of Retained Isolation Forest Anomalies

At this point, we do not need a second deep row\-level review of individual Isolation Forest anomalies\. The more useful question is whether the retained anomaly surfaces look coherent in aggregate and whether they carry an interpretable directional meaning\. This subpart does two things\. First, it reviews the aggregate taxi\-zone footprint of the retained Isolation Forest outputs, including the taxi\-zone QA we just ran to confirm where anomaly mass is actually landing\. Second, it checks anomaly directionality: whether retained anomalies tend to reflect above\-normal congestion\-like conditions, below\-normal conditions, or a mixed pattern depending on modality and location\.

The goal is to make the retained Isolation Forest branches easier to interpret in practical terms\. Instead of only knowing that a row is unusual, we want to know whether it is unusual because mobility conditions are running hotter than normal, colder than normal, or in a more mixed configuration\. That gives us a stronger read on whether these anomaly surfaces are meaningful enough to carry forward into downstream framework comparison\.

Summarize the aggregate top anomaly zones within each retained Isolation Forest modality\. This table is a compact spatial audit of the retained Isolation Forest outputs at the modality level\. It ranks the top 10 taxi zones within each modality by anomaly volume and shows anomaly\_share\_within\_modality, which is the share of that modality’s total anomaly rows contributed by each zone\. The key thing to read here is not just the top zone, but the overall shape of the top 10: whether anomaly mass is concentrated in a few core Manhattan zones, pushed toward airports and outer\-borough edges, or split across both\.

In [45]:
# ---------------------------------------------------------------------
# Investigate retained Isolation Forest hotspot patterns from the top down
# ---------------------------------------------------------------------

IF_TAXI_TEMPORAL_BUCKET_TO_REVIEW = "weekday_pm_peak"

assert IF_TAXI_TEMPORAL_BUCKET_TO_REVIEW in [
    "weekday_am_peak",
    "weekday_midday",
    "weekday_evening",
    "weekday_pm_peak",
    "weekday_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_evening",
    "weekend_pm_peak",
    "weekend_overnight",
], (
    f"Unsupported Taxi temporal bucket: {IF_TAXI_TEMPORAL_BUCKET_TO_REVIEW}"
)

if_zone_concentration_records = []

for feature_set in MODEL_FEATURE_SET_NAMES:
    branch_df = load_final_if_export_branch(feature_set)
    branch_df = ensure_if_final_map_metadata(branch_df, feature_set)
    branch_df["is_if_anomaly"] = detect_final_if_anomaly_flag(branch_df)
    branch_df[ZONE_ID_COLUMN] = branch_df[ZONE_ID_COLUMN].astype(str)

    anomaly_df = branch_df.loc[branch_df["is_if_anomaly"]].copy()

    zone_summary_df = (
        anomaly_df.groupby(
            [ZONE_ID_COLUMN, BOROUGH_COLUMN, ZONE_COLUMN],
            dropna=False,
        )
        .size()
        .rename("anomaly_rows")
        .reset_index()
        .sort_values("anomaly_rows", ascending=False)
        .reset_index(drop=True)
    )

    total_anomaly_rows = int(zone_summary_df["anomaly_rows"].sum())

    if total_anomaly_rows > 0:
        zone_summary_df["anomaly_share_within_modality"] = (
            zone_summary_df["anomaly_rows"] / total_anomaly_rows
        )
    else:
        zone_summary_df["anomaly_share_within_modality"] = np.nan

    zone_summary_df["feature_set"] = feature_set
    zone_summary_df["zone_rank"] = np.arange(1, len(zone_summary_df) + 1)

    if_zone_concentration_records.append(zone_summary_df)

if_zone_concentration_rank_df = (
    pd.concat(if_zone_concentration_records, ignore_index=True)
    .sort_values(["feature_set", "zone_rank"])
    .reset_index(drop=True)
)

# 1) Aggregate modality-level top zones from the rebuilt full-universe zone table.
if_aggregate_top_zones_df = (
    if_zone_concentration_rank_df.loc[
        if_zone_concentration_rank_df["zone_rank"].le(10)
    ]
    .copy()
    .sort_values(["feature_set", "zone_rank"])
    .reset_index(drop=True)
)

display(
    if_aggregate_top_zones_df[
        [
            "feature_set",
            "zone_rank",
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            "anomaly_rows",
            "anomaly_share_within_modality",
        ]
    ].style.format(
        {
            "anomaly_share_within_modality": "{:.3f}",
        }
    )
)

,feature_set,zone_rank,borough,zone,anomaly_rows,anomaly_share_within_modality
0,bus,1,Queens,Jamaica,2614,0.355
1,bus,2,Queens,Jamaica Bay,1511,0.205
2,bus,3,Manhattan,Financial District South,730,0.099
3,bus,4,Brooklyn,Carroll Gardens,701,0.095
4,bus,5,Queens,Breezy Point/Fort Tilden/Riis Beach,347,0.047
5,bus,6,Brooklyn,Columbia Street,226,0.031
6,bus,7,Bronx,Rikers Island,186,0.025
7,bus,8,Brooklyn,Gowanus,151,0.021
8,bus,9,Staten Island,Oakwood,139,0.019
9,bus,10,Staten Island,Freshkills Park,87,0.012


Findings\. The retained Isolation Forest zone footprints remain highly modality\-specific\. Bus is still the clearest outer\-borough case, dominated by Jamaica and Jamaica Bay; FHVHV also leans peripheral, with LaGuardia, Freshkills Park, Breezy Point/Fort Tilden/Riis Beach, and Willets Point all prominent\. Multimodal is more mixed, combining a strong Queens concentration with a clear Manhattan core around Midtown Center and Times Square/Theatre District\. Subway and Taxi are the most Manhattan\-centered branches: Subway is led by East Chelsea, Garment District, and Times Square/Theatre District, while Taxi is dominated by Upper East Side South, Midtown Center, East Village, and Upper East Side North\. So the practical takeaway is the same: Bus and FHVHV skew peripheral, Multimodal is mixed, and Subway and Taxi are much more Manhattan\-core oriented\.

Inspect the dominant Taxi anomaly zones inside the selected temporal bucket\. This table zooms in from the aggregate Taxi surface to one specific operating slice: the retained Taxi temporal bucket we mapped\. Here anomaly\_share\_within\_bucket tells us what share of that bucket’s Taxi anomalies comes from each zone\. This is useful because it answers a narrower question than the aggregate table: once we hold time context fixed, where is the anomaly mass actually going?

In [46]:
# 2) Rebuild Taxi bucket-specific top zones directly from the final IF Taxi export.

taxi_branch_df = load_final_if_export_branch("taxi")
taxi_branch_df = ensure_if_final_map_metadata(taxi_branch_df, "taxi")
taxi_branch_df["is_if_anomaly"] = detect_final_if_anomaly_flag(taxi_branch_df)
taxi_branch_df[ZONE_ID_COLUMN] = taxi_branch_df[ZONE_ID_COLUMN].astype(str)

taxi_bucket_top_zones_df = (
    taxi_branch_df.loc[
        taxi_branch_df["is_if_anomaly"]
        & taxi_branch_df[TEMPORAL_BUCKET_COLUMN].eq(IF_TAXI_TEMPORAL_BUCKET_TO_REVIEW)
    ]
    .groupby([ZONE_ID_COLUMN, BOROUGH_COLUMN, ZONE_COLUMN], dropna=False)
    .size()
    .rename("anomaly_rows")
    .reset_index()
    .sort_values("anomaly_rows", ascending=False)
    .reset_index(drop=True)
)

taxi_bucket_total_rows = int(taxi_bucket_top_zones_df["anomaly_rows"].sum())

if taxi_bucket_total_rows > 0:
    taxi_bucket_top_zones_df["anomaly_share_within_bucket"] = (
        taxi_bucket_top_zones_df["anomaly_rows"] / taxi_bucket_total_rows
    )
else:
    taxi_bucket_top_zones_df["anomaly_share_within_bucket"] = np.nan

taxi_bucket_top_zones_df["zone_rank"] = np.arange(1, len(taxi_bucket_top_zones_df) + 1)

display(
    taxi_bucket_top_zones_df.loc[
        taxi_bucket_top_zones_df["zone_rank"].le(10),
        [
            "zone_rank",
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            "anomaly_rows",
            "anomaly_share_within_bucket",
        ],
    ].style.format(
        {
            "anomaly_share_within_bucket": "{:.3f}",
        }
    )
)

,zone_rank,borough,zone,anomaly_rows,anomaly_share_within_bucket
0,1,Manhattan,Midtown Center,312,0.282
1,2,Manhattan,Upper East Side South,244,0.220
2,3,Manhattan,Upper East Side North,114,0.103
3,4,Manhattan,Midtown East,77,0.070
4,5,Queens,JFK Airport,35,0.032
5,6,Manhattan,Union Sq,32,0.029
6,7,Manhattan,Lincoln Square East,28,0.025
7,8,Manhattan,Midtown North,24,0.022
8,9,Manhattan,Murray Hill,22,0.020
9,10,Manhattan,Lenox Hill West,17,0.015


Findings\. Within the selected Taxi bucket, the anomaly surface is strongly concentrated rather than broadly diffuse\. Midtown Center alone accounts for 28\.2% of bucket anomalies and Upper East Side South adds another 22\.0%, so just those two Manhattan zones make up about half of the bucket’s anomaly mass\. After that, the footprint drops off quickly into a smaller supporting set led mostly by other Manhattan core zones, with JFK Airport as the only notable non\-Manhattan entry in the top five\. In plain language, once we hold Taxi to this one operating context, Isolation Forest is pointing very hard at a narrow Manhattan core with only a small airport tail\.

Check how Manhattan\-heavy the top retained Isolation Forest anomaly zones are by modality\. This is a very simple sanity\-check table\. It counts how many anomaly rows live inside each modality’s top 10 zones \(top10\_zone\_rows\) and how many of those rows come from Manhattan zones specifically \(manhattan\_top10\_rows\)\. The goal is not to replace the richer zone tables, but to give us one fast read on whether each modality’s retained hotspot structure is mostly Manhattan\-core or not\.

In [47]:
# 3) A small Manhattan/CBD check for narrative sanity.
manhattan_check_df = (
    if_aggregate_top_zones_df.assign(
        is_manhattan=lambda df: df["borough"].astype(str).eq("Manhattan")
    )
    .groupby("feature_set", dropna=False)
    .agg(
        top10_zone_rows=("anomaly_rows", "sum"),
        manhattan_top10_rows=("is_manhattan", lambda s: int(s.sum())),
    )
    .reset_index()
)

display(manhattan_check_df)

,feature_set,top10_zone_rows,manhattan_top10_rows
0,bus,6692,1
1,fhvhv,4723,4
2,multimodal,5690,5
3,subway,4378,7
4,taxi,5638,8


Findings\. This Manhattan sanity check reinforces the same modality split\. Taxi and Subway are the most Manhattan\-heavy branches, with 8 and 7 Manhattan zones respectively in their top 10, which fits the strong core\-oriented pattern seen in the tables and maps\. Multimodal sits in the middle with 5 Manhattan top\-10 zones, while FHVHV is less core\-centered with 4\. Bus is the clearest outer\-borough exception, with only 1 Manhattan zone in its top 10\. So the fast read is simple: Taxi and Subway behave like Manhattan\-core hotspot detectors, Multimodal is mixed, and Bus remains the most peripheral\.

### Section Summary

This section kept the final Isolation Forest sanity check focused on aggregate coherence rather than another row\-level deep dive\. The retained PCA\-based anomaly surfaces show clear and interpretable spatial structure: Bus remains strongly peripheral, FHVHV also leans outer\-borough and airport\-oriented, Multimodal is mixed, and Subway and Taxi are much more Manhattan\-core concentrated\. The zone rankings, Manhattan check, and choropleths all reinforce the same basic story rather than pulling in different directions\. That is enough to conclude that the retained Isolation Forest outputs are behaving plausibly as sparse, selective mobility\-state anomaly surfaces and are ready to carry forward into downstream framework comparison\.

## Close

This notebook established a retained Isolation Forest anomaly branch that is coherent enough to carry forward into downstream framework comparison\. Contamination calibration and stability review narrowed the framework to PCA\-based branches only, with a retained contamination setting of 0\.005 for every modality, and the final full\-universe exports confirmed that those retained branches behave as sparse, selective anomaly surfaces rather than noisy or degenerate ones\. The aggregate spatial review also showed that Isolation Forest is not simply reproducing one generic geography: Bus stays peripheral, FHVHV is outer\-borough and airport\-oriented, Multimodal is mixed, and Subway and Taxi are much more Manhattan\-core concentrated\. With the final row\-level exports and manifest now written, Isolation Forest is ready to enter the next stage of cross\-framework comparison alongside DBSCAN and, later, GMM\.

### 3\.3\.3 Final Tables Inventory

selected\_isolation\_forest\_configurations\.csv
Final retained Isolation Forest configuration per modality, including the selected representation, contamination setting, and selection rationale\.

isolation\_forest\_anomaly\_export\_manifest\.csv
Compact manifest pointing downstream notebooks to the final Isolation Forest row\-level anomaly exports for each modality\.

isolation\_forest\_candidate\_anomaly\_row\_level\_parts/
Directory containing the final metadata\-enriched Isolation Forest anomaly row\-level parquet outputs, one file per modality\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>